[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ushaloy/PINN-LEWS-HYDROL86065/blob/main/notebooks/01_main_chronological.ipynb)

# Physics-informed soil-moisture model for landslide early warning — Dev108 / Dev107

Code for the revised manuscript **HYDROL86065** (Journal of Hydrology). The notebook reproduces every result table and figure of the revision from the raw node files in `data/`.

**Run order:** run the cells from top to bottom (*Runtime → Run all* on Colab). Two notebooks share the same code and differ only in `SPLIT_MODE`:

| Notebook | `SPLIT_MODE` | What it produces | GPU time (T4) |
|---|---|---|---|
| `01_main_chronological.ipynb` | `chrono` | main results, λ check, boundary scenarios, LSTM, transfer to Dev107, alerts | ≈ 8–9 h |
| `02_out_of_season.ipynb` | `season` | out-of-season test (train Nov–Mar, test May–Sep) | ≈ 2.5 h |

**After a disconnect:** run the setup cell, the main cell (set `PREP_ONLY = True` to skip re-training), the boundary-mode cell, then the cell you were in. Finished runs are saved and skipped.

**Pipeline check:** set `QUICK_TEST = True` in the setup cell. Every cell then runs with tiny budgets in a separate `quicktest_*` folder; the numbers are not results.

Terms: **PIML** = the physics-informed network (PINN); **ML** / **data-only** = the same network and training with every physics weight set to 0; **persistence** = θ(t) = θ(t − 2 min).


## 1. Setup
Choose `SPLIT_MODE` and (optionally) `QUICK_TEST`. The cell finds `data/`, creates the output folder `results/<split>/` (on Colab: `MyDrive/PINN-LEWS-HYDROL86065_results/<split>/`), records package versions and data checksums in `environment.txt`, and copies everything printed to `run_log.txt`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1. SETUP — run this cell first, in every session
# ═══════════════════════════════════════════════════════════════════════════
import os, sys, datetime, platform, hashlib

# ── choices that define a run ────────────────────────────────────────────────
SPLIT_MODE = "chrono"     # "chrono": main results (80/10/10 in time, test = 12–29 Sep 2026)
                          # "season": out-of-season test (train Nov–Mar, validate Apr, test May–Sep)
QUICK_TEST = False        # True: tiny budgets and 2 seeds, to check that every cell runs.
                          #       Quick-test numbers are NOT results; they go to a separate folder.

ARCH = dict(depth=5, width=128, act="tanh", taper=True)   # selected by the architecture search
LAM_FINAL, LAM_BC_FINAL = 0.1, 0.0     # λ_phys = 0.1; surface-flux term off (λ_BC = 0): it gave the
                                       # lowest validation error and never reduced its own residual

# training budgets (paper values); QUICK_TEST replaces them with tiny ones
BUDGET = dict(main_adam=400, main_lbfgs=200,      # main run and multi-seed result
              sens_adam=130, sens_lbfgs=70,       # λ check and boundary scenarios (~1/3 budget)
              seeds=[42, 43, 44],                 # every multi-seed analysis
              tl_steps=2000, tl_eval=50)          # transfer: max steps, evaluation interval
if QUICK_TEST:
    BUDGET = dict(main_adam=2, main_lbfgs=1, sens_adam=1, sens_lbfgs=1,
                  seeds=[42, 43], tl_steps=20, tl_eval=5)

# optional analyses (the main cell, boundary modes, multi-seed and raw-reading check always run)
RUN = dict(sens_grid   = SPLIT_MODE == "chrono",   # λ check (grid); season run: references only
           bc_scen     = SPLIT_MODE == "chrono",   # lower-boundary / initial-condition scenarios
           lstm        = SPLIT_MODE == "chrono",   # LSTM baseline
           transfer    = SPLIT_MODE == "chrono",   # Dev108 -> Dev107
           alerts      = SPLIT_MODE == "chrono",   # alert levels and lead times
           arch_search = False,                    # architecture search (≈ 3 h GPU)
           lam_ext     = False)                    # λ_phys 2–10 extension (≈ 4 h GPU)

# PREP_ONLY = True: the main cell prepares data and loss functions, then STOPS before training
# (nothing is overwritten). Use it to re-run one later cell after a disconnect.
PREP_ONLY = False

# REUSE_STORED = True (default): do not train again. The checkpoints and result tables stored in the repository
# (results/<split>/) are loaded and every table and figure is regenerated from them; stored results are never
# overwritten (new files go to results_rerun/<split>/). Only parts that are missing are computed.
# REUSE_STORED = False: train everything from scratch (about 8-9 h for the chrono run on a Colab T4).
REUSE_STORED = True

# ── locations ────────────────────────────────────────────────────────────────
#   data    : the repository's data/ folder (or PINN_DATA_DIR)
#   results : <repository>/results/<split>  (on Colab: Google Drive, so a disconnect loses nothing)
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_NAME = "PINN-LEWS-HYDROL86065"
REPO_URL  = "https://github.com/Ushaloy/PINN-LEWS-HYDROL86065"
if IN_COLAB and not os.environ.get("PINN_REPO_DIR") and not os.path.isdir(os.path.join("/content", REPO_NAME, "data")):
    # Colab opens only this notebook file, so fetch the repository (code + data) first
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, os.path.join("/content", REPO_NAME)], check=True)
_here = os.getcwd()
_cands = [os.environ.get("PINN_REPO_DIR", ""),
          os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here,
          os.path.join("/content", REPO_NAME)]
REPO_DIR = next((p for p in _cands if p and os.path.isdir(os.path.join(p, "data"))), None)
assert REPO_DIR, (f"data/ folder not found. On Colab run first:  !git clone <repository URL> /content/{REPO_NAME}  "
                  "— or set PINN_REPO_DIR to the repository folder.")
DATA_DIR = os.environ.get("PINN_DATA_DIR", os.path.join(REPO_DIR, "data"))

if REUSE_STORED and not QUICK_TEST:
    RESULTS_ROOT = os.environ.get("PINN_RESULTS_DIR", os.path.join(REPO_DIR, "results_rerun"))   # no Google Drive needed
elif IN_COLAB:
    if not os.environ.get("PINN_RESULTS_DIR"):
        drive.mount("/content/drive")
    RESULTS_ROOT = os.environ.get("PINN_RESULTS_DIR", f"/content/drive/MyDrive/{REPO_NAME}_results")
else:
    RESULTS_ROOT = os.environ.get("PINN_RESULTS_DIR", os.path.join(REPO_DIR, "results"))

DATA_CSV     = os.path.join(DATA_DIR, "dev_108_updated.csv")   # source node (model development)
DATA_CSV_107 = os.path.join(DATA_DIR, "dev_107_updated.csv")   # transfer target
for _p in (DATA_CSV, DATA_CSV_107):
    assert os.path.exists(_p), f"missing data file: {_p}"

ARCH_TAG = f"d{ARCH['depth']}w{ARCH['width']}{ARCH['act']}{'T' if ARCH['taper'] else ''}"
RUN_TAG  = f"{ARCH_TAG}_lp{LAM_FINAL:g}_bc{LAM_BC_FINAL:g}_{SPLIT_MODE}_fy"   # in every resume-file name
RUN_DIR  = os.path.join(RESULTS_ROOT, ("quicktest_" if QUICK_TEST else "") + SPLIT_MODE)
os.makedirs(RUN_DIR, exist_ok=True)
STORED_DIR = os.path.join(REPO_DIR, "results", SPLIT_MODE)
if REUSE_STORED and not QUICK_TEST and os.path.isdir(STORED_DIR) and os.path.abspath(STORED_DIR) != os.path.abspath(RUN_DIR):
    import shutil     # copy the stored checkpoints, caches and tables, so that finished runs are reused
    for _root, _dirs, _files in os.walk(STORED_DIR):
        _dst = os.path.join(RUN_DIR, os.path.relpath(_root, STORED_DIR)); os.makedirs(_dst, exist_ok=True)
        for _f in _files:
            if _f not in ("run_log.txt", "environment.txt") and not os.path.exists(os.path.join(_dst, _f)):
                shutil.copy2(os.path.join(_root, _f), _dst)


class _Tee:
    """Writes everything printed to the notebook AND to run_log.txt in RUN_DIR (line-buffered)."""
    def __init__(self, stream, path):
        self.stream, self.file = stream, open(path, "a", buffering=1, encoding="utf-8")
    def write(self, s):
        self.stream.write(s); self.file.write(s); return len(s)
    def flush(self):
        self.stream.flush(); self.file.flush()
        try: os.fsync(self.file.fileno())
        except OSError: pass
    def __getattr__(self, name): return getattr(self.stream, name)

if type(sys.stdout).__name__ == "_Tee":
    sys.stdout = sys.stdout.stream
sys.stdout = _Tee(sys.stdout, os.path.join(RUN_DIR, "run_log.txt"))

# ── environment record (for reproducibility) ────────────────────────────────
import numpy, pandas, scipy, sklearn, matplotlib, torch
def _sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
_env = [f"date        {datetime.datetime.now():%Y-%m-%d %H:%M}",
        f"run         {RUN_TAG}  quick_test={QUICK_TEST}",
        f"python      {platform.python_version()}  ({platform.platform()})",
        f"torch       {torch.__version__}  cuda={torch.cuda.is_available()}"
        + (f"  gpu={torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else ""),
        f"numpy       {numpy.__version__}", f"pandas      {pandas.__version__}",
        f"scipy       {scipy.__version__}", f"sklearn     {sklearn.__version__}",
        f"matplotlib  {matplotlib.__version__}",
        f"data 108    sha256 {_sha256(DATA_CSV)}", f"data 107    sha256 {_sha256(DATA_CSV_107)}",
        f"budget      {BUDGET}", f"optional    {RUN}"]
open(os.path.join(RUN_DIR, "environment.txt"), "a", encoding="utf-8").write("\n".join(_env) + "\n\n")
print(f"\n===== session started {datetime.datetime.now():%Y-%m-%d %H:%M} | run {RUN_TAG} | outputs: {RUN_DIR} =====")
print("\n".join("  " + l for l in _env))
if QUICK_TEST:
    print("\n  QUICK_TEST = True — pipeline check only; numbers are NOT results.")


## 2. Main cell — data, PIML and data-only network, evaluation, factor of safety
Data cleaning and the 2-min grid, the Richards-constrained network and the data-only baseline (same network and schedule, all λ = 0), test scores against persistence, pore pressure and FoS, figures fig01–fig11, and the FoS sensitivity to slope angle and the Ks proxy. The two trained models of this cell (seed 42) are the source models for transfer and the alert layer.


In [ ]:
# -*- coding: utf-8 -*-
"""
2. MAIN CELL — data, Richards-constrained network (PIML), data-only baseline, evaluation,
   pore pressure and factor of safety, figures.  Device 108 (source node).

Data (Section 2 of the paper)
  * Rain is rebuilt from the tipping-bucket counter (0.2794 mm per tip): the logged value is
    the running total since the last reset; positive steps are new rain. Input feature and
    surface-flux target use the 30-min rainfall depth R (mm), q_rain = R x 1e-3 / 1800 m/s.
  * Duplicate timestamps are dropped; the sensor warm-up at the start of the Dev108 record is
    trimmed; straight-line runs (values filled in by the logger between uploads) are removed.
  * Readings are put on a fixed 2-min grid. Outages <= 30 min are bridged with the last value,
    used only as lagged INPUTS (never as targets or in any score); longer outages start a new
    segment. Lag windows never cross a segment, and long outages count as 30 min on the model
    time axis.
  * Target: rolling median (5 points, 10 min) of θ at z = 0.30 m. Inputs: R(t) and 15 lags of
    θ (the previous 30 min). Split: chronological 80/10/10 by measured points, or the season
    split (SPLIT_MODE in the setup cell).

Model and loss  (z positive DOWNWARD, surface z = 0, base Z_MAX = 0.5 m)
  θ = NN(z, t, inputs), bounded to (θr + 0.01, θs − 0.01) by a sigmoid output.
  Richards residual (Eq. 8):  ∂θ/∂t − ∂/∂z [ K(ψ) (∂ψ/∂z − 1) ]  at z = 0.30 and 0.50 m,
  all derivatives by automatic differentiation; van Genuchten–Mualem ψ(θ), K(θ).
  Total loss (Eq. 13):
      L = L_data + λ_phys L_phys + λ_zvar L_zvar + λ_BC L_BC + λ_IC L_IC + λ_LBC L_LBC
    L_data : MSE at the sensor depth (scaled target)
    L_phys : mean squared Richards residual, normalised by (θs − θr) / (t_max − t_min)
    L_zvar : − variance of ∂θ/∂z across the collocation depths (prevents a depth-uniform solution)
    L_BC   : surface flux K(1 − ∂ψ/∂z) vs min(q_rain, Ks), normalised by Ks (λ_BC = 0 in the final setting)
    L_IC   : θ(z_sensor, t0) = median of the first 48 measured training values
    L_LBC  : free drainage at the base, ∂ψ/∂z = 0 (other conditions: boundary-mode cell)
  Data-only baseline: the SAME network, data and schedule with every λ = 0.
  Training: AdamW (warm-up + cosine) then full-batch L-BFGS (strong Wolfe); the checkpoint with
  the lowest validation loss is kept.

Slope stability: infinite slope, 33°, c' = 5 kPa, φ' = 28°, γ = 14.715 kN/m³, H = 1 m;
  pore pressure u = Se · γw · ψ, ψ bounded at −165 m.

Outputs (RUN_DIR): model checkpoints (.pt), fig01–fig11, fos_timeseries_main.csv,
  main_run_summary.csv, fos_sensitivity_ks.csv, fos_sensitivity_slope.csv,
  fos_sensitivity_grid_min.csv, run_log.txt.
"""
import warnings; warnings.filterwarnings("ignore")
import os, json, datetime
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler  # scaler_y only
from sklearn.metrics import r2_score, mean_squared_error
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ── Reproducibility ─────────────────────────────────────────
SEED = 42            # main run; the multi-seed cell uses BUDGET['seeds']
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

OUT = RUN_DIR   # set in the setup cell (one folder per SPLIT_MODE)
os.makedirs(OUT, exist_ok=True)

STYLE = {
    "font.family": "serif", "font.size": 10,
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "axes.labelsize": 11, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
    "xtick.major.size": 3.5, "ytick.major.size": 3.5,
    "legend.frameon": False, "legend.fontsize": 9,
    "figure.dpi": 300, "savefig.dpi": 300,
    "savefig.bbox": "tight", "savefig.facecolor": "white",
}
plt.rcParams.update(STYLE)

C = {
    "rain": "#4895EF", "obs": "#E63946", "pred": "#2EC4B6",
    "train": "#3A86FF", "val": "#FF006E", "test": "#FB5607",
    "psi": "#7209B7", "u": "#F77F00", "fos": "#06D6A0",
    "warn": "#EF233C", "grid": "#CCCCCC", "resid": "#480CA8",
    "phys": "#023E8A", "vg1": "#0077B6", "vg2": "#00B4D8",
    "coral": "#FF7F50",
}

def savefig(fig, name):
    path = os.path.join(OUT, name)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  → Saved: {path}")
    return path

def add_panel_label(ax, label, x=-0.08, y=1.06):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="top", ha="right")

def light_grid(ax, axis="y"):
    ax.grid(axis=axis, color=C["grid"], lw=0.5, ls="--", alpha=0.7)

# ═══════════════════════════════════════════════════════════
# VAN GENUCHTEN PARAMETERS  (Sandy Clay Loam, Device 108)
# ═══════════════════════════════════════════════════════════
VG = dict(
    theta_r=0.05, theta_s=0.55,
    alpha=5.9, n=1.48, m=1 - 1/1.48,
    Ks=0.0043,           # m/h
)
VG["Ks_ms"] = VG["Ks"] / 3600.0  # m/s

# Collocation depths (m, positive downward)
Z_SENSOR    = 0.30
Z_COLLOC    = np.array([ 0.30, 0.50], dtype=np.float32)

# ═══════════════════════════════════════════════════════════
# VAN GENUCHTEN TORCH UTILITIES
# ═══════════════════════════════════════════════════════════

def vg_Se_t(theta, vg):
    tc = theta.clamp(vg["theta_r"] + 1e-6, vg["theta_s"] - 1e-6)
    return (tc - vg["theta_r"]) / (vg["theta_s"] - vg["theta_r"])

def vg_psi_t(theta, vg):
    Se = vg_Se_t(theta, vg).clamp(1e-6, 1 - 1e-6)
    h  = (1.0 / vg["alpha"]) * (Se**(-1.0 / vg["m"]) - 1.0)**(1.0 / vg["n"])
    return -h

def vg_K_t(theta, vg):
    Se = vg_Se_t(theta, vg).clamp(1e-6, 1 - 1e-6)
    Kr = Se**0.5 * (1.0 - (1.0 - Se**(1.0 / vg["m"]))**vg["m"])**2
    return vg["Ks_ms"] * Kr

# NumPy equivalents (evaluation only)
def vg_Se_np(theta, vg):
    tc = np.clip(theta, vg["theta_r"]+1e-6, vg["theta_s"]-1e-6)
    return (tc - vg["theta_r"]) / (vg["theta_s"] - vg["theta_r"])

def vg_psi_np(theta, vg):
    Se = np.clip(vg_Se_np(theta, vg), 1e-6, 1-1e-6)
    return -(1.0/vg["alpha"]) * (Se**(-1.0/vg["m"]) - 1.0)**(1.0/vg["n"])

def vg_Kr_np(theta, vg):
    Se = np.clip(vg_Se_np(theta, vg), 1e-6, 1-1e-6)
    return Se**0.5 * (1-(1-Se**(1/vg["m"]))**vg["m"])**2

def vg_K_np(theta, vg):
    return vg["Ks_ms"] * vg_Kr_np(theta, vg)

# ═══════════════════════════════════════════════════════════
# 1-2.  LOAD & PREPROCESS  (Device 108)
# ═══════════════════════════════════════════════════════════
print("\n[Step 1-2] Loading data (full year, fixed 2-min grid) ...")
# ── Rainfall reconstruction ─────────────────────────
# The logged `rain` value is NOT an intensity. It is the tipping-bucket total
# (0.2794 mm per tip) accumulated since the last counter reset: it rises while it
# rains, HOLDS its value after the rain stops, and drops at the reset. We rebuild
# per-sample increments and use the 30-min rainfall depth R (mm per 30 min) as the
# input feature and in the surface-flux target, q_rain = R x 1e-3 / 1800 [m/s].
def counter_to_increments(c, eps=1e-6):
    """Positive steps of the counter are new rain; at each reset the first
    post-reset value is counted as rain already fallen in the new cycle."""
    c = np.asarray(c, float); inc = np.zeros(len(c)); falling = False
    for i in range(1, len(c)):
        d = c[i] - c[i - 1]
        if d < -eps:
            falling = True
        else:
            if falling:
                inc[i - 1] += c[i - 1]; falling = False
            inc[i] += max(d, 0.0)
    return inc

def flag_linear_fill(theta, min_run=10, tol=1e-7):
    """Straight-line runs (constant non-zero slope for >= min_run steps) are values
    filled in between logger uploads, not measurements. They are dropped."""
    x = np.asarray(theta, float); d1 = np.diff(x)
    lin = (np.abs(np.diff(d1)) < tol) & (np.abs(d1[1:]) > tol)
    out = np.zeros(len(x), bool); i = 0
    while i < len(lin):
        if lin[i]:
            j = i
            while j < len(lin) and lin[j]: j += 1
            if j - i >= min_run: out[i:j + 2] = True
            i = j
        else:
            i += 1
    return out

GRID   = "2min"    # fixed time step: 15 lags = the previous 30 minutes, in every season
GAP_S  = 1800      # outages up to 30 min are bridged; a longer outage starts a new segment
WARMUP_THETA_OK = 0.40

def prepare_site(csv, dev_id, warmup_theta=None, label=""):
    """Raw telemetry -> fixed 2-min grid with 30-min rain depth, segments and gap-compressed time."""
    raw = pd.read_csv(csv, low_memory=False)
    if "theta" not in raw.columns: raw = raw.rename(columns={"soil": "theta"})
    if "devID" in raw.columns: raw = raw[raw["devID"].astype(int) == dev_id]
    raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="mixed", errors="coerce")
    raw = raw.dropna(subset=["timestamp", "theta", "rain"]).sort_values("timestamp")
    n0 = len(raw); raw = raw.drop_duplicates("timestamp", keep="last").reset_index(drop=True)
    n_dup = n0 - len(raw)
    raw["rain_inc"] = counter_to_increments(raw["rain"].values)
    raw["R30"] = raw.set_index("timestamp")["rain_inc"].rolling("30min").sum().values   # mm / 30 min
    raw["fill"] = flag_linear_fill(raw["theta"].values)
    i0 = int(np.argmax(raw["theta"].values >= warmup_theta)) if warmup_theta else 0     # sensor warm-up
    raw = raw.iloc[i0:].reset_index(drop=True)
    raw["bin"] = raw["timestamp"].dt.floor(GRID)
    g = (raw.groupby("bin").agg(theta=("theta", "last"), rain=("R30", "last"),
                                rain_inc=("rain_inc", "sum"), fill=("fill", "all"))
            .reset_index().rename(columns={"bin": "timestamp"}))
    n_fill = int(g["fill"].sum())
    g = g.loc[~g["fill"]].drop(columns="fill").reset_index(drop=True)
    _dt = g["timestamp"].diff().dt.total_seconds().fillna(0.0)
    g["seg"] = (_dt > GAP_S).cumsum().astype(int)
    # Short outages (LoRaWAN packet loss, <= 30 min): every missing 2-min point gets the last
    # received reading, as the operational system would see it. These "held" points are used
    # only as lagged INPUTS, never as training targets or in any score.
    parts = []
    for _, sg in g.groupby("seg"):
        full = pd.date_range(sg["timestamp"].iloc[0], sg["timestamp"].iloc[-1], freq=GRID)
        sg = sg.set_index("timestamp").reindex(full)
        sg["held"] = sg["theta"].isna()
        sg[["theta", "rain", "seg"]] = sg[["theta", "rain", "seg"]].ffill()
        sg["rain_inc"] = sg["rain_inc"].fillna(0.0)
        parts.append(sg.rename_axis("timestamp").reset_index())
    n_real = len(g); g = pd.concat(parts, ignore_index=True); g["seg"] = g["seg"].astype(int)
    _dt = g["timestamp"].diff().dt.total_seconds().fillna(0.0)
    g["t_sec"] = _dt.clip(upper=GAP_S).cumsum()          # long outages count as GAP_S on the model time axis
    print(f"  [{label}] {n0:,} raw rows | {n_dup:,} duplicates | "
          f"warm-up {i0} rows | {n_real:,} measured 2-min points ({n_fill:,} filled-in points dropped, "
          f"{int(g['held'].sum()):,} short-outage points held as inputs only) | "
          f"{g['timestamp'].min():%Y-%m-%d} -> {g['timestamp'].max():%Y-%m-%d}")
    print(f"  [{label}] rain {raw['rain_inc'].sum():.0f} mm | max 30-min depth {raw['R30'].max():.1f} mm | "
          f"θ {g['theta'].min():.3f}–{g['theta'].max():.3f} | {int(g['seg'].max()) + 1} segments, "
          f"longest gap {_dt.max() / 86400:.1f} d")
    return g

df = prepare_site(DATA_CSV, 108, warmup_theta=WARMUP_THETA_OK, label="Dev108")

LAG = 15
_cum = np.cumsum(~df["held"].values)                      # split by MEASURED points, not held ones
_at  = lambda k: int(np.searchsorted(_cum, k))
if SPLIT_MODE == "season":
    TEST_START = pd.Timestamp("2026-05-01")              # SW monsoon: wetting of dry soil
    n_pre = int((df["timestamp"] < TEST_START).sum())
    n_tr, n_val = _at(0.85 * _cum[n_pre - 1]), n_pre      # validation = last 15 % before May
else:
    n_tr, n_val = _at(0.80 * _cum[-1]), _at(0.90 * _cum[-1])   # chronological 80 / 10 / 10

df_tr  = df.iloc[:n_tr].copy().reset_index(drop=True)
df_val = df.iloc[n_tr:n_val].copy().reset_index(drop=True)
df_te  = df.iloc[n_val:].copy().reset_index(drop=True)
for _nm, _d in [("train", df_tr), ("val", df_val), ("test", df_te)]:
    print(f"  {_nm:5s}: {_d['timestamp'].min():%Y-%m-%d} -> {_d['timestamp'].max():%Y-%m-%d} | "
          f"{int((~_d['held']).sum()):,} measured points | θ {_d['theta'].min():.3f}–{_d['theta'].max():.3f} | rain {_d['rain_inc'].sum():.0f} mm")

THETA_RAW_BY_TS = df.set_index("timestamp")["theta"].copy()   # unsmoothed sensor theta (raw-reading check)
ROLL_W = 5
for _d in [df_tr, df_val, df_te]:
    _d["theta"] = (_d.groupby("seg")["theta"]
                     .transform(lambda s: s.rolling(ROLL_W, center=False, min_periods=1).median()))

def build_features(d, lag=1):
    th, seg = d["theta"].values.astype(np.float64), d["seg"].values
    idx = np.arange(lag, len(d))
    idx = idx[seg[idx] == seg[idx - lag]]          # lag window would cross a data gap
    if "held" in d: idx = idx[~d["held"].values[idx].astype(bool)]   # held points are never targets
    lags = np.stack([th[idx - k] for k in range(1, lag + 1)], axis=1)
    X = np.column_stack([d["rain"].values[idx], lags]).astype(np.float32)
    y = th[idx].astype(np.float32).reshape(-1, 1)
    return (X, y, d["rain"].values[idx].astype(np.float32), th[idx].astype(np.float32),
            d["t_sec"].values[idx].astype(np.float32), d["timestamp"].values[idx])

X_tr_raw,  y_tr_raw,  rain_tr,  theta_tr,  t_sec_tr,  ts_tr  = build_features(df_tr,  LAG)
X_val_raw, y_val_raw, rain_val, theta_val, t_sec_val, ts_val  = build_features(df_val, LAG)
X_te_raw,  y_te_raw,  rain_te,  theta_te,  t_sec_te,  ts_te   = build_features(df_te,  LAG)

# ── Physics-based input scaling ──────────────────────────────────
# Rain (30-min depth, mm) is divided by a fixed bound RAIN_MAX; lagged θ is scaled with the
# van Genuchten θr / θs, so the scaling does not depend on the training data.
RAIN_MAX = 75.0   # mm per 30 min — feature-scaling constant (observed max 30-min depth ~30 mm)

def physics_normalize_X(X_raw):
    """Rain / RAIN_MAX; lagged θ -> effective saturation with the VG bounds."""
    X_norm = X_raw.copy()
    X_norm[:, 0]  = X_raw[:, 0] / RAIN_MAX                                           # rain
    X_norm[:, 1:] = (X_raw[:, 1:] - VG["theta_r"]) / (VG["theta_s"] - VG["theta_r"])  # lags
    return X_norm.astype(np.float32)

scaler_y = MinMaxScaler().fit(y_tr_raw)  # target scaling, fitted on the training split only

X_tr  = physics_normalize_X(X_tr_raw)
X_val = physics_normalize_X(X_val_raw)
X_te  = physics_normalize_X(X_te_raw)
y_tr  = scaler_y.transform(y_tr_raw).astype(np.float32)
y_val = scaler_y.transform(y_val_raw).astype(np.float32)
y_te  = scaler_y.transform(y_te_raw).astype(np.float32)

X_np      = np.concatenate([X_tr_raw, X_val_raw, X_te_raw], axis=0)
theta_np  = np.concatenate([theta_tr,  theta_val,  theta_te],  axis=0)
rain_np   = np.concatenate([rain_tr,   rain_val,   rain_te],   axis=0)
t_sec_np  = np.concatenate([t_sec_tr,  t_sec_val,  t_sec_te],  axis=0)
ts_all_np = np.concatenate([ts_tr,     ts_val,     ts_te],     axis=0)
X_norm    = physics_normalize_X(X_np)

T_MIN   = float(t_sec_tr.min())
T_MAX   = float(t_sec_tr.max()) + 1e-8
Z_MAX   = float(Z_COLLOC.max()) + 1e-8

def to_t(a): return torch.tensor(a, dtype=torch.float32).to(DEVICE)

Xt, yt   = to_t(X_tr),  to_t(y_tr)
Xv, yv   = to_t(X_val), to_t(y_val)
Xte, yte = to_t(X_te),  to_t(y_te)

rain_t  = to_t(rain_tr[:, None])
t_sec_t = to_t(t_sec_tr[:, None])
dt_tr   = np.diff(t_sec_tr, prepend=t_sec_tr[0]).clip(min=1.0)
dt_t    = to_t(dt_tr[:, None])

# ═══════════════════════════════════════════════════════════
# 3.  MODEL
# ═══════════════════════════════════════════════════════════
print("\n[Step 3] Building PINN (true autograd Richards) ...")

ARCH = globals().get("ARCH", dict(depth=5, width=128, act="tanh", taper=True))   # set in the setup cell

class RichardsPINN(nn.Module):
    """θ(z, t, features). `depth` hidden layers of `width`, plus one of width//2 if `taper`.
    Same construction as the architecture search, so the selected network is reproduced exactly."""
    def __init__(self, feat_dim, width=ARCH["width"], depth=ARCH["depth"], act=ARCH["act"], taper=ARCH["taper"]):
        super().__init__()
        A = {"tanh": nn.Tanh, "silu": nn.SiLU}[act]; d, L = feat_dim + 2, []
        for _ in range(depth): L += [nn.Linear(d, width), A()]; d = width
        if taper: L += [nn.Linear(d, width // 2), A()]; d = width // 2
        L += [nn.Linear(d, 1), nn.Sigmoid()]
        self.net = nn.Sequential(*L)
        self.theta_lo = VG["theta_r"] + 0.01
        self.theta_hi = VG["theta_s"] - 0.01

    def forward(self, z_norm, t_norm, feat):
        x   = torch.cat([z_norm, t_norm, feat], dim=1)
        raw = self.net(x)
        theta = self.theta_lo + (self.theta_hi - self.theta_lo) * raw
        return theta

model = RichardsPINN(feat_dim=X_tr.shape[1]).to(DEVICE)
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ═══════════════════════════════════════════════════════════
# 4.  PHYSICS LOSS (Richards residual, surface flux)
# ═══════════════════════════════════════════════════════════

def richards_residual(model, z_norm, t_norm, feat):
    """
    Compute 1D Richards PDE residual + dθ/dz for z-gradient penalty.

    Returns
    -------
    residual   : (B,1)  m³/m³/s
    theta      : (B,1)  predicted θ̂
    dtheta_dz  : (B,1)  ∂θ/∂z (used by the z-gradient term L_zvar)
    """
    theta = model(z_norm, t_norm, feat)

    dtheta_dt = torch.autograd.grad(
        outputs=theta, inputs=t_norm,
        grad_outputs=torch.ones_like(theta),
        create_graph=True, retain_graph=True
    )[0]

    psi = vg_psi_t(theta, VG)

    dpsi_dz = torch.autograd.grad(
        outputs=psi, inputs=z_norm,
        grad_outputs=torch.ones_like(psi),
        create_graph=True, retain_graph=True
    )[0]

    K = vg_K_t(theta, VG)

    dpsi_dz_phys = dpsi_dz / Z_MAX
    flux = K * (dpsi_dz_phys - 1.0)   # z positive downward: H = psi - z

    dflux_dz = torch.autograd.grad(
        outputs=flux, inputs=z_norm,
        grad_outputs=torch.ones_like(flux),
        create_graph=True, retain_graph=True
    )[0]

    dflux_dz_phys = dflux_dz / Z_MAX
    dtheta_dt_phys = dtheta_dt / (T_MAX - T_MIN)
    residual = dtheta_dt_phys - dflux_dz_phys

    # ── ∂θ/∂z — for z-gradient variance penalty ─────
    dtheta_dz = torch.autograd.grad(
        outputs=theta, inputs=z_norm,
        grad_outputs=torch.ones_like(theta),
        create_graph=True, retain_graph=True
    )[0]

    return residual, theta, dtheta_dz


def surface_bc_flux(model, z_norm, t_norm, feat):
    """
    Predicted Darcy flux at the soil surface (z = 0).

    Mirrors the upper (flux) boundary condition of the Richards equation
    (Eq. 5). z is measured positive DOWNWARD from the surface (z = 0), so the
    total head is H = psi - z and the downward Darcy flux is
        q_down = -K dH/dz = K (1 - dpsi/dz).                      
    The interior flux in richards_residual() is K(psi)(dpsi/dz - 1) = -q_down,
    so the PDE residual (Eq. 8) and this boundary term (Eq. 11) share one sign
    convention.

    Returns
    -------
    q_pred : (B,1)  predicted surface infiltration flux, m/s  [Eq. 12]
    """
    theta = model(z_norm, t_norm, feat)
    psi   = vg_psi_t(theta, VG)

    dpsi_dz = torch.autograd.grad(
        outputs=psi, inputs=z_norm,
        grad_outputs=torch.ones_like(psi),
        create_graph=True, retain_graph=True
    )[0]

    K = vg_K_t(theta, VG)
    dpsi_dz_phys = dpsi_dz / Z_MAX
    q_pred = K * (1.0 - dpsi_dz_phys)   # downward infiltration flux, z positive downward
    return q_pred


# ═══════════════════════════════════════════════════════════
# 5.  COLLOCATION POINT BUILDER
# ═══════════════════════════════════════════════════════════

def make_colloc_batch(t_sec_batch, rain_batch, feat_batch):
    Nz = len(Z_COLLOC)
    B  = t_sec_batch.shape[0]
    z_phys   = np.tile(Z_COLLOC, B).reshape(-1, 1).astype(np.float32)
    z_norm_c = z_phys / Z_MAX
    t_phys_c = np.repeat(t_sec_batch, Nz).reshape(-1, 1).astype(np.float32)
    t_norm_c = (t_phys_c - T_MIN) / (T_MAX - T_MIN)
    feat_c   = np.repeat(feat_batch, Nz, axis=0).astype(np.float32)
    z_c    = torch.tensor(z_norm_c, dtype=torch.float32,
                          device=DEVICE, requires_grad=True)
    t_c    = torch.tensor(t_norm_c, dtype=torch.float32,
                          device=DEVICE, requires_grad=True)
    feat_c = torch.tensor(feat_c,   dtype=torch.float32, device=DEVICE)
    return z_c, t_c, feat_c


# ═══════════════════════════════════════════════════════════
# 6.  TRAINING LOOP
# ═══════════════════════════════════════════════════════════
print("\n[Step 4-6] Training PINN with true Richards autograd loss ...")
EPOCHS   = BUDGET["main_adam"] + BUDGET["main_lbfgs"]   # 400 Adam + 200 L-BFGS
LR       = 5e-4
LAM      = LAM_FINAL       # λ_phys (setup cell)
BATCH    = 512
WARMUP   = 100
LAM_ZVAR = 0.1   # z-gradient term weight
LAM_BC   = LAM_BC_FINAL    # λ_BC, surface-flux term (setup cell; 0 = off)
LAM_IC   = 0.1  # initial-condition loss weight (sensor depth)
LAM_LBC  = 0.1  # lower-boundary (free drainage) loss weight
N_IC     = 48   # first N_IC post-warm-up training samples define t0 / theta0
IC_MODE  = "sensor"          # "sensor" (data only) | "uniform" (assumed profile, sensitivity)
LBC_MODE = "free_drainage"   # dpsi/dz = 0 at z = Z_MAX

# Adam (mini-batch, warm-up + cosine) followed by full-batch L-BFGS. The same schedule is used
# for the PIML and the data-only network, so the two differ only in the loss.
ADAM_EPOCHS  = BUDGET["main_adam"]
LBFGS_EPOCHS = BUDGET["main_lbfgs"]
assert ADAM_EPOCHS + LBFGS_EPOCHS == EPOCHS, "Adam+L-BFGS split must sum to EPOCHS"
LBFGS_MAX_ITER = 2     # internal line-search iterations per L-BFGS step
LBFGS_HISTORY  = 50     # curvature history size

# ── Physical scale for residual normalization ───────
THETA_RANGE = VG["theta_s"] - VG["theta_r"]          # 0.50 m³/m³
T_RANGE_SEC = float(T_MAX - T_MIN)                    # seconds
DTHDT_SCALE = THETA_RANGE / T_RANGE_SEC               # typical ∂θ/∂t, m³/m³/s
print(f"  ∂θ/∂t scale for normalization: {DTHDT_SCALE:.4e} m³/m³/s")

# ── Rainfall → surface flux (Eq. 12) ──────────────────────────
# R is the 30-min rainfall depth (mm) rebuilt from the tipping-bucket counter:
# q_rain [m/s] = R [mm] * 1e-3 [m/mm] / 1800 [s]
RAIN_MM_TO_M     = 1e-3
RAIN_WINDOW_S    = 1800.0          # R is a 30-min depth -> divide by 1800 s
BC_FLUX_SCALE = VG["Ks_ms"]   # normalizes q_pred/q_rain residual to O(1), like DTHDT_SCALE does for L_phys

y_min_t = torch.tensor(scaler_y.data_min_[0], dtype=torch.float32, device=DEVICE)
y_scl_t = torch.tensor(scaler_y.scale_[0],    dtype=torch.float32, device=DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return float(epoch + 1) / float(WARMUP)
    progress = (epoch - WARMUP) / max(1, ADAM_EPOCHS - WARMUP)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
mse_loss  = nn.MSELoss()

z_sens_norm = torch.tensor([[Z_SENSOR / Z_MAX]], dtype=torch.float32, device=DEVICE)

t_tr_norm   = ((t_sec_tr  - T_MIN) / (T_MAX - T_MIN)).astype(np.float32)
t_tr_norm_t = to_t(t_tr_norm[:, None])

dataset = TensorDataset(Xt, yt, rain_t, t_tr_norm_t, dt_t,
                        to_t(t_sec_tr[:, None]))
loader  = DataLoader(dataset, batch_size=BATCH, shuffle=True,
                     generator=torch.Generator().manual_seed(SEED))

# ═══════════════════════════════════════════════════════════
#  INITIAL CONDITION + LOWER BOUNDARY (free drainage)
# Both are soft penalties added to the total loss (Eq. 13).
# ═══════════════════════════════════════════════════════════
IC_THETA0 = float(np.median(theta_tr[:N_IC]))   # measured theta at z = 0.30 m, post warm-up
print(f"  [IC] theta0 (sensor, z={Z_SENSOR} m) = {IC_THETA0:.4f} m³/m³  (mode={IC_MODE})")

def ic_loss(m, mode=IC_MODE, n_ic=N_IC):
    """IC at t ~ T_MIN. "sensor": theta(z_s, t0) = measured theta0 (only observed depth).
    "uniform": same theta0 imposed at z = 0, z_s, Z_COLLOC.max() (ASSUMED profile)."""
    depths = [Z_SENSOR] if mode == "sensor" else [0.0, Z_SENSOR, float(Z_COLLOC.max())]
    loss = torch.zeros((), device=DEVICE)
    for zd in depths:
        z = torch.full((n_ic, 1), zd / Z_MAX, dtype=torch.float32, device=DEVICE)
        pred = m(z, t_tr_norm_t[:n_ic], Xt[:n_ic])
        loss = loss + (((pred - IC_THETA0) / THETA_RANGE) ** 2).mean()
    return loss / len(depths)

def lower_bc_free_drainage_loss(m, t_norm, feat):
    """Free drainage at z = Z_MAX: unit total hydraulic gradient <=> dpsi/dz = 0
    (independent of the gravity-sign convention). Loss in (m/m)^2."""
    B = t_norm.shape[0]
    z = torch.ones(B, 1, dtype=torch.float32, device=DEVICE).requires_grad_(True)
    theta = m(z, t_norm, feat)
    psi = vg_psi_t(theta, VG)
    dpsi_dz = torch.autograd.grad(psi.sum(), z, create_graph=True)[0] / Z_MAX
    return (dpsi_dz ** 2).mean()

data_losses  = []
phys_losses  = []
bc_losses    = []   # surface-flux loss history (Eq. 11)
ic_losses    = []   # initial-condition loss history
lbc_losses   = []   # lower-boundary (free drainage) loss history
train_losses = []
val_losses   = []
val_phys_losses  = []   # val-set physics (Richards residual) loss
val_total_losses = []   # val data + LAM * val physics
lr_history   = []

best_val   = np.inf
best_state = None

z_v_rep  = z_sens_norm.expand(Xv.shape[0], -1).detach()
t_v_norm = to_t(((t_sec_val - T_MIN) / (T_MAX - T_MIN))[:, None])

if PREP_ONLY:
    print("\nPREP_ONLY = True: data, model functions and loss terms are ready; main training skipped "
          "(saved models and figures untouched). Now run the boundary-mode cell, then the cell you need.")
    raise SystemExit("stopped after data preparation (PREP_ONLY = True) — this is expected")

# ── REUSE_STORED: use the stored checkpoint instead of training ──────────────
_ckpt_pinn = os.path.join(OUT, "piml_slope_model_108_pinn_richards.pt")
_ckpt_ml   = os.path.join(OUT, "piml_slope_model_108_ml_baseline_final.pt")
LOAD_STORED = bool(REUSE_STORED) and os.path.exists(_ckpt_pinn) and os.path.exists(_ckpt_ml)
if LOAD_STORED:
    print("\nREUSE_STORED = True: loading the stored PIML checkpoint, no training")
    _ck = torch.load(_ckpt_pinn, map_location=DEVICE, weights_only=False)
    best_state = {k: v.to(DEVICE) for k, v in _ck["model_state_dict"].items()}
    train_losses, val_losses, data_losses = _ck["train_losses"], _ck["val_losses"], _ck["data_losses"]
    phys_losses, bc_losses = _ck["phys_losses"], _ck["bc_losses"]
    ic_losses, lbc_losses = _ck["ic_losses"], _ck["lbc_losses"]
    val_phys_losses, val_total_losses = _ck["val_phys_losses"], _ck["val_total_losses"]
    lr_history, best_val = _ck["lr_history"], _ck["best_val_loss"]
    model.load_state_dict(best_state)
if not LOAD_STORED:
    for epoch in range(1, ADAM_EPOCHS + 1):
        model.train()
        ep_data  = 0.0
        ep_phys  = 0.0
        ep_bc    = 0.0
        ep_ic    = 0.0
        ep_lbc   = 0.0

        for (Xb, yb, rb, tb_norm, dtb, tb_sec) in loader:
            optimizer.zero_grad()
            B = Xb.shape[0]

            # ── Data loss at sensor depth ─────────────────────────
            z_d = z_sens_norm.expand(B, -1).requires_grad_(True)
            t_d = tb_norm.clone().requires_grad_(True)
            theta_pred = model(z_d, t_d, Xb)

            pred_norm = (theta_pred - y_min_t) * y_scl_t
            loss_d = mse_loss(pred_norm, yb)

            # ── Physics loss at collocation depths ────────────────
            tb_sec_np = tb_sec.detach().cpu().numpy().flatten()
            Xb_np     = Xb.detach().cpu().numpy()
            z_c, t_c, feat_c = make_colloc_batch(tb_sec_np, None, Xb_np)

            # richards_residual now returns 3 values
            resid, _, dtheta_dz = richards_residual(model, z_c, t_c, feat_c)

            # ── Normalize residual → dimensionless ────────
            resid_norm = resid / (DTHDT_SCALE + 1e-12)
            loss_p = (resid_norm ** 2).mean()

            # ── Z-gradient variance penalty ───────────────
            # Reshape to (B, Nz), penalize if all depths have same dθ/dz
            Nz = len(Z_COLLOC)
            dtheta_dz_r = dtheta_dz.view(B, Nz)          # (B, Nz)
            z_var_loss  = -dtheta_dz_r.var(dim=1).mean()  # maximize z-variance

            # ── Surface-flux loss at z = 0 (Eq. 5, 11, 12) ───────────────
            # Same batch times/inputs as the data loss, evaluated at the surface. The target is
            # capped at Ks (infiltration excess runs off): q_target = min(q_rain, Ks). This is the
            # unit-gradient simplification; it is weighted by LAM_BC (0 in the final setting).
            z_bc = torch.zeros(B, 1, dtype=torch.float32, device=DEVICE).requires_grad_(True)
            t_bc = tb_norm.clone().requires_grad_(True)
            q_pred_bc   = surface_bc_flux(model, z_bc, t_bc, Xb)                  # [Eq. 12] m/s
            q_rain_bc   = (rb * RAIN_MM_TO_M) / RAIN_WINDOW_S                     # raw rainfall flux, m/s
            q_target_bc = torch.clamp(q_rain_bc, max=BC_FLUX_SCALE)              # Ks-capped target, m/s
            loss_bc     = (((q_pred_bc - q_target_bc) / (BC_FLUX_SCALE + 1e-12)) ** 2).mean()  # [Eq. 11]

            # ── initial condition + lower (free-drainage) boundary ──
            loss_ic  = ic_loss(model)
            loss_lbc = lower_bc_free_drainage_loss(model, tb_norm, Xb)

            loss = (loss_d + LAM * loss_p + LAM_ZVAR * z_var_loss + LAM_BC * loss_bc
                    + LAM_IC * loss_ic + LAM_LBC * loss_lbc)  # [Eq. 13, extended]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            ep_data += loss_d.item()
            ep_phys += loss_p.item()
            ep_bc   += loss_bc.item()
            ep_ic   += loss_ic.item()
            ep_lbc  += loss_lbc.item()

        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]

        model.eval()
        with torch.no_grad():
            vl = mse_loss(
                (model(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t,
                yv
            ).item()

        # Validation physics loss: same Richards residual on the validation inputs. Autograd is
        # needed for the derivatives, nothing is back-propagated.
        with torch.enable_grad():
            z_c_val, t_c_val, feat_c_val = make_colloc_batch(t_sec_val, None, X_val)
            resid_val, _, _ = richards_residual(model, z_c_val, t_c_val, feat_c_val)
            resid_val_norm = resid_val / (DTHDT_SCALE + 1e-12)
            vl_phys = (resid_val_norm ** 2).mean().item()
        model.train()

        n_batches = len(loader)
        data_losses.append(ep_data / n_batches)
        phys_losses.append(ep_phys / n_batches)
        bc_losses.append(ep_bc / n_batches)
        ic_losses.append(ep_ic / n_batches)
        lbc_losses.append(ep_lbc / n_batches)
        train_losses.append((ep_data + LAM * ep_phys + LAM_BC * ep_bc
                             + LAM_IC * ep_ic + LAM_LBC * ep_lbc) / n_batches)
        val_losses.append(vl)
        val_phys_losses.append(vl_phys)
        val_total_losses.append(vl + LAM * vl_phys)
        lr_history.append(current_lr)

        if vl < best_val:
            best_val   = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 100 == 0:
            # λ·Phys / Data ratio (balance of the two loss terms)
            ratio = (LAM * phys_losses[-1]) / (data_losses[-1] + 1e-15)
            print(f"  Epoch {epoch:4d} | Data {data_losses[-1]:.5f} | "
                  f"Phys(norm) {phys_losses[-1]:.5f} | BC(norm) {bc_losses[-1]:.5f} | "
                  f"IC {ic_losses[-1]:.5f} | LBC {lbc_losses[-1]:.5f} | "
                  f"λ·Phys/Data ratio: {ratio:.3f} | "
                  f"Val(data) {vl:.5f} | Val(phys) {vl_phys:.5f} | LR {current_lr:.2e}")

    # ── L-BFGS phase (PIML: same total loss, full batch) ────────
    print(f"\n[Step 6b] Switching to L-BFGS fine-tuning "
          f"({LBFGS_EPOCHS} steps, full-batch) ...")
    if best_state is not None:
        model.load_state_dict(best_state)  # resume from Adam's best point, not the last

    optimizer_lbfgs = optim.LBFGS(
        model.parameters(), lr=1.0,
        max_iter=LBFGS_MAX_ITER, history_size=LBFGS_HISTORY,
        line_search_fn="strong_wolfe",
    )

    # L-BFGS performs best full-batch (its curvature estimate assumes a
    # consistent objective across calls) — mini-batching would make the
    # line search chase a moving target, so we use the whole training set.
    tb_sec_full_np = t_sec_tr[:, None].astype(np.float32)
    Xb_full_np     = X_tr

    def lbfgs_closure():
        optimizer_lbfgs.zero_grad()
        B = Xt.shape[0]

        z_d = z_sens_norm.expand(B, -1).requires_grad_(True)
        t_d = t_tr_norm_t.clone().requires_grad_(True)
        theta_pred = model(z_d, t_d, Xt)
        pred_norm = (theta_pred - y_min_t) * y_scl_t
        loss_d = mse_loss(pred_norm, yt)

        z_c, t_c, feat_c = make_colloc_batch(tb_sec_full_np.flatten(), None, Xb_full_np)
        resid, _, dtheta_dz = richards_residual(model, z_c, t_c, feat_c)
        resid_norm = resid / (DTHDT_SCALE + 1e-12)
        loss_p = (resid_norm ** 2).mean()

        Nz = len(Z_COLLOC)
        dtheta_dz_r = dtheta_dz.view(B, Nz)
        z_var_loss  = -dtheta_dz_r.var(dim=1).mean()

        z_bc = torch.zeros(B, 1, dtype=torch.float32, device=DEVICE).requires_grad_(True)
        t_bc = t_tr_norm_t.clone().requires_grad_(True)
        q_pred_bc   = surface_bc_flux(model, z_bc, t_bc, Xt)
        q_rain_bc   = (rain_t * RAIN_MM_TO_M) / RAIN_WINDOW_S
        q_target_bc = torch.clamp(q_rain_bc, max=BC_FLUX_SCALE)
        loss_bc     = (((q_pred_bc - q_target_bc) / (BC_FLUX_SCALE + 1e-12)) ** 2).mean()

        loss_ic  = ic_loss(model)
        loss_lbc = lower_bc_free_drainage_loss(model, t_tr_norm_t, Xt)

        loss = (loss_d + LAM * loss_p + LAM_ZVAR * z_var_loss + LAM_BC * loss_bc
                + LAM_IC * loss_ic + LAM_LBC * loss_lbc)
        loss.backward()
        lbfgs_closure.last = (loss_d.item(), loss_p.item(), loss_bc.item(),
                              loss_ic.item(), loss_lbc.item())
        return loss

    for step in range(1, LBFGS_EPOCHS + 1):
        optimizer_lbfgs.step(lbfgs_closure)
        ld, lp, lbc, lic, llbc = lbfgs_closure.last

        model.eval()
        with torch.no_grad():
            vl = mse_loss(
                (model(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t, yv
            ).item()

        # validation physics loss, as in the Adam phase
        with torch.enable_grad():
            z_c_val, t_c_val, feat_c_val = make_colloc_batch(t_sec_val, None, X_val)
            resid_val, _, _ = richards_residual(model, z_c_val, t_c_val, feat_c_val)
            resid_val_norm = resid_val / (DTHDT_SCALE + 1e-12)
            vl_phys = (resid_val_norm ** 2).mean().item()
        model.train()

        epoch = ADAM_EPOCHS + step
        data_losses.append(ld)
        phys_losses.append(lp)
        bc_losses.append(lbc)
        ic_losses.append(lic)
        lbc_losses.append(llbc)
        train_losses.append(ld + LAM * lp + LAM_BC * lbc + LAM_IC * lic + LAM_LBC * llbc)
        val_losses.append(vl)
        val_phys_losses.append(vl_phys)
        val_total_losses.append(vl + LAM * vl_phys)
        lr_history.append(np.nan)  # L-BFGS has no single scalar LR (strong-Wolfe line search)

        if vl < best_val:
            best_val   = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if step % 20 == 0 or step == LBFGS_EPOCHS:
            print(f"  [L-BFGS] Step {step:3d}/{LBFGS_EPOCHS} (epoch {epoch}) | "
                  f"Data {ld:.5f} | Phys(norm) {lp:.5f} | BC(norm) {lbc:.5f} | "
                  f"IC {lic:.5f} | LBC {llbc:.5f} | Val(data) {vl:.5f} | Val(phys) {vl_phys:.5f}")

    model.load_state_dict(best_state)

# ── Post-training IC and lower-boundary residuals ──
# A soft penalty is only meaningful if its residual is small after training.
model.eval()
with torch.enable_grad():
    _idx = np.arange(0, len(t_sec_np), 5)
    _t = to_t(((t_sec_np[_idx] - T_MIN) / (T_MAX - T_MIN))[:, None])
    _X = to_t(X_norm[_idx])
    _z = torch.ones(len(_idx), 1, dtype=torch.float32, device=DEVICE).requires_grad_(True)
    _th = model(_z, _t, _X)
    _dpsi = torch.autograd.grad(vg_psi_t(_th, VG).sum(), _z)[0] / Z_MAX
    _q_over_Ks = (vg_K_t(_th, VG) * (1.0 - _dpsi) / BC_FLUX_SCALE).detach().cpu().numpy().flatten()
    _dpsi_np = _dpsi.detach().cpu().numpy().flatten()
    _th0_pred = model(torch.full((N_IC, 1), Z_SENSOR / Z_MAX, dtype=torch.float32, device=DEVICE),
                      t_tr_norm_t[:N_IC], Xt[:N_IC]).detach().cpu().numpy().flatten()
BCIC_DIAG = {
    "lbc_mode": LBC_MODE, "ic_mode": IC_MODE, "z_bottom_m": float(Z_MAX),
    "lbc_rms_dpsi_dz": float(np.sqrt(np.mean(_dpsi_np ** 2))),
    "lbc_mean_abs_dpsi_dz": float(np.mean(np.abs(_dpsi_np))),
    "lbc_mean_q_bottom_over_Ks": float(np.mean(_q_over_Ks)),
    "ic_theta0_sensor": IC_THETA0,
    "ic_mean_abs_err_sensor": float(np.mean(np.abs(_th0_pred - IC_THETA0))),
}
print("\n  [IC / lower-BC diagnostics]", {k: (round(v, 5) if isinstance(v, float) else v)
                                          for k, v in BCIC_DIAG.items()})
model.train()

print(f"\n  Best val: {best_val:.5f}  @ epoch {int(np.argmin(val_losses))+1}")

def predict_sensor(X_norm_np, t_sec_arr):
    model.eval()
    N = X_norm_np.shape[0]
    z_s  = z_sens_norm.expand(N, -1)
    t_s  = to_t(((t_sec_arr - T_MIN) / (T_MAX - T_MIN))[:, None])
    Xn   = to_t(X_norm_np)
    with torch.no_grad():
        return model(z_s, t_s, Xn).cpu().numpy().flatten()

# ═══════════════════════════════════════════════════════════
# MODEL SAVE
# ═══════════════════════════════════════════════════════════
print("\n[Model Save] Saving model ...")
model_path = os.path.join(OUT, "piml_slope_model_108_pinn_richards.pt")
torch.save({
    "model_state_dict"  : best_state,
    "model_architecture":  {"class": "RichardsPINN", "feat_dim": X_tr.shape[1], **ARCH},
    "governing_equation": (
        "1D Full Richards: ∂θ/∂t = ∂/∂z[K(ψ(θ))(∂ψ/∂z − 1)]  (Eq. 8, z positive downward)\n"
        "Upper BC (Eq. 5): q_down(z=0,t) = K(θ)(1 − ∂ψ/∂z)  [z positive downward]\n"
        "Lower BC: free drainage at z=Z_MAX, dψ/dz = 0 (unit total hydraulic gradient)\n"
        "Initial condition: θ(z_sensor, t0) = measured θ0 (post warm-up)\n"
        "Total loss (Eq. 13): L = L_D + λ_phys·L_phys + λ_zvar·L_zvar + λ_BC·L_BC + λ_IC·L_IC + λ_LBC·L_LBC\n"
        "All derivatives via torch.autograd.grad — no approximation."
    ),
    "hyperparameters": {
        "epochs": EPOCHS, "lr": LR, "lambda_phys": LAM,
        "lambda_zvar": LAM_ZVAR, "lambda_bc": LAM_BC,
        "lambda_ic": LAM_IC, "lambda_lbc": LAM_LBC, "n_ic": N_IC,
        "ic_mode": IC_MODE, "lbc_mode": LBC_MODE, "warmup_theta_ok": WARMUP_THETA_OK,
        "batch_size": BATCH, "lag": LAG,
        "z_colloc_m": Z_COLLOC.tolist(), "z_sensor_m": Z_SENSOR,
        "dthdt_scale": DTHDT_SCALE, "bc_flux_scale_ms": BC_FLUX_SCALE,
        "optimizer_schedule": {
            "adam_epochs": ADAM_EPOCHS, "lbfgs_epochs": LBFGS_EPOCHS,
            "lbfgs_max_iter": LBFGS_MAX_ITER, "lbfgs_history_size": LBFGS_HISTORY,
            "lbfgs_line_search": "strong_wolfe",
        },
    },
    "scalers": {
        "rain_max": RAIN_MAX,
        "vg_theta_r": VG["theta_r"],
        "vg_theta_s": VG["theta_s"],
        "scaler_y_min": scaler_y.data_min_.tolist(),
        "scaler_y_scale": scaler_y.scale_.tolist(),
        "T_MIN": T_MIN, "T_MAX": T_MAX, "Z_MAX": Z_MAX,
    },
    "vg_params": VG,
    "slope_params": dict(slope_deg=33.0, c_kPa=5.0, phi_deg=28.0, gamma_s=14.715, H_m=1.0),
    "device_id": 108,
    "train_losses": train_losses, "val_losses": val_losses,
    "data_losses": data_losses,   "phys_losses": phys_losses,
    "bc_losses": bc_losses,
    "ic_losses": ic_losses,
    "lbc_losses": lbc_losses,
    "bc_ic_diagnostics": BCIC_DIAG,
    "val_phys_losses": val_phys_losses,
    "val_total_losses": val_total_losses,   # val data + LAM*val physics
    "lr_history": lr_history,
    "best_val_loss": best_val,
    "timestamp": datetime.datetime.now().isoformat(),
}, model_path)
print(f"  → Model saved: {model_path}")

# ═══════════════════════════════════════════════════════════════
# ██  DATA-ONLY BASELINE: same network and schedule, all λ = 0  ██
# ═══════════════════════════════════════════════════════════════
print("\n" + "█"*65)
print("  Training the data-only baseline (same network, no physics terms)")
print("  LAM=0  LAM_ZVAR=0  — No physics constraint")
print("█"*65)

# ── Re-init same architecture from scratch (same seed) ──────
torch.manual_seed(SEED); np.random.seed(SEED)
model_ml = RichardsPINN(feat_dim=X_tr.shape[1]).to(DEVICE)
print(f"  ML Baseline Parameters: {sum(p.numel() for p in model_ml.parameters()):,}")

optimizer_ml = optim.AdamW(model_ml.parameters(), lr=LR, weight_decay=1e-4)
scheduler_ml = optim.lr_scheduler.LambdaLR(optimizer_ml, lr_lambda)

data_losses_ml  = []
train_losses_ml = []
val_losses_ml   = []

best_val_ml   = np.inf
best_state_ml = None

EPOCHS_ML = EPOCHS  # same total epoch budget for fair comparison (Adam+L-BFGS, symmetric with PIML)

# ── REUSE_STORED: use the stored data-only checkpoint instead of training ───
if LOAD_STORED:
    print("\nREUSE_STORED = True: loading the stored data-only checkpoint, no training")
    _ckm = torch.load(_ckpt_ml, map_location=DEVICE, weights_only=False)
    best_state_ml = {k: v.to(DEVICE) for k, v in _ckm["model_state_dict"].items()}
    train_losses_ml, val_losses_ml, data_losses_ml = _ckm["train_losses"], _ckm["val_losses"], _ckm["data_losses"]
    best_val_ml = _ckm["best_val_loss"]
    model_ml.load_state_dict(best_state_ml)
if not LOAD_STORED:
    for epoch in range(1, ADAM_EPOCHS + 1):
        model_ml.train()
        ep_data_ml = 0.0

        for (Xb, yb, rb, tb_norm, dtb, tb_sec) in loader:
            optimizer_ml.zero_grad()
            B = Xb.shape[0]

            # ── Data loss ONLY (no physics, no z-grad) ─────────────
            z_d = z_sens_norm.expand(B, -1)          # no requires_grad needed
            t_d = tb_norm.clone()                     # no requires_grad needed
            theta_pred_ml = model_ml(z_d, t_d, Xb)

            pred_norm_ml = (theta_pred_ml - y_min_t) * y_scl_t
            loss_ml = mse_loss(pred_norm_ml, yb)     # ONLY data loss

            loss_ml.backward()
            torch.nn.utils.clip_grad_norm_(model_ml.parameters(), 1.0)
            optimizer_ml.step()
            ep_data_ml += loss_ml.item()

        scheduler_ml.step()

        model_ml.eval()
        with torch.no_grad():
            vl_ml = mse_loss(
                (model_ml(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t,
                yv
            ).item()

        n_batches = len(loader)
        data_losses_ml.append(ep_data_ml / n_batches)
        train_losses_ml.append(ep_data_ml / n_batches)
        val_losses_ml.append(vl_ml)

        if vl_ml < best_val_ml:
            best_val_ml   = vl_ml
            best_state_ml = {k: v.clone() for k, v in model_ml.state_dict().items()}

        if epoch % 100 == 0:
            print(f"  [ML] Epoch {epoch:4d} | Data {data_losses_ml[-1]:.5f} | "
                  f"Val {vl_ml:.5f} | LR {optimizer_ml.param_groups[0]['lr']:.2e}")

    # ── L-BFGS phase (data-only baseline, same schedule as the PIML) ──
    print(f"\n[ML] Switching to L-BFGS fine-tuning "
          f"({LBFGS_EPOCHS} steps, full-batch, data-only) ...")
    if best_state_ml is not None:
        model_ml.load_state_dict(best_state_ml)

    optimizer_ml_lbfgs = optim.LBFGS(
        model_ml.parameters(), lr=1.0,
        max_iter=LBFGS_MAX_ITER, history_size=LBFGS_HISTORY,
        line_search_fn="strong_wolfe",
    )

    def lbfgs_closure_ml():
        optimizer_ml_lbfgs.zero_grad()
        B = Xt.shape[0]
        z_d = z_sens_norm.expand(B, -1)
        t_d = t_tr_norm_t.clone()
        theta_pred_ml = model_ml(z_d, t_d, Xt)
        pred_norm_ml = (theta_pred_ml - y_min_t) * y_scl_t
        loss_ml = mse_loss(pred_norm_ml, yt)
        loss_ml.backward()
        lbfgs_closure_ml.last = loss_ml.item()
        return loss_ml

    for step in range(1, LBFGS_EPOCHS + 1):
        optimizer_ml_lbfgs.step(lbfgs_closure_ml)
        ld_ml = lbfgs_closure_ml.last

        model_ml.eval()
        with torch.no_grad():
            vl_ml = mse_loss(
                (model_ml(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t, yv
            ).item()
        model_ml.train()

        epoch = ADAM_EPOCHS + step
        data_losses_ml.append(ld_ml)
        train_losses_ml.append(ld_ml)
        val_losses_ml.append(vl_ml)

        if vl_ml < best_val_ml:
            best_val_ml   = vl_ml
            best_state_ml = {k: v.clone() for k, v in model_ml.state_dict().items()}

        if step % 20 == 0 or step == LBFGS_EPOCHS:
            print(f"  [ML-LBFGS] Step {step:3d}/{LBFGS_EPOCHS} (epoch {epoch}) | "
                  f"Data {ld_ml:.5f} | Val {vl_ml:.5f}")

    model_ml.load_state_dict(best_state_ml)
best_ep_ml = int(np.argmin(val_losses_ml)) + 1
print(f"\n  [ML] Best val: {best_val_ml:.5f}  @ epoch {best_ep_ml}")

def predict_sensor_ml(X_norm_np, t_sec_arr):
    model_ml.eval()
    N = X_norm_np.shape[0]
    z_s  = z_sens_norm.expand(N, -1)
    t_s  = to_t(((t_sec_arr - T_MIN) / (T_MAX - T_MIN))[:, None])
    Xn   = to_t(X_norm_np)
    with torch.no_grad():
        return model_ml(z_s, t_s, Xn).cpu().numpy().flatten()

# ═══════════════════════════════════════════════════════════
# ML BASELINE MODEL SAVE
# ═══════════════════════════════════════════════════════════
print("\n[Model Save] Saving ML baseline checkpoint ...")
ml_model_path = os.path.join(OUT, "piml_slope_model_108_ml_baseline_final.pt")
torch.save({
    "model_state_dict"  : best_state_ml,
    "model_architecture":  {"class": "RichardsPINN", "feat_dim": X_tr.shape[1], **ARCH},
    "governing_equation": "Data-only baseline (LAM=0, LAM_ZVAR=0) — no physics constraint.",
    "hyperparameters": {
        "epochs": EPOCHS_ML, "lr": LR, "lambda_phys": 0, "lambda_zvar": 0,
        "batch_size": BATCH, "lag": LAG,
        "z_colloc_m": Z_COLLOC.tolist(), "z_sensor_m": Z_SENSOR,
        "optimizer_schedule": {
            "adam_epochs": ADAM_EPOCHS, "lbfgs_epochs": LBFGS_EPOCHS,
            "lbfgs_max_iter": LBFGS_MAX_ITER, "lbfgs_history_size": LBFGS_HISTORY,
            "lbfgs_line_search": "strong_wolfe",
        },
    },
    "scalers": {
        "rain_max": RAIN_MAX,
        "vg_theta_r": VG["theta_r"],
        "vg_theta_s": VG["theta_s"],
        "scaler_y_min": scaler_y.data_min_.tolist(),
        "scaler_y_scale": scaler_y.scale_.tolist(),
        "T_MIN": T_MIN, "T_MAX": T_MAX, "Z_MAX": Z_MAX,
    },
    "vg_params": VG,
    "slope_params": dict(slope_deg=33.0, c_kPa=5.0, phi_deg=28.0, gamma_s=14.715, H_m=1.0),
    "device_id": 108,
    "train_losses": train_losses_ml, "val_losses": val_losses_ml,
    "data_losses": data_losses_ml,
    "best_val_loss": best_val_ml,
    "timestamp": datetime.datetime.now().isoformat(),
}, ml_model_path)
print(f"  → ML Baseline checkpoint saved: {ml_model_path}")


# ═══════════════════════════════════════════════════════════════
# 7.  EVALUATE — PIML (Physics-Informed)
# ═══════════════════════════════════════════════════════════════
print("\n[Step 7a] Evaluate PIML ...")
theta_pred_tr  = predict_sensor(X_tr,   t_sec_tr)
theta_pred_val = predict_sensor(X_val,  t_sec_val)
theta_pred_te  = predict_sensor(X_te,   t_sec_te)
theta_pred_all = predict_sensor(X_norm, t_sec_np)
theta_obs_all  = theta_np.copy()

r2_tr  = r2_score(theta_tr,  theta_pred_tr)
r2_val = r2_score(theta_val, theta_pred_val)
r2_te  = r2_score(theta_te,  theta_pred_te)
rmse_te = np.sqrt(mean_squared_error(theta_te, theta_pred_te))
residuals = theta_te - theta_pred_te

print(f"  [PIML] Train R²: {r2_tr:.4f}  |  Val R²: {r2_val:.4f}  |  Test R²: {r2_te:.4f}")
print(f"  [PIML] Test RMSE: {rmse_te:.5f} m³/m³")

rain_all = rain_np.copy()
ts       = pd.to_datetime(ts_all_np)

def _gb(t, y, gap_s=6 * 3600):
    """Insert a NaN wherever consecutive samples are more than gap_s apart (default 6 h),
    so plotted lines break at logger outages instead of drawing straight lines across them.
    (Shorter outages are invisible at this time scale and are left joined.)"""
    tv = pd.to_datetime(t).values; y = np.asarray(y, dtype=float)
    br = np.where(np.diff(tv) > np.timedelta64(int(gap_s), "s"))[0]
    if len(br) == 0: return pd.to_datetime(tv), y
    mid = tv[br] + (tv[br + 1] - tv[br]) / 2
    return pd.to_datetime(np.insert(tv, br + 1, mid)), np.insert(y, br + 1, np.nan)

# ═══════════════════════════════════════════════════════════════
# 7b. EVALUATE — ML Baseline (Data-Only)
# ═══════════════════════════════════════════════════════════════
print("\n[Step 7b] Evaluate ML Baseline ...")
theta_ml_tr  = predict_sensor_ml(X_tr,   t_sec_tr)
theta_ml_val = predict_sensor_ml(X_val,  t_sec_val)
theta_ml_te  = predict_sensor_ml(X_te,   t_sec_te)
theta_ml_all = predict_sensor_ml(X_norm, t_sec_np)

r2_ml_tr   = r2_score(theta_tr,  theta_ml_tr)
r2_ml_val  = r2_score(theta_val, theta_ml_val)
r2_ml_te   = r2_score(theta_te,  theta_ml_te)
rmse_ml_te = np.sqrt(mean_squared_error(theta_te, theta_ml_te))
residuals_ml = theta_te - theta_ml_te

print(f"  [ML]   Train R²: {r2_ml_tr:.4f}  |  Val R²: {r2_ml_val:.4f}  |  Test R²: {r2_ml_te:.4f}")
print(f"  [ML]   Test RMSE: {rmse_ml_te:.5f} m³/m³")

# Persistence baseline theta_hat(t) = theta(t-1): any model must beat this
theta_pers_te = X_te_raw[:, 1]
r2_pers_te    = r2_score(theta_te, theta_pers_te)
rmse_pers_te  = np.sqrt(mean_squared_error(theta_te, theta_pers_te))
print(f"  [Persist] Test R²: {r2_pers_te:.4f}  |  Test RMSE: {rmse_pers_te:.5f} m³/m³")


# ═══════════════════════════════════════════════════════════════
# 8.  Van Genuchten θ→ψ  — BOTH MODELS
# ═══════════════════════════════════════════════════════════════
print("\n[Step 8] Van Genuchten θ→ψ (both models) ...")

# ── PIML ──────────────────────────────────────────────────────
frac_low = (theta_pred_all < 0.10).mean()
psi_all_raw = vg_psi_np(theta_pred_all, VG)
PSI_MIN_M = -165.0   # fixed physical bound (permanent wilting point), not a data percentile
psi_all = np.clip(psi_all_raw, PSI_MIN_M, 0.0)
K_all   = vg_K_np(theta_pred_all, VG)
print(f"  [PIML] θ outlier frac (<0.10): {frac_low:.4f}")
print(f"  [PIML] |ψ| clipped range: [{-psi_all.max():.3f}, {-psi_all.min():.3f}] m")

# ── ML Baseline ───────────────────────────────────────────────
frac_low_ml = (theta_ml_all < 0.10).mean()
psi_ml_raw  = vg_psi_np(theta_ml_all, VG)
psi_ml_all  = np.clip(psi_ml_raw, PSI_MIN_M, 0.0)
K_ml_all    = vg_K_np(theta_ml_all, VG)
print(f"  [ML]   θ outlier frac (<0.10): {frac_low_ml:.4f}")
print(f"  [ML]   |ψ| clipped range: [{-psi_ml_all.max():.3f}, {-psi_ml_all.min():.3f}] m")

theta_vg = np.linspace(VG["theta_r"]+0.001, VG["theta_s"]-0.001, 300)
psi_vg   = vg_psi_np(theta_vg, VG)
kr_vg    = vg_Kr_np(theta_vg, VG)

# ═══════════════════════════════════════════════════════════════
# 9.  Pore pressure & FoS — BOTH MODELS
# ═══════════════════════════════════════════════════════════════
gamma_w = 9.81
SLOPE = 33.0; beta = np.radians(SLOPE)
c_ = 5.0; phi_ = np.radians(28.0)
gamma_s = 14.715; H = 1.0
sigma_n = gamma_s * H * np.cos(beta)**2
tau_d   = gamma_s * H * np.sin(beta) * np.cos(beta)

# PIML FoS
u_all   = vg_Se_np(theta_pred_all, VG) * gamma_w * psi_all   # Eq. 14: u_w = Se * gamma_w * psi (Bishop)
FoS_all = (c_ + np.maximum(sigma_n - u_all, 0) * np.tan(phi_)) / (tau_d + 1e-8)

# ML FoS
u_ml    = vg_Se_np(theta_ml_all, VG) * gamma_w * psi_ml_all
FoS_ml  = (c_ + np.maximum(sigma_n - u_ml, 0) * np.tan(phi_)) / (tau_d + 1e-8)

print(f"\n  [PIML] FoS min={FoS_all.min():.3f}  mean={FoS_all.mean():.3f}  FoS<1={(FoS_all<1).sum()}")
print(f"  [ML]   FoS min={FoS_ml.min():.3f}  mean={FoS_ml.mean():.3f}   FoS<1={(FoS_ml<1).sum()}")

# ── whole-record series of the main-run models (used by the alert cell and for figures) ──
_split = np.array(["train"] * len(theta_tr) + ["val"] * len(theta_val) + ["test"] * len(theta_te))
pd.DataFrame(dict(timestamp=ts, split=_split, rain_30min_mm=rain_np, rain_mm_h=2.0 * rain_np,
                  theta_obs=theta_obs_all, theta_piml=theta_pred_all, theta_ml=theta_ml_all,
                  psi_piml_m=psi_all, u_piml_kPa=u_all, FoS_piml=FoS_all, FoS_ml=FoS_ml)
             ).to_csv(os.path.join(OUT, "fos_timeseries_main.csv"), index=False)
print(f"  → Saved: {os.path.join(OUT, 'fos_timeseries_main.csv')}")

# ═══════════════════════════════════════════════════════════════
# 10. Richards Residual — PIML (exact autograd) + ML (effective)
# ═══════════════════════════════════════════════════════════════
print("\n[Step 10] Computing Richards residual ...")

# PIML residual via autograd
N_all  = X_norm.shape[0]
X_all_t = to_t(X_norm)
model.eval()
CHUNK = 1024
resid_list = []
for i in range(0, N_all, CHUNK):
    z_c = torch.full((min(CHUNK, N_all-i), 1), Z_SENSOR/Z_MAX,
                     dtype=torch.float32, device=DEVICE, requires_grad=True)
    t_c = to_t(((t_sec_np[i:i+CHUNK] - T_MIN)/(T_MAX-T_MIN))[:,None])
    t_c.requires_grad_(True)
    f_c = X_all_t[i:i+CHUNK]
    resid_c, _, _ = richards_residual(model, z_c, t_c, f_c)
    resid_list.append(resid_c.detach().cpu().numpy().flatten())

phys_residual = np.concatenate(resid_list)
mu_r = phys_residual.mean(); sd_r = phys_residual.std()
print(f"  [PIML] Richards residual  μ = {mu_r:.4e}  σ = {sd_r:.4e}  m³/m³/s")

# ML "effective" Richards residual (how much it violates physics)
resid_ml_list = []
model_ml.eval()
for i in range(0, N_all, CHUNK):
    z_c = torch.full((min(CHUNK, N_all-i), 1), Z_SENSOR/Z_MAX,
                     dtype=torch.float32, device=DEVICE, requires_grad=True)
    t_c = to_t(((t_sec_np[i:i+CHUNK] - T_MIN)/(T_MAX-T_MIN))[:,None])
    t_c.requires_grad_(True)
    f_c = X_all_t[i:i+CHUNK]
    resid_c, _, _ = richards_residual(model_ml, z_c, t_c, f_c)
    resid_ml_list.append(resid_c.detach().cpu().numpy().flatten())

phys_residual_ml = np.concatenate(resid_ml_list)
mu_r_ml = phys_residual_ml.mean(); sd_r_ml = phys_residual_ml.std()
print(f"  [ML]   Richards residual  μ = {mu_r_ml:.4e}  σ = {sd_r_ml:.4e}  m³/m³/s")


# ═══════════════════════════════════════════════════════════════
# ██████████  F I G U R E S  (Ablation Study)  ████████████████
# ═══════════════════════════════════════════════════════════════
print("\n[Step 11] Figures: PIML vs data-only network ...")

n_tr_plot  = len(theta_tr)
n_val_plot = n_tr_plot + len(theta_val)
epochs_arr = np.arange(1, EPOCHS + 1)
best_ep    = int(np.argmin(val_losses)) + 1

# colour shorthands
C_PIML = C["pred"]   # teal  — PIML
C_ML   = C["warn"]   # red   — ML baseline

# ──────────────────────────────────────────────────────────────
# FIG 1 — Loss Convergence Comparison (PIML vs ML)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(11, 12),
                         gridspec_kw={"hspace": 0.45})

ax = axes[0]
ax.semilogy(epochs_arr, train_losses,    color=C_PIML, lw=1.8, label="PIML train (data+physics)")
ax.semilogy(epochs_arr, val_losses,      color=C_PIML, lw=1.4, ls="--", alpha=0.8, label="PIML val")
ax.semilogy(epochs_arr, train_losses_ml, color=C_ML,   lw=1.8, label="ML train (data only)")
ax.semilogy(epochs_arr, val_losses_ml,   color=C_ML,   lw=1.4, ls="--", alpha=0.8, label="ML val")
ax.axvline(best_ep, color="gray", lw=1.0, ls=":", label=f"PIML best ep ({best_ep})")
ax.axvline(best_ep_ml, color="orange", lw=1.0, ls=":", label=f"ML best ep ({best_ep_ml})")
ax.axvline(ADAM_EPOCHS, color="black", lw=1.0, ls="-.", alpha=0.6, label="Adam→L-BFGS switch")
ax.set_ylabel("MSE Loss"); light_grid(ax)
ax.set_title("(A)  Train / Validation Loss — PIML vs ML Baseline", loc="left", fontweight="bold")
ax.legend(fontsize=8, ncol=2)

ax = axes[1]
ax.semilogy(epochs_arr, np.array(phys_losses), color=C["phys"], lw=1.8,
            label="PIML physics loss (normalised Richards PDE)")
ax.semilogy(epochs_arr, LAM * np.array(phys_losses), color=C["phys"], lw=1.2,
            ls="--", alpha=0.6, label=f"λ·Physics (λ={LAM})")
ax.axhline(1.0, color="gray", lw=0.8, ls=":", alpha=0.5, label="Reference = 1.0")
ax.axvline(ADAM_EPOCHS, color="black", lw=1.0, ls="-.", alpha=0.6, label="Adam→L-BFGS switch")
ax.set_ylabel("Richards PDE Loss (−)"); light_grid(ax)
ax.set_title("(B)  Physics Loss Convergence (PIML only)", loc="left", fontweight="bold")
ax.legend(fontsize=8)

ax = axes[2]
data_arr = np.array(data_losses)
data_ml_arr = np.array(data_losses_ml)
ax.semilogy(epochs_arr, data_arr,    color=C_PIML, lw=1.8, label="PIML data component")
ax.semilogy(epochs_arr, data_ml_arr, color=C_ML,   lw=1.8, label="ML data loss")
ax.axvline(ADAM_EPOCHS, color="black", lw=1.0, ls="-.", alpha=0.6, label="Adam→L-BFGS switch")
ax.set_xlabel("Epoch"); ax.set_ylabel("Data MSE Loss"); light_grid(ax)
ax.set_title("(C)  Data Loss Component — PIML vs ML", loc="left", fontweight="bold")
ax.legend(fontsize=8)
ax.text(0.98, 0.05,
        f"PIML best val: {min(val_losses):.2e}\nML best val:   {min(val_losses_ml):.2e}",
        transform=ax.transAxes, fontsize=9, ha="right",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))

fig.suptitle(
    f"Training Convergence — Ablation: PIML (λ={LAM}, λ_zvar={LAM_ZVAR}) vs data-only network (λ=0)\n"
    f"Same architecture · Same data · Same epochs ({EPOCHS} = {ADAM_EPOCHS} Adam + {LBFGS_EPOCHS} L-BFGS) · AdamW cosine → L-BFGS",
    fontsize=11, fontweight="bold"
)
savefig(fig, "fig01_loss_convergence_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 2 — Scatter R² : PIML vs ML side by side (6 panels)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

splits_piml = [
    ("PIML — Train", theta_tr,  theta_pred_tr,  r2_tr,     C_PIML),
    ("PIML — Val",   theta_val, theta_pred_val, r2_val,    C_PIML),
    ("PIML — Test",  theta_te,  theta_pred_te,  r2_te,     C_PIML),
]
splits_ml = [
    ("ML — Train", theta_tr,  theta_ml_tr,  r2_ml_tr,  C_ML),
    ("ML — Val",   theta_val, theta_ml_val, r2_ml_val, C_ML),
    ("ML — Test",  theta_te,  theta_ml_te,  r2_ml_te,  C_ML),
]

for row, splits in enumerate([splits_piml, splits_ml]):
    for col, (label, obs, pred, r2, col_c) in enumerate(splits):
        ax = axes[row][col]
        ax.scatter(obs, pred, s=6, alpha=0.3, color=col_c, rasterized=True)
        lim = [min(obs.min(), pred.min())-0.002, max(obs.max(), pred.max())+0.002]
        ax.plot(lim, lim, "k--", lw=1.2, label="1:1")
        m, b = np.polyfit(obs, pred, 1)
        xfit = np.linspace(*lim, 100)
        ax.plot(xfit, m*xfit+b, color=col_c, lw=1.4, alpha=0.8, label="OLS")
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel("θ Observed (m³/m³)")
        ax.set_ylabel("θ Predicted (m³/m³)")
        rmse_s = np.sqrt(mean_squared_error(obs, pred))
        ax.set_title(f"{label}\nR²={r2:.4f}  RMSE={rmse_s:.4f}")
        ax.legend(fontsize=7); light_grid(ax)

fig.suptitle("Ablation Study — Scatter R²: PIML (top row) vs data-only network (bottom row)\n"
             "Device 108 · Full Richards Autograd · Same Architecture",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
savefig(fig, "fig02_scatter_r2_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 3 — θ Time-series Comparison (Test window)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={"height_ratios":[1, 2.5, 2.5], "hspace": 0.1})

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm per 30 min)"); light_grid(axes[0])
axes[0].set_title("A — Rainfall (30-min depth)", fontweight="bold", loc="left")

axes[1].plot(*_gb(ts, theta_obs_all),  color=C["obs"],  lw=1.3, alpha=0.9, label="θ Observed", zorder=3)
axes[1].plot(*_gb(ts, theta_pred_all), color=C_PIML,    lw=1.3, alpha=0.9,
             label=f"PIML (R²={r2_te:.4f})", ls="--", zorder=2)
tr_end  = ts[n_tr_plot-1]; val_end = ts[n_val_plot-1]
axes[1].axvspan(ts[0], tr_end,   alpha=0.05, color=C["train"], label="Train")
axes[1].axvspan(tr_end, val_end, alpha=0.08, color=C["val"],   label="Val")
axes[1].axvspan(val_end, ts[-1], alpha=0.08, color=C["test"],  label="Test")
axes[1].set_ylabel("θ PIML (m³/m³)")
axes[1].legend(ncol=2, fontsize=8); light_grid(axes[1])
axes[1].set_title(f"B — PIML Prediction  [Test R²={r2_te:.4f}  RMSE={rmse_te:.4f}]",
                  fontweight="bold", loc="left")

axes[2].plot(*_gb(ts, theta_obs_all), color=C["obs"], lw=1.3, alpha=0.9, label="θ Observed", zorder=3)
axes[2].plot(*_gb(ts, theta_ml_all),  color=C_ML,     lw=1.3, alpha=0.9,
             label=f"ML Baseline (R²={r2_ml_te:.4f})", ls="--", zorder=2)
axes[2].axvspan(ts[0], tr_end,   alpha=0.05, color=C["train"])
axes[2].axvspan(tr_end, val_end, alpha=0.08, color=C["val"])
axes[2].axvspan(val_end, ts[-1], alpha=0.08, color=C["test"])
axes[2].set_ylabel("θ ML (m³/m³)"); axes[2].set_xlabel("Timestamp")
axes[2].legend(ncol=2, fontsize=8); light_grid(axes[2])
axes[2].set_title(f"C — ML Baseline Prediction  [Test R²={r2_ml_te:.4f}  RMSE={rmse_ml_te:.4f}]",
                  fontweight="bold", loc="left")

for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)

fig.suptitle("θ Time-Series: PIML vs data-only network — Device 108", fontsize=12, fontweight="bold")
savefig(fig, "fig03_theta_timeseries_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 4 — Residual Comparison (Test set)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
ts_te_plot = ts[n_val_plot:]

# PIML residuals over time
axes[0][0].axhline(0, color="black", lw=0.8)
axes[0][0].fill_between(ts_te_plot, residuals, 0, where=(residuals>=0),
                        color=C_PIML, alpha=0.5, label="Under-prediction (obs > pred)")
axes[0][0].fill_between(ts_te_plot, residuals, 0, where=(residuals<0),
                        color=C["phys"], alpha=0.5, label="Over-prediction (obs < pred)")
axes[0][0].set_title("PIML Residuals vs Time (Test)"); axes[0][0].legend(fontsize=8)
axes[0][0].set_ylabel("θ_obs − θ_pred (m³/m³)"); light_grid(axes[0][0])
for t in axes[0][0].get_xticklabels(): t.set_rotation(15)

# PIML residual histogram
axes[0][1].hist(residuals, bins=40, color=C_PIML, edgecolor="white", lw=0.3, alpha=0.85)
axes[0][1].axvline(0, color="black", lw=1.0, ls="--")
axes[0][1].axvline(residuals.mean(), color=C_ML, lw=1.4, ls="--",
                   label=f"μ={residuals.mean():.4f}")
axes[0][1].axvline(residuals.std(), color="gray", lw=1.2, ls=":",
                   label=f"σ={residuals.std():.4f}")
axes[0][1].axvline(-residuals.std(), color="gray", lw=1.2, ls=":")
axes[0][1].set_title("PIML Residual Distribution (Test)")
axes[0][1].set_xlabel("Residual (m³/m³)"); axes[0][1].legend(fontsize=8); light_grid(axes[0][1])

# ML residuals over time
axes[1][0].axhline(0, color="black", lw=0.8)
axes[1][0].fill_between(ts_te_plot, residuals_ml, 0, where=(residuals_ml>=0),
                        color=C_ML, alpha=0.5, label="Under-prediction (obs > pred)")
axes[1][0].fill_between(ts_te_plot, residuals_ml, 0, where=(residuals_ml<0),
                        color=C["u"], alpha=0.5, label="Over-prediction (obs < pred)")
axes[1][0].set_title("ML Baseline Residuals vs Time (Test)"); axes[1][0].legend(fontsize=8)
axes[1][0].set_ylabel("θ_obs − θ_pred (m³/m³)"); axes[1][0].set_xlabel("Timestamp")
light_grid(axes[1][0])
for t in axes[1][0].get_xticklabels(): t.set_rotation(15)

# ML residual histogram
axes[1][1].hist(residuals_ml, bins=40, color=C_ML, edgecolor="white", lw=0.3, alpha=0.85)
axes[1][1].axvline(0, color="black", lw=1.0, ls="--")
axes[1][1].axvline(residuals_ml.mean(), color=C_PIML, lw=1.4, ls="--",
                   label=f"μ={residuals_ml.mean():.4f}")
axes[1][1].axvline(residuals_ml.std(), color="gray", lw=1.2, ls=":",
                   label=f"σ={residuals_ml.std():.4f}")
axes[1][1].axvline(-residuals_ml.std(), color="gray", lw=1.2, ls=":")
axes[1][1].set_title("ML Baseline Residual Distribution (Test)")
axes[1][1].set_xlabel("Residual (m³/m³)"); axes[1][1].legend(fontsize=8); light_grid(axes[1][1])

fig.suptitle("Residual Analysis — PIML (top) vs data-only network (bottom) — Test Set  Dev108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig04_residuals_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 5 — Van Genuchten (same as original — PIML only)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(-psi_vg, theta_vg, color=C["vg1"], lw=2.0, label="VG Model")
psi_te_abs = np.abs(vg_psi_np(theta_pred_te, VG))
sort_idx   = np.argsort(psi_te_abs)
axes[0].scatter(psi_te_abs[sort_idx[::5]], theta_pred_te[sort_idx[::5]],
                s=6, alpha=0.3, color=C_PIML, rasterized=True, label="PIML θ̂ (test)")
psi_ml_te_abs = np.abs(vg_psi_np(theta_ml_te, VG))
sort_idx_ml   = np.argsort(psi_ml_te_abs)
axes[0].scatter(psi_ml_te_abs[sort_idx_ml[::5]], theta_ml_te[sort_idx_ml[::5]],
                s=6, alpha=0.2, color=C_ML, rasterized=True, label="ML θ̂ (test)")
axes[0].set_xlabel("|ψ| Matric Suction (m)"); axes[0].set_ylabel("θ (m³/m³)")
axes[0].set_title("(a) Soil Water Characteristic — PIML vs ML"); axes[0].set_xscale("log")
axes[0].legend(fontsize=8); light_grid(axes[0])
axes[0].text(0.98, 0.06,
             f"θ_r={VG['theta_r']} θ_s={VG['theta_s']}\nα={VG['alpha']} n={VG['n']}",
             transform=axes[0].transAxes, fontsize=8, ha="right",
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))

axes[1].plot(theta_vg, kr_vg, color=C["psi"], lw=2.0)
axes[1].fill_between(theta_vg, kr_vg, alpha=0.15, color=C["psi"])
axes[1].set_xlabel("θ (m³/m³)"); axes[1].set_ylabel("K_r (−)")
axes[1].set_title("(b) Relative Hydraulic Conductivity (VG)"); light_grid(axes[1])
fig.suptitle("van Genuchten curves — Dev108", fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig05_vg_characteristic_curves.png")

# ──────────────────────────────────────────────────────────────
# FIG 6 — Matric Suction Comparison (PIML vs ML)
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True,
                         gridspec_kw={"height_ratios":[1,2,2,2], "hspace":0.1})

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm per 30 min)")
axes[0].set_title("Matric suction: PIML vs data-only — Dev108  [ψ bounded at −165 m]", loc="left")
light_grid(axes[0])

axes[1].plot(*_gb(ts, theta_pred_all), color=C_PIML, lw=1.1, alpha=0.9, label="PIML θ̂")
axes[1].plot(*_gb(ts, theta_ml_all),   color=C_ML,   lw=1.1, alpha=0.7, label="ML θ̂", ls="--")
axes[1].plot(*_gb(ts, theta_obs_all),  color=C["obs"], lw=0.8, alpha=0.5, ls=":", label="Observed")
axes[1].set_ylabel("θ (m³/m³)"); axes[1].legend(fontsize=8, ncol=3); light_grid(axes[1])

axes[2].plot(*_gb(ts, -psi_all),    color=C_PIML, lw=1.4, label="PIML |ψ|")
axes[2].fill_between(*_gb(ts, -psi_all), alpha=0.10, color=C_PIML)
axes[2].set_ylabel("|ψ| PIML (m)"); axes[2].legend(fontsize=8); light_grid(axes[2])

axes[3].plot(*_gb(ts, -psi_ml_all), color=C_ML,   lw=1.4, label="ML |ψ|")
axes[3].fill_between(*_gb(ts, -psi_ml_all), alpha=0.10, color=C_ML)
axes[3].set_ylabel("|ψ| ML (m)"); axes[3].set_xlabel("Timestamp")
axes[3].legend(fontsize=8); light_grid(axes[3])

for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)
savefig(fig, "fig06_matric_suction_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 7 — FoS Comparison Time-series
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True,
                         gridspec_kw={"height_ratios":[1, 2, 2.5, 2.5], "hspace":0.1})

axes[0].fill_between(ts, rain_all, color=C["rain"], alpha=0.7, step="mid")
axes[0].set_ylabel("Rainfall\n(mm per 30 min)")
axes[0].set_title("Factor of Safety: PIML vs ML Baseline — Dev108", loc="left")
light_grid(axes[0])

axes[1].plot(*_gb(ts, u_all),  color=C_PIML, lw=1.3, label="PIML pore pressure u")
axes[1].plot(*_gb(ts, u_ml),   color=C_ML,   lw=1.3, label="ML pore pressure u", ls="--", alpha=0.7)
axes[1].axhline(0, color="gray", lw=0.7, ls=":")
axes[1].set_ylabel("u (kPa)"); axes[1].legend(fontsize=8, ncol=2); light_grid(axes[1])

axes[2].plot(*_gb(ts, FoS_all), color=C_PIML, lw=1.4, label=f"PIML FoS  (min={FoS_all.min():.3f})")
axes[2].axhline(1.0, color=C["warn"],    lw=1.5, ls="--", label="FoS=1.0 ⚠")
axes[2].axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3 Warning")
axes[2].fill_between(ts, FoS_all, 1.0, where=(FoS_all<1.0),
                     color=C["warn"], alpha=0.35, label=f"Failure (n={(FoS_all<1.0).sum()})")
axes[2].set_ylabel("FoS — PIML (−)"); axes[2].legend(ncol=2, fontsize=8); light_grid(axes[2])
axes[2].set_ylim(bottom=max(0, min(FoS_all.min(), FoS_ml.min())-0.1))

axes[3].plot(*_gb(ts, FoS_ml), color=C_ML, lw=1.4, label=f"ML FoS  (min={FoS_ml.min():.3f})")
axes[3].axhline(1.0, color=C["warn"],    lw=1.5, ls="--", label="FoS=1.0 ⚠")
axes[3].axhline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3 Warning")
axes[3].fill_between(ts, FoS_ml, 1.0, where=(FoS_ml<1.0),
                     color=C["warn"], alpha=0.35, label=f"Failure (n={(FoS_ml<1.0).sum()})")
axes[3].set_ylabel("FoS — ML (−)"); axes[3].set_xlabel("Timestamp")
axes[3].legend(ncol=2, fontsize=8); light_grid(axes[3])
axes[3].set_ylim(bottom=max(0, min(FoS_all.min(), FoS_ml.min())-0.1))

for ax in axes:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)
savefig(fig, "fig07_fos_timeseries_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 8 — FoS Distribution Comparison
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

all_fos = np.concatenate([FoS_all, FoS_ml])
bins    = np.linspace(all_fos.min()-0.05, all_fos.max()+0.05, 60)

axes[0].hist(FoS_all, bins=bins, color=C_PIML, edgecolor="white", lw=0.3, alpha=0.75,
             label=f"PIML  (μ={FoS_all.mean():.3f}, min={FoS_all.min():.3f})")
axes[0].hist(FoS_ml,  bins=bins, color=C_ML,   edgecolor="white", lw=0.3, alpha=0.55,
             label=f"ML    (μ={FoS_ml.mean():.3f}, min={FoS_ml.min():.3f})")
axes[0].axvline(1.0, color="black",       lw=2.0, ls="--", label="FoS=1.0 ⚠")
axes[0].axvline(1.3, color="darkorange",  lw=1.5, ls=":",  label="FoS=1.3 Warning")
axes[0].set_xlabel("FoS (−)"); axes[0].set_ylabel("Count")
axes[0].set_title("(a) FoS Histogram — PIML vs ML"); axes[0].legend(fontsize=8); light_grid(axes[0])

sorted_piml = np.sort(FoS_all); exc_piml = 1 - np.arange(1, len(sorted_piml)+1)/len(sorted_piml)
sorted_ml   = np.sort(FoS_ml);  exc_ml   = 1 - np.arange(1, len(sorted_ml)+1)/len(sorted_ml)
axes[1].semilogy(sorted_piml, exc_piml, color=C_PIML, lw=2.0, label="PIML")
axes[1].semilogy(sorted_ml,   exc_ml,   color=C_ML,   lw=2.0, label="ML Baseline", ls="--")
axes[1].axvline(1.0, color="black",      lw=1.5, ls="--", label="FoS=1.0")
axes[1].axvline(1.3, color="darkorange", lw=1.0, ls=":",  label="FoS=1.3")
axes[1].set_xlabel("FoS (−)"); axes[1].set_ylabel("Exceedance Prob.")
axes[1].set_title("(b) FoS Exceedance Probability"); axes[1].legend(fontsize=8); light_grid(axes[1])

fig.suptitle("Factor of Safety Distribution: PIML vs data-only network — Dev108",
             fontsize=12, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig08_fos_distribution_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 9 — Richards PDE Residual: PIML vs ML
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# PIML residual time-series
axes[0][0].plot(*_gb(ts, phys_residual), color=C_PIML, lw=0.8, alpha=0.85)
axes[0][0].axhline(0, color="gray", lw=0.7, ls=":")
axes[0][0].fill_between(ts, phys_residual, 0,
                         where=(np.abs(phys_residual) > 2*sd_r),
                         color=C["warn"], alpha=0.4, label=f">2σ  σ={sd_r:.2e}")
axes[0][0].set_title("PIML — Richards residual ∂θ/∂t − $\\nabla\\cdot$flux  (m³/m³/s)")
axes[0][0].set_ylabel("Residual (m³/m³/s)"); axes[0][0].legend(fontsize=8); light_grid(axes[0][0])
axes[0][0].text(0.99, 0.97, f"μ={mu_r:.2e}\nσ={sd_r:.2e}",
                transform=axes[0][0].transAxes, fontsize=8, va="top", ha="right",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))
for t in axes[0][0].get_xticklabels(): t.set_rotation(15)

# PIML residual histogram
axes[0][1].hist(phys_residual, bins=50, color=C_PIML, edgecolor="white", lw=0.3, alpha=0.85)
axes[0][1].axvline(0, color="black", lw=1.0, ls="--")
axes[0][1].axvline( sd_r, color="gray", lw=1.2, ls=":", label=f"±σ={sd_r:.2e}")
axes[0][1].axvline(-sd_r, color="gray", lw=1.2, ls=":")
axes[0][1].set_title("PIML — PDE Residual Distribution")
axes[0][1].set_xlabel("Residual (m³/m³/s)"); axes[0][1].legend(fontsize=8); light_grid(axes[0][1])

# ML residual time-series
axes[1][0].plot(*_gb(ts, phys_residual_ml), color=C_ML, lw=0.8, alpha=0.85)
axes[1][0].axhline(0, color="gray", lw=0.7, ls=":")
axes[1][0].fill_between(ts, phys_residual_ml, 0,
                         where=(np.abs(phys_residual_ml) > 2*sd_r_ml),
                         color=C["warn"], alpha=0.4, label=f">2σ  σ={sd_r_ml:.2e}")
axes[1][0].set_title("ML Baseline — Effective Richards Residual (m³/m³/s)")
axes[1][0].set_ylabel("Residual (m³/m³/s)"); axes[1][0].set_xlabel("Timestamp")
axes[1][0].legend(fontsize=8); light_grid(axes[1][0])
axes[1][0].text(0.99, 0.97, f"μ={mu_r_ml:.2e}\nσ={sd_r_ml:.2e}",
                transform=axes[1][0].transAxes, fontsize=8, va="top", ha="right",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.7))
for t in axes[1][0].get_xticklabels(): t.set_rotation(15)

# ML residual histogram
axes[1][1].hist(phys_residual_ml, bins=50, color=C_ML, edgecolor="white", lw=0.3, alpha=0.85)
axes[1][1].axvline(0, color="black", lw=1.0, ls="--")
axes[1][1].axvline( sd_r_ml, color="gray", lw=1.2, ls=":", label=f"±σ={sd_r_ml:.2e}")
axes[1][1].axvline(-sd_r_ml, color="gray", lw=1.2, ls=":")
axes[1][1].set_title("ML Baseline — PDE Residual Distribution")
axes[1][1].set_xlabel("Residual (m³/m³/s)"); axes[1][1].legend(fontsize=8); light_grid(axes[1][1])

fig.suptitle(
    "Richards PDE residual: PIML (top) vs data-only model (bottom)\n"
    "Consistency check only: the PIML is trained to reduce this residual, the data-only model is not",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
savefig(fig, "fig09_pde_residual_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 10 — Full dashboard
# ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 17))
gs  = gridspec.GridSpec(5, 1, figure=fig, hspace=0.50)
ax0 = fig.add_subplot(gs[0])
ax0.fill_between(ts, rain_all, color=C["rain"], alpha=0.75, step="mid")
ax0.set_ylabel("Rainfall (mm per 30 min)")
ax0.set_title("A — Rainfall (30-min depth)", fontweight="bold", loc="left")
light_grid(ax0); add_panel_label(ax0, "(A)", y=1.08)

ax1 = fig.add_subplot(gs[1])
ax1.plot(*_gb(ts, theta_obs_all),  color=C["obs"],  lw=1.2, alpha=0.85, label="θ Observed")
ax1.plot(*_gb(ts, theta_pred_all), color=C_PIML,    lw=1.2, alpha=0.85, label="θ PIML-Richards", ls="--")
ax1.plot(*_gb(ts, theta_ml_all),   color=C_ML,      lw=1.0, alpha=0.65, label="θ ML-Baseline",  ls=":")
ax1.set_ylabel("θ (m³/m³)")
ax1.set_title(f"B — Soil Moisture  [PIML R²={r2_te:.3f}  ML R²={r2_ml_te:.3f}]",
              fontweight="bold", loc="left")
ax1.legend(ncol=3, fontsize=8); light_grid(ax1); add_panel_label(ax1, "(B)", y=1.08)

ax2 = fig.add_subplot(gs[2])
ax2.plot(*_gb(ts, -psi_all),    color=C_PIML, lw=1.3, label="PIML |ψ|")
ax2.plot(*_gb(ts, -psi_ml_all), color=C_ML,   lw=1.1, alpha=0.7, label="ML |ψ|", ls="--")
ax2.fill_between(*_gb(ts, -psi_all), alpha=0.08, color=C_PIML)
ax2.set_ylabel("|ψ| clipped (m)")
ax2.set_title("C — Matric Suction (VG, clipped)", fontweight="bold", loc="left")
ax2.legend(fontsize=8); light_grid(ax2); add_panel_label(ax2, "(C)", y=1.08)

ax3 = fig.add_subplot(gs[3])
ax3.plot(*_gb(ts, u_all), color=C_PIML, lw=1.3, label="PIML u")
ax3.plot(*_gb(ts, u_ml),  color=C_ML,   lw=1.1, alpha=0.7, label="ML u", ls="--")
ax3.axhline(0, color="gray", lw=0.7, ls=":")
ax3.fill_between(ts, u_all, 0, where=(u_all>0), color=C["warn"], alpha=0.2, label="Positive u")
ax3.set_ylabel("u Pore Pressure (kPa)")
ax3.set_title("D — Pore Water Pressure", fontweight="bold", loc="left")
ax3.legend(fontsize=8, ncol=3); light_grid(ax3); add_panel_label(ax3, "(D)", y=1.08)

ax4 = fig.add_subplot(gs[4])
ax4.plot(*_gb(ts, FoS_all), color=C_PIML, lw=1.4,
         label=f"PIML FoS (min={FoS_all.min():.3f})")
ax4.plot(*_gb(ts, FoS_ml),  color=C_ML,   lw=1.2, alpha=0.75, ls="--",
         label=f"ML FoS   (min={FoS_ml.min():.3f})")
ax4.axhline(1.0, color=C["warn"],    lw=1.8, ls="--", label="FoS=1.0 ⚠ Failure")
ax4.axhline(1.3, color="darkorange", lw=1.2, ls=":",  label="FoS=1.3 Warning")
ax4.fill_between(ts, FoS_all, 1.0, where=(FoS_all<1.0), color=C["warn"], alpha=0.35)
ax4.set_ylabel("FoS (−)"); ax4.set_xlabel("Timestamp")
ax4.set_title("E — Factor of Safety (Infinite Slope)", fontweight="bold", loc="left")
ax4.legend(ncol=2, fontsize=8); light_grid(ax4); add_panel_label(ax4, "(E)", y=1.08)
ax4.set_ylim(bottom=max(0, min(FoS_all.min(), FoS_ml.min())-0.1))

for ax in [ax0, ax1, ax2, ax3, ax4]:
    ax.set_xlim(ts[0], ts[-1])
    for t in ax.get_xticklabels(): t.set_rotation(15)

fig.suptitle(
    "Full Dashboard — PIML (Richards, Autograd) vs ML Baseline (Data-Only)\n"
    f"Device 108  |  λ_phys={LAM}  λ_zvar={LAM_ZVAR}  |  PIML R²={r2_te:.4f}  ML R²={r2_ml_te:.4f}",
    fontsize=12, fontweight="bold", y=1.002)
savefig(fig, "fig10_full_dashboard_ablation.png")

# ──────────────────────────────────────────────────────────────
# FIG 11 — ABLATION TABLE (publication-ready matplotlib table)
# ──────────────────────────────────────────────────────────────
_pers = {"tr": (theta_tr, X_tr_raw[:, 1]), "val": (theta_val, X_val_raw[:, 1]), "te": (theta_te, X_te_raw[:, 1])}
r2_p = {k: r2_score(o, p) for k, (o, p) in _pers.items()}          # persistence: θ(t) = θ(t−1)
rmse_p_te = np.sqrt(mean_squared_error(*_pers["te"]))
_skill = lambda rm: 1 - rm ** 2 / rmse_p_te ** 2
_npar = sum(p.numel() for p in model.parameters())
_arch = (f"{ARCH['depth']}×{ARCH['width']}" + (f" + {ARCH['width'] // 2}" if ARCH["taper"] else "")
         + f", {ARCH['act']}, {_npar:,} params")
f4 = lambda x: f"{x:.4f}"
col_labels = ["Metric", "PIML", "Data-only", "Persistence θ(t−1)", "PIML − data-only"]
row_data = [
    ["Train R²", f4(r2_tr), f4(r2_ml_tr), f4(r2_p["tr"]), f"{r2_tr - r2_ml_tr:+.4f}"],
    ["Validation R²", f4(r2_val), f4(r2_ml_val), f4(r2_p["val"]), f"{r2_val - r2_ml_val:+.4f}"],
    ["Test R²", f4(r2_te), f4(r2_ml_te), f4(r2_p["te"]), f"{r2_te - r2_ml_te:+.4f}"],
    ["Test RMSE (m³/m³)", f"{rmse_te:.5f}", f"{rmse_ml_te:.5f}", f"{rmse_p_te:.5f}", f"{rmse_te - rmse_ml_te:+.5f}"],
    ["Test skill vs persistence (1 − MSE/MSE_pers)", f"{_skill(rmse_te):.2f}", f"{_skill(rmse_ml_te):.2f}", "0",
     f"{_skill(rmse_te) - _skill(rmse_ml_te):+.2f}"],
    ["Richards residual σ (m³/m³/s)", f"{sd_r:.2e}", f"{sd_r_ml:.2e}", "—", "PIML is trained on it"],
    ["Minimum FoS (whole record)", f"{FoS_all.min():.3f}", f"{FoS_ml.min():.3f}", "—", f"{FoS_all.min() - FoS_ml.min():+.3f}"],
    ["Samples with FoS < 1.3 (whole record)", f"{(FoS_all < 1.3).sum()}", f"{(FoS_ml < 1.3).sum()}", "—", "—"],
    ["Loss weights", f"λ_phys {LAM:g}, λ_BC {LAM_BC:g}, λ_IC {LAM_IC:g}, λ_LBC {LAM_LBC:g}, λ_zvar {LAM_ZVAR:g}",
     "all λ = 0", "—", "—"],
    ["Network (same for both)", _arch, "same", "—", "—"],
    ["Training (same for both)", f"{ADAM_EPOCHS} Adam + {LBFGS_EPOCHS} L-BFGS, best-validation checkpoint", "same", "—", "—"],
]
fig, ax = plt.subplots(figsize=(15, 6.2))
ax.axis("off")
tbl = ax.table(cellText=row_data, colLabels=col_labels, cellLoc="center",
               colWidths=[0.24, 0.30, 0.12, 0.14, 0.16], bbox=[0, 0, 1, 0.86])
tbl.auto_set_font_size(False); tbl.set_fontsize(9.5)
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor("#023E8A"); tbl[0, j].set_text_props(color="white", fontweight="bold")
for i in range(1, len(row_data) + 1):
    for j in range(len(col_labels)):
        tbl[i, j].set_facecolor("#E3F2FD" if i in (3, 4, 5) else ("#F8F9FA" if i % 2 == 0 else "white"))
ax.set_title(f"Table 1 — PIML vs data-only network vs persistence, Device 108 (test n = {len(theta_te):,})\n"
             f"Infinite slope {SLOPE:g}° · van Genuchten θr {VG['theta_r']}, θs {VG['theta_s']}, α {VG['alpha']}, n {VG['n']}",
             fontsize=11, fontweight="bold", pad=6)
savefig(fig, "fig11_ablation_table.png")


# ═══════════════════════════════════════════════════════════════
# SENSITIVITY ANALYSIS: Ks (hydraulic conductivity) & Slope angle β
# ═══════════════════════════════════════════════════════════════
print("\n" + "█"*65)
print("  SENSITIVITY ANALYSIS")
print("  Perturbing Ks and slope angle β independently")
print("█"*65)

import itertools

# ── Perturbation ranges ────────────────────────────────────────
KS_SCALES  = np.array([0.60, 0.70, 0.80, 0.90, 1.00,
                        1.10, 1.20, 1.30, 1.40])          # ×baseline
SLOPE_DEGS = np.array([23.0, 25.0, 27.0, 29.0, 31.0,
                        33.0, 35.0, 37.0, 39.0, 41.0])   # degrees

BASE_KS    = VG["Ks_ms"]      # baseline hydraulic conductivity (m/s)
BASE_SLOPE = 33.0             # baseline slope (degrees)

# ── Fixed geotechnical params ──────────────────────────────────
c_kPa   = 5.0
phi_deg = 28.0
phi_rad = np.radians(phi_deg)
gamma_s = 14.715
H_m     = 1.0


def compute_fos_series(theta_arr, slope_deg, ks_scale=1.0):
    """FoS series for a slope angle and a Ks scale factor, from fixed θ predictions.

    Ks does not enter ψ(θ). Here a Ks change is represented by a PROXY, ψ_eff = ψ / ks_scale
    (faster drainage -> less suction); the flow problem is NOT re-solved. Report the Ks result
    as a screening of the pore-pressure path only. Suction is capped at σn (u ≥ −σn) here,
    so FoS values on the dry side can differ slightly from the main FoS series.
    """
    # ψ from VG (theta only — Ks doesn't enter ψ directly)
    psi_raw = vg_psi_np(theta_arr, VG)
    psi     = np.clip(psi_raw, -165.0, 0.0)

    # proxy: ψ_eff = ψ / ks_scale
    psi_eff = psi / ks_scale

    beta_r  = np.radians(slope_deg)
    sigma_n = gamma_s * H_m * np.cos(beta_r)**2
    tau_d   = gamma_s * H_m * np.sin(beta_r) * np.cos(beta_r)

    u   = vg_Se_np(theta_arr, VG) * gamma_w * psi_eff

    u_clamped = np.clip(u, -sigma_n, None)
    effective_stress = np.maximum(sigma_n - u_clamped, 0)

    FoS = (c_kPa + effective_stress * np.tan(phi_rad)) / (tau_d + 1e-8)
    return FoS, u, psi_eff


# ══════════════════════════════════════════════════════════════
# (A) Ks sensitivity — vary Ks scale, keep slope = 33°
# ══════════════════════════════════════════════════════════════
print("\n[SA-A] Ks sensitivity ...")
ks_results = {}
for scale in KS_SCALES:
    fos, u_, psi_ = compute_fos_series(theta_pred_all, BASE_SLOPE, ks_scale=scale)
    ks_results[scale] = {
        "FoS_mean": fos.mean(), "FoS_min": fos.min(),
        "FoS_p5": np.percentile(fos, 5),
        "n_fail": (fos < 1.0).sum(),
        "n_warn": (fos < 1.3).sum(),
        "FoS_series": fos,
    }
    print(f"  Ks×{scale:.2f}  FoS_mean={fos.mean():.3f}  "
          f"FoS_min={fos.min():.3f}  fail={(fos<1.0).sum()}")

# ══════════════════════════════════════════════════════════════
# (B) Slope sensitivity — vary β, keep Ks = baseline
# ══════════════════════════════════════════════════════════════
print("\n[SA-B] Slope angle sensitivity ...")
slope_results = {}
for deg in SLOPE_DEGS:
    fos, u_, psi_ = compute_fos_series(theta_pred_all, deg, ks_scale=1.0)
    slope_results[deg] = {
        "FoS_mean": fos.mean(), "FoS_min": fos.min(),
        "FoS_p5": np.percentile(fos, 5),
        "n_fail": (fos < 1.0).sum(),
        "n_warn": (fos < 1.3).sum(),
        "FoS_series": fos,
    }
    print(f"  β={deg:.0f}°  FoS_mean={fos.mean():.3f}  "
          f"FoS_min={fos.min():.3f}  fail={(fos<1.0).sum()}")

# ══════════════════════════════════════════════════════════════
# (C) 2-D grid: Ks × Slope → FoS_min  (heatmap)
# ══════════════════════════════════════════════════════════════
print("\n[SA-C] 2-D sensitivity grid (Ks × slope) ...")
grid_fos_min  = np.zeros((len(SLOPE_DEGS), len(KS_SCALES)))
grid_fos_mean = np.zeros_like(grid_fos_min)
grid_n_fail   = np.zeros_like(grid_fos_min)

for i, deg in enumerate(SLOPE_DEGS):
    for j, scale in enumerate(KS_SCALES):
        fos, _, _ = compute_fos_series(theta_pred_all, deg, ks_scale=scale)
        grid_fos_min[i, j]  = fos.min()
        grid_fos_mean[i, j] = fos.mean()
        grid_n_fail[i, j]   = (fos < 1.0).sum()

print("  Grid complete.")

# ── tables for the paper ──────────────────────────────────────────
pd.DataFrame([dict(ks_scale=k, **{c: v[c] for c in ("FoS_mean", "FoS_min", "FoS_p5", "n_fail", "n_warn")})
              for k, v in ks_results.items()]).to_csv(os.path.join(OUT, "fos_sensitivity_ks.csv"), index=False)
pd.DataFrame([dict(slope_deg=k, **{c: v[c] for c in ("FoS_mean", "FoS_min", "FoS_p5", "n_fail", "n_warn")})
              for k, v in slope_results.items()]).to_csv(os.path.join(OUT, "fos_sensitivity_slope.csv"), index=False)
pd.DataFrame(grid_fos_min, index=pd.Index(SLOPE_DEGS, name="slope_deg"),
             columns=[f"ks_x{k:.2f}" for k in KS_SCALES]).to_csv(os.path.join(OUT, "fos_sensitivity_grid_min.csv"))

pd.DataFrame([
    dict(model="PIML", train_r2=r2_tr, val_r2=r2_val, test_r2=r2_te, test_rmse=rmse_te,
         test_skill=1 - rmse_te ** 2 / rmse_pers_te ** 2, richards_resid_sd=sd_r,
         fos_min=FoS_all.min(), fos_mean=FoS_all.mean(), n_fos_below_1p3=int((FoS_all < 1.3).sum())),
    dict(model="data-only", train_r2=r2_ml_tr, val_r2=r2_ml_val, test_r2=r2_ml_te, test_rmse=rmse_ml_te,
         test_skill=1 - rmse_ml_te ** 2 / rmse_pers_te ** 2, richards_resid_sd=sd_r_ml,
         fos_min=FoS_ml.min(), fos_mean=FoS_ml.mean(), n_fos_below_1p3=int((FoS_ml < 1.3).sum())),
    dict(model="persistence", test_r2=r2_pers_te, test_rmse=rmse_pers_te, test_skill=0.0),
]).to_csv(os.path.join(OUT, "main_run_summary.csv"), index=False)
print(f"  → Saved: main_run_summary.csv, fos_sensitivity_ks.csv, fos_sensitivity_slope.csv, fos_sensitivity_grid_min.csv")
print("\nMain cell complete: models, figures, FoS series and sensitivity tables saved.")


## 3. Boundary modes
Defines the selectable lower boundary (free drainage, no-flux, water table) and initial condition (sensor, uniform). The defaults reproduce the main run. Run it right after the main cell, also after a disconnect.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3. Selectable lower boundary and initial condition
#    Run right after the main cell. The defaults reproduce the main run (free drainage,
#    IC at the sensor); the boundary-scenario cell uses the other modes.
#    z is positive DOWNWARD; downward Darcy flux q = K(1 - dpsi/dz).
#      free_drainage : unit total gradient      -> dpsi/dz = 0   (q = K)
#      no_flux       : impermeable base         -> dpsi/dz = 1   (q = 0 for any K)
#      water_table   : near-saturated base      -> psi = PSI_WT  (fixed head)
# ═══════════════════════════════════════════════════════════════════════════
LBC_MODE = "free_drainage"
IC_MODE  = "sensor"
PSI_WT   = -0.05          # m; pressure head imposed at the base for "water_table"

def lower_bc_loss(m, t_norm, feat, mode=None):
    mode = mode or LBC_MODE
    B = t_norm.shape[0]
    z = torch.ones(B, 1, dtype=torch.float32, device=DEVICE).requires_grad_(True)   # z = Z_MAX
    theta = m(z, t_norm, feat)
    psi = vg_psi_t(theta, VG)
    if mode == "water_table":
        return ((psi - PSI_WT) ** 2).mean()                                          # m^2
    dpsi_dz = torch.autograd.grad(psi.sum(), z, create_graph=True)[0] / Z_MAX
    if mode == "free_drainage":
        return (dpsi_dz ** 2).mean()
    if mode == "no_flux":
        return ((dpsi_dz - 1.0) ** 2).mean()        # zero total-head gradient -> q = 0 for any K
    raise ValueError(mode)

# the training losses call these names at run time, so redefining them switches every loss
lower_bc_free_drainage_loss = lambda m, t_norm, feat: lower_bc_loss(m, t_norm, feat)

def ic_loss(m, mode=None, n_ic=N_IC):
    """sensor : theta(z_s, t0) = measured theta0 (only observed depth).
       uniform: theta0 imposed at z = 0, z_s and Z_MAX (ASSUMED profile)."""
    mode = mode or IC_MODE
    depths = [Z_SENSOR] if mode == "sensor" else [0.0, Z_SENSOR, float(Z_COLLOC.max())]
    loss = torch.zeros((), device=DEVICE)
    for zd in depths:
        z = torch.full((n_ic, 1), zd / Z_MAX, dtype=torch.float32, device=DEVICE)
        loss = loss + (((m(z, t_tr_norm_t[:n_ic], Xt[:n_ic]) - IC_THETA0) / THETA_RANGE) ** 2).mean()
    return loss / len(depths)

print(f"Boundary modes ready: LBC_MODE={LBC_MODE}, IC_MODE={IC_MODE}")

## 4. Loss-weight (λ) check
One-at-a-time check around the final weights (λ_phys 0.001 / 0.1 / 1; λ_BC 0 / 0.001 / 0.01 / 0.1) plus the data-only reference, 3 seeds, ~1/3 budget, judged on validation. Output: `sensitivity_lambda.csv`, `sensitivity_lambda_per_seed.csv`, `fig_sensitivity_lambda.png`. In the season run only the default and the data-only reference are trained.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 4. Loss-weight (λ) check — one-at-a-time, around the final setting
#    Does the result depend on the chosen weights? Each configuration is re-trained at a
#    reduced budget (~1/3 of the main run) with the same seeds; the default and the data-only
#    reference (all λ = 0) are trained at the same budget, so comparisons are paired.
#    λ is judged on VALIDATION metrics; test metrics are shown for transparency only.
#    Each swept weight also gets a targeted diagnostic (λ·Phys/Data, surface-flux residual).
#    Limits: local OAT screening (no interactions), reduced budget, n = number of seeds
#    (low power: "no difference" is not "equivalent"), metrics at z = 0.30 m only.
# ═══════════════════════════════════════════════════════════════════════════
import time
from scipy import stats

RUN_SENSITIVITY   = True
SENS_ADAM_EPOCHS  = BUDGET["sens_adam"]    # 130 (~1/3 of the main run)
SENS_LBFGS_EPOCHS = BUDGET["sens_lbfgs"]   # 70
SENS_SEEDS        = BUDGET["seeds"]         # 42, 43, 44
SENS_DEFAULT = dict(LAM=LAM, LAM_ZVAR=LAM_ZVAR, LAM_BC=LAM_BC,
                    LAM_IC=LAM_IC, LAM_LBC=LAM_LBC)
SENS_ZERO    = {k: 0.0 for k in SENS_DEFAULT}       # data-only reference
# Grid around the final setting (λ_phys 0.1, λ_BC 0): λ_phys one decade either side, and the
# surface-flux weight from 0 (off) to 0.1. The data-only reference is trained as well.
SENS_GRID = {
    "LAM":    [1e-3, 1e-1, 1.0],      # λ_phys   (Richards PDE)
    "LAM_BC": [0.0, 1e-3, 1e-2, 1e-1],   # λ_BC     (surface flux); 0 = surface-flux term off
}
SENS_ACTIVE = ["LAM", "LAM_BC"] if RUN["sens_grid"] else []
if not SENS_ACTIVE:                   # season run: only the default and data-only references
    SENS_GRID = {"LAM": [LAM]}; SENS_ACTIVE = ["LAM"]
SENS_GRID   = {k: v for k, v in SENS_GRID.items() if k in SENS_ACTIVE}

# targeted diagnostic shown in the 4th column of the figure, per swept weight
SENS_TARGET = {
    "LAM":      ("phys_over_data", "λ·Phys / Data (val, best ckpt)"),
    "LAM_BC":   ("bc_res",         "Surface-flux residual (norm., val)"),
    "LAM_IC":   ("ic_err",         "IC error at sensor (m³/m³)"),
    "LAM_LBC":  ("lbc_rms",        "RMS ∂ψ/∂z at z = Z_MAX (−)"),
    "LAM_ZVAR": ("test_rmse",      "Test RMSE (m³/m³)"),
}


def _sens_loss(m, Xb, yb, rb, tb_norm, tb_sec_np, cfg):
    """Same total loss as the main run (Eq. 13), weights taken from cfg."""
    B = Xb.shape[0]
    z_d = z_sens_norm.expand(B, -1).requires_grad_(True)
    t_d = tb_norm.clone().requires_grad_(True)
    loss_d = mse_loss((m(z_d, t_d, Xb) - y_min_t) * y_scl_t, yb)

    z_c, t_c, feat_c = make_colloc_batch(tb_sec_np, None, Xb.detach().cpu().numpy())
    resid, _, dtheta_dz = richards_residual(m, z_c, t_c, feat_c)
    loss_p = ((resid / (DTHDT_SCALE + 1e-12)) ** 2).mean()
    z_var_loss = -dtheta_dz.view(B, len(Z_COLLOC)).var(dim=1).mean()

    z_bc = torch.zeros(B, 1, dtype=torch.float32, device=DEVICE).requires_grad_(True)
    t_bc = tb_norm.clone().requires_grad_(True)
    q_pred = surface_bc_flux(m, z_bc, t_bc, Xb)
    q_tgt  = torch.clamp((rb * RAIN_MM_TO_M) / RAIN_WINDOW_S, max=BC_FLUX_SCALE)
    loss_bc = (((q_pred - q_tgt) / (BC_FLUX_SCALE + 1e-12)) ** 2).mean()

    loss_ic  = ic_loss(m)
    loss_lbc = lower_bc_free_drainage_loss(m, tb_norm, Xb)

    loss = (loss_d + cfg["LAM"] * loss_p + cfg["LAM_ZVAR"] * z_var_loss
            + cfg["LAM_BC"] * loss_bc + cfg["LAM_IC"] * loss_ic
            + cfg["LAM_LBC"] * loss_lbc)
    return loss


def _sens_val(m):
    """Val data MSE (scaled units, as in main run) + UNWEIGHTED val physics residual."""
    m.eval()
    with torch.no_grad():
        vl = mse_loss((m(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t, yv).item()
    with torch.enable_grad():
        zc, tc, fc = make_colloc_batch(t_sec_val, None, X_val)
        r, _, _ = richards_residual(m, zc, tc, fc)
        vp = ((r / (DTHDT_SCALE + 1e-12)) ** 2).mean().item()
    m.train()
    return vl, vp


def _sens_bcic_diag(m):
    """Targeted diagnostics at the best checkpoint (mirrors BCIC_DIAG of the main run),
    evaluated on VALIDATION inputs: lower-BC slope, surface-flux residual, IC error."""
    m.eval()
    with torch.enable_grad():
        n = Xv.shape[0]
        z1 = torch.ones(n, 1, dtype=torch.float32, device=DEVICE).requires_grad_(True)
        th = m(z1, t_v_norm, Xv)
        dpsi = torch.autograd.grad(vg_psi_t(th, VG).sum(), z1)[0] / Z_MAX
        lbc_rms = dpsi.pow(2).mean().sqrt().item()

        z0 = torch.zeros(n, 1, dtype=torch.float32, device=DEVICE).requires_grad_(True)
        q_pred = surface_bc_flux(m, z0, t_v_norm, Xv)
        q_tgt  = torch.clamp(to_t(rain_val[:, None]) * RAIN_MM_TO_M / RAIN_WINDOW_S,
                             max=BC_FLUX_SCALE)
        bc_res = (((q_pred - q_tgt) / (BC_FLUX_SCALE + 1e-12)) ** 2).mean().sqrt().item()
    with torch.no_grad():
        z_ic = torch.full((N_IC, 1), Z_SENSOR / Z_MAX, dtype=torch.float32, device=DEVICE)
        ic_err = (m(z_ic, t_tr_norm_t[:N_IC], Xt[:N_IC]) - IC_THETA0).abs().mean().item()
    m.train()
    return dict(lbc_rms=lbc_rms, bc_res=bc_res, ic_err=ic_err)


def _sens_predict(m, Xn_np, t_sec_arr):
    m.eval()
    z_s = z_sens_norm.expand(Xn_np.shape[0], -1)
    t_s = to_t(((t_sec_arr - T_MIN) / (T_MAX - T_MIN))[:, None])
    with torch.no_grad():
        return m(z_s, t_s, to_t(Xn_np)).cpu().numpy().flatten()


def train_piml_sens(cfg, seed, adam_ep, lbfgs_ep):
    """Adam (cosine) -> L-BFGS, same schedule as the main run; the budget is an argument.
    Also used at the full budget by the multi-seed cell (return keys test_r2, test_rmse, ...)."""
    torch.manual_seed(seed); np.random.seed(seed)
    loader.generator.manual_seed(seed)
    m   = RichardsPINN(feat_dim=X_tr.shape[1]).to(DEVICE)
    opt = optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    warm = max(1, int(round(WARMUP * adam_ep / ADAM_EPOCHS)))
    def _lr(e):
        if e < warm: return float(e + 1) / float(warm)
        return 0.5 * (1.0 + np.cos(np.pi * (e - warm) / max(1, adam_ep - warm)))
    sch = optim.lr_scheduler.LambdaLR(opt, _lr)

    best_val, best_state = np.inf, None

    # ── Adam phase (mini-batch) ──
    for _ in range(adam_ep):
        m.train()
        for (Xb, yb, rb, tb_norm, dtb, tb_sec) in loader:
            opt.zero_grad()
            loss = _sens_loss(m, Xb, yb, rb, tb_norm,
                              tb_sec.detach().cpu().numpy().flatten(), cfg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
        sch.step()
        vl, _ = _sens_val(m)
        if vl < best_val:
            best_val = vl
            best_state = {k: v.clone() for k, v in m.state_dict().items()}

    # ── L-BFGS phase (full-batch), resume from Adam best ──
    m.load_state_dict(best_state)
    opt_lb = optim.LBFGS(m.parameters(), lr=1.0, max_iter=LBFGS_MAX_ITER,
                         history_size=LBFGS_HISTORY, line_search_fn="strong_wolfe")
    t_full = t_sec_tr[:, None].astype(np.float32).flatten()
    def closure():
        opt_lb.zero_grad()
        loss = _sens_loss(m, Xt, yt, rain_t, t_tr_norm_t, t_full, cfg)
        loss.backward()
        return loss
    for _ in range(lbfgs_ep):
        opt_lb.step(closure)
        vl, _ = _sens_val(m)
        if vl < best_val:
            best_val = vl
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state)

    # ── metrics at the best-val checkpoint (all on held-out data) ──
    vl, vp = _sens_val(m)
    p_val = _sens_predict(m, X_val, t_sec_val)
    p_te  = _sens_predict(m, X_te,  t_sec_te)
    out = dict(
        val_rmse       = float(np.sqrt(mean_squared_error(theta_val, p_val))),
        val_phys       = vp,                               # unweighted, normalized
        test_r2        = float(r2_score(theta_te, p_te)),
        test_rmse      = float(np.sqrt(mean_squared_error(theta_te, p_te))),
        phys_over_data = float(cfg["LAM"] * vp / (vl + 1e-15)),   # at best ckpt, val
    )
    out.update(_sens_bcic_diag(m))
    return out


def _agg(runs):
    out = {}
    for k in runs[0]:
        a = np.array([r[k] for r in runs], dtype=float)
        out[k] = a.mean()
        out[k + "_std"] = a.std(ddof=1) if len(a) > 1 else np.nan
    return out


if RUN_SENSITIVITY:
    import pickle
    SENS_PKL = os.path.join(OUT, f"sens_cache_{RUN_TAG}.pkl")              # resumable: finished configs are reused
    _cache = pickle.load(open(SENS_PKL, "rb")) if os.path.exists(SENS_PKL) else {}
    if _cache: print(f"  resuming λ sweep: {len(_cache)} configurations already saved")
    _all_cfgs = {tuple(sorted(SENS_DEFAULT.items())), tuple(sorted(SENS_ZERO.items()))}
    for _p, _vals in SENS_GRID.items():
        for _v in set(_vals) | {SENS_DEFAULT[_p]}:
            _c = dict(SENS_DEFAULT); _c[_p] = _v
            _all_cfgs.add(tuple(sorted(_c.items())))
    _n_total = len(_all_cfgs) * len(SENS_SEEDS)
    _t_start = time.time()

    def run_cfg(cfg):
        """Each unique config is trained once per seed (default is shared by all params)."""
        key = tuple(sorted(cfg.items()))
        if key not in _cache:
            runs = [train_piml_sens(cfg, s, SENS_ADAM_EPOCHS, SENS_LBFGS_EPOCHS)
                    for s in SENS_SEEDS]
            _cache[key] = runs
            pickle.dump(_cache, open(SENS_PKL, "wb"))
            _done = len(set(_cache) & _all_cfgs) * len(SENS_SEEDS)
            _el = time.time() - _t_start
            print(f"    [{_done}/{_n_total} runs | {_el/60:.1f} min elapsed | "
                  f"ETA {(_el/_done)*(_n_total-_done)/60:.1f} min]")
        return _cache[key]

    print("\n" + "═" * 70)
    print(f"[Step 8] Loss-weight sensitivity (OAT) — budget {SENS_ADAM_EPOCHS} Adam + "
          f"{SENS_LBFGS_EPOCHS} L-BFGS, seeds={SENS_SEEDS}, unique configs={len(_all_cfgs)}")
    print("═" * 70)

    print("  reference: data-only (all λ = 0), same budget")
    ref_runs = run_cfg(SENS_ZERO)
    ref_agg  = _agg(ref_runs)
    print("  reference: default λ")
    def_runs = run_cfg(SENS_DEFAULT)
    def_val  = np.array([r["val_rmse"] for r in def_runs])

    # persistence reference (theta(t-1)) — same definition as the main run
    pers_val_rmse = float(np.sqrt(mean_squared_error(theta_val, X_val_raw[:, 1])))
    pers_te_rmse  = float(np.sqrt(mean_squared_error(theta_te,  X_te_raw[:, 1])))
    pers_te_r2    = float(r2_score(theta_te, X_te_raw[:, 1]))

    rows, per_seed_rows = [], []
    for pname, values in SENS_GRID.items():
        for v in sorted(set(values) | {SENS_DEFAULT[pname]}):
            cfg = dict(SENS_DEFAULT); cfg[pname] = v
            runs = run_cfg(cfg)
            is_def = bool(np.isclose(v, SENS_DEFAULT[pname]))
            row = {"param": pname, "value": v, "is_default": is_def,
                   "n_seeds": len(SENS_SEEDS)}
            row.update(_agg(runs))

            # paired Δ(val RMSE) vs default — same seeds, >0 means WORSE than default
            d = np.array([r["val_rmse"] for r in runs]) - def_val
            row["d_val_rmse_mean"] = float(d.mean())
            if len(d) > 1 and not is_def:
                half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))
                row["d_val_rmse_ci95"]  = float(half)
                row["differs_from_default"] = bool(abs(d.mean()) > half)
            else:
                row["d_val_rmse_ci95"], row["differs_from_default"] = np.nan, False
            rows.append(row)
            for s, r in zip(SENS_SEEDS, runs):
                per_seed_rows.append({"param": pname, "value": v, "seed": s, **r})

            print(f"  {pname}={v:<8g}{' (default)' if is_def else '          '}"
                  f" | val RMSE {row['val_rmse']:.5f}±{row['val_rmse_std']:.5f}"
                  f" | val phys {row['val_phys']:.4f}"
                  f" | test R² {row['test_r2']:.4f}"
                  f" | λP/D {row['phys_over_data']:.3f}"
                  f" | IC err {row['ic_err']:.4f} | LBC rms {row['lbc_rms']:.3f}"
                  f" | BC res {row['bc_res']:.3f}")

    sens_df = pd.DataFrame(rows)
    ref_row = {"param": "DATA_ONLY_all_lambda_0", "value": 0.0, "is_default": False,
               "n_seeds": len(SENS_SEEDS), **ref_agg}
    pd.concat([sens_df, pd.DataFrame([ref_row])], ignore_index=True).to_csv(
        os.path.join(OUT, "sensitivity_lambda.csv"), index=False)
    pd.DataFrame(per_seed_rows).to_csv(
        os.path.join(OUT, "sensitivity_lambda_per_seed.csv"), index=False)
    print(f"  → Saved: {os.path.join(OUT, 'sensitivity_lambda.csv')} (+ _per_seed.csv)")

    # ── compact verdict ─────────────────────────────────────────
    print("\n  ── Paired Δ(val RMSE) vs default λ (same seeds; >0 = worse; 95% CI half-width) ──")
    for _, r in sens_df[~sens_df.is_default].iterrows():
        flag = "DIFFERS" if r.differs_from_default else "n.s."
        print(f"  {r.param:<9s}={r.value:<7g} Δ={r.d_val_rmse_mean:+.5f} ± {r.d_val_rmse_ci95:.5f}  {flag}")
    print(f"  data-only reference  : val RMSE {ref_agg['val_rmse']:.5f}±{ref_agg['val_rmse_std']:.5f}"
          f" | val phys {ref_agg['val_phys']:.3f} | test R² {ref_agg['test_r2']:.4f}")
    print(f"  persistence reference: val RMSE {pers_val_rmse:.5f} | test R² {pers_te_r2:.4f}")
    print("  NOTE: n.s. with few seeds is low power — it does NOT prove equivalence; "
          "report as 'no detectable dependence within OAT screening'.")

    _d = sens_df[(sens_df.param == "LAM") & sens_df.is_default]
    if len(_d) and _d.phys_over_data.iloc[0] < 0.01:
        print("  [WARN] default λ_phys: λ·Phys/Data < 0.01 — physics term may be "
              "negligible; compare with the larger-λ rows.")

    # ── figure: metrics vs swept weight (categorical x so λ = 0 can be shown) ──
    _params = list(SENS_GRID.keys())
    fig, axes = plt.subplots(len(_params), 4, figsize=(15, 2.9 * len(_params)), squeeze=False)
    for i, pn in enumerate(_params):
        d = sens_df[sens_df.param == pn].sort_values("value").reset_index(drop=True)
        x = np.arange(len(d))
        cols = [("val_rmse", "Val RMSE (m³/m³)"),
                ("val_phys", "Val Richards residual (norm.)"),
                ("test_r2",  "Test R²"),
                SENS_TARGET[pn]]
        for j, (col, lab) in enumerate(cols):
            ax = axes[i][j]
            ax.errorbar(x, d[col], yerr=d[col + "_std"].fillna(0),
                        marker="o", lw=1.2, capsize=3, color=C["pred"])
            dd = d[d.is_default]
            ax.plot(dd.index, dd[col], "o", ms=11, mfc="none", mec=C["warn"], label="default")
            if col in ref_agg and col != "phys_over_data":
                ax.axhline(ref_agg[col], color=C["obs"], ls="--", lw=1.0,
                           label="data-only (λ=0)")
            if col == "val_rmse":
                ax.axhline(pers_val_rmse, color="gray", ls=":", lw=1.0, label="persistence")
            if col == "test_r2":
                ax.axhline(pers_te_r2, color="gray", ls=":", lw=1.0, label="persistence")
            if col == "test_rmse":
                ax.axhline(pers_te_rmse, color="gray", ls=":", lw=1.0, label="persistence")
            ax.set_xticks(x); ax.set_xticklabels([f"{v:g}" for v in d.value])
            ax.set_xlabel(pn); ax.set_ylabel(lab)
            if col == "val_phys": ax.set_yscale("log")
            light_grid(ax)
            if i == 0 and j == 0: ax.legend(fontsize=7)
    fig.suptitle(f"Loss-weight sensitivity of PIML (one-at-a-time, mean ± SD over "
                 f"{len(SENS_SEEDS)} seeds, reduced budget)", fontweight="bold")
    fig.tight_layout()
    savefig(fig, "fig_sensitivity_lambda.png")


## 5. Physics vs data-only at the λ-check budget
Paired difference over seeds and validation skill against persistence. Output: `sensitivity_skill_vs_persistence.csv`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 5. Physics vs data-only, and skill against persistence (λ-check budget)
#    Paired over the same seeds. Skill = 1 − MSE / MSE_persistence (persistence: θ(t) = θ(t−1)).
# ═══════════════════════════════════════════════════════════════════════════
def _runs(cfg): return _cache[tuple(sorted(cfg.items()))]
_def, _ref = _runs(SENS_DEFAULT), _runs(SENS_ZERO)
d = np.array([a["val_rmse"] - b["val_rmse"] for a, b in zip(_def, _ref)])     # >0: physics worse
half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))
print(f"Default λ minus data-only, val RMSE: {d.mean():+.5f} ± {half:.5f} (95% CI, n={len(d)} seeds)"
      f" -> {'physics WORSE' if d.mean() - half > 0 else 'physics BETTER' if d.mean() + half < 0 else 'no detectable difference'}")

pers_val = float(np.sqrt(mean_squared_error(theta_val, X_val_raw[:, 1])))
rows_ss = []
for name, cfg in [("data-only (all λ=0)", SENS_ZERO), ("default λ", SENS_DEFAULT)] + \
        [(f"{p}={v:g}", {**SENS_DEFAULT, p: v}) for p, vals in SENS_GRID.items() for v in vals
         if not np.isclose(v, SENS_DEFAULT[p])]:
    r = np.array([x["val_rmse"] for x in _runs(cfg)])
    rows_ss.append(dict(config=name, val_rmse=r.mean(), val_rmse_sd=r.std(ddof=1),
                        skill_vs_persistence=1 - (r ** 2).mean() / pers_val ** 2))
ss_df = pd.DataFrame(rows_ss)
print(ss_df.round(5).to_string(index=False))
print(f"Persistence val RMSE = {pers_val:.5f}. Skill > 0 means better than repeating the last reading.")
ss_df.to_csv(os.path.join(OUT, "sensitivity_skill_vs_persistence.csv"), index=False)

## 8. Lower-boundary and initial-condition scenarios
Free drainage (sensor and uniform initial state), no-flux and water table (weights 1 and 10). Each scenario is checked for whether its condition was actually met at 0.5 m. Output: `sensitivity_lower_bc_ic_per_seed.csv`, `sensitivity_lower_bc_ic_summary.csv`, `sensitivity_lower_bc_identifiability.csv`, `fig_sensitivity_lower_bc_ic.png`.

*(Cells are numbered by topic; this cell runs before the multi-seed cell, as in the original runs.)*


In [ ]:
if not RUN["bc_scen"]:
    print("Boundary scenarios skipped (RUN['bc_scen'] = False).")
else:
    # ═══════════════════════════════════════════════════════════════════════════
    # 8. Lower-boundary and initial-condition scenarios
    #    How much does the solution depend on the (unobserved) lower boundary and initial state?
    #    Uses train_piml_sens at the λ-check budget and the same seeds.
    #    - each scenario has its own lower-boundary weight; the alternatives are also run at a
    #      strong weight (10) so that they are actually imposed
    #    - checks whether each condition was MET at z = 0.5 m (free drainage dψ/dz ≈ 0;
    #      no-flux q ≈ 0; water table ψ ≈ PSI_WT); only scenarios that pass are interpreted
    #    - reports the profile (θ at 0, 0.30, 0.50 m), the shift of the deep profile, and FoS
    #      from θ at the sensor and at 0.50 m
    #    - no-flux is imposed on the gradient, dψ/dz = 1 (zero total-head gradient)
    #    - results saved after every run (resumable)
    # ═══════════════════════════════════════════════════════════════════════════
    import os, time, pickle

    # ── Lower-boundary penalties (same as the boundary-mode cell, with version tag) ──
    #    z positive DOWNWARD; downward Darcy flux q = K(ψ)·(1 − dψ/dz).
    #      free_drainage : dψ/dz = 0      (unit total gradient, q = K)
    #      no_flux       : dψ/dz = 1      (zero total-head gradient, q = 0 for ANY K)
    #      water_table   : ψ = PSI_WT     (fixed pressure head at the base)
    LBC_VERSION = 2
    def lower_bc_loss(m, t_norm, feat, mode=None):
        mode = mode or LBC_MODE
        B = t_norm.shape[0]
        z = torch.ones(B, 1, dtype=torch.float32, device=DEVICE).requires_grad_(True)   # z = Z_MAX
        theta = m(z, t_norm, feat)
        psi = vg_psi_t(theta, VG)
        if mode == "water_table":
            return ((psi - PSI_WT) ** 2).mean()
        dpsi_dz = torch.autograd.grad(psi.sum(), z, create_graph=True)[0] / Z_MAX
        if mode == "free_drainage":
            return (dpsi_dz ** 2).mean()
        if mode == "no_flux":
            return ((dpsi_dz - 1.0) ** 2).mean()
        raise ValueError(mode)
    lower_bc_free_drainage_loss = lambda m, t_norm, feat: lower_bc_loss(m, t_norm, feat)   # used by the training loss

    #             lower BC         IC         λ_LBC
    BC_SCENARIOS = [("free_drainage", "sensor",  0.1),    # reference (= main run)
                    ("free_drainage", "uniform", 0.1),    # initial-condition test
                    ("no_flux",       "sensor",  1.0),    # moderate and strong enforcement: is the
                    ("no_flux",       "sensor",  10.0),   # accuracy cost real at both weights?
                    ("water_table",   "sensor",  1.0),
                    ("water_table",   "sensor",  10.0)]   # (at λ = 0.1 the alternatives are not imposed)
    BC_SEEDS = SENS_SEEDS
    REF = ("free_drainage", "sensor", 0.1)

    # tolerances for "the boundary condition was met" (medians over the test inputs)
    TOL_DPSI = 0.10        # |dψ/dz| for free drainage (−)
    TOL_QREL = 0.20        # |q_base| / K(θ_base) for no-flux (−): flow < 20 % of gravity drainage
    TOL_PSI  = 0.02        # |ψ_base − PSI_WT| for water table (m)

    BC_PKL = os.path.join(OUT, f"bc_ic_scenarios_{RUN_TAG}.pkl")
    bc_store = pickle.load(open(BC_PKL, "rb")) if os.path.exists(BC_PKL) else {}
    _stale = [k for k, v in bc_store.items() if k[0] == "no_flux" and v.get("lbc_version", 1) < LBC_VERSION]
    for k in _stale: del bc_store[k]
    if _stale:
        pickle.dump(bc_store, open(BC_PKL, "wb"))
        print(f"discarded {len(_stale)} no-flux runs made with an older penalty — they will be retrained")
    if bc_store:
        print(f"resuming: {len(bc_store)} runs already done")

    _ORIG_PINN = RichardsPINN
    _last = {}
    def _recording_pinn(*a, **k):            # keeps a handle on the model trained inside train_piml_sens
        m = _ORIG_PINN(*a, **k); _last["m"] = m; return m

    def _fos(theta):
        """Infinite-slope FoS exactly as in the paper (Eq. 14–16), from a θ series."""
        Se  = vg_Se_np(theta, VG)
        u   = Se * 9.81 * np.clip(vg_psi_np(theta, VG), -165.0, 0.0)
        b   = np.radians(33.0)
        sn  = 14.715 * 1.0 * np.cos(b) ** 2
        tau = 14.715 * 1.0 * np.sin(b) * np.cos(b)
        return (5.0 + np.maximum(sn - u, 0) * np.tan(np.radians(28.0))) / tau

    def _profile_diag(m):
        """θ, ψ, dψ/dz and downward flux at 0 m, 0.30 m and 0.50 m on the TEST inputs."""
        m.eval(); n = Xte.shape[0]
        t_n = to_t(((t_sec_te - T_MIN) / (T_MAX - T_MIN))[:, None])
        out = {}
        for name, zv in [("surf", 0.0), ("sens", Z_SENSOR), ("base", float(Z_COLLOC.max()))]:
            z = torch.full((n, 1), zv / Z_MAX, dtype=torch.float32, device=DEVICE).requires_grad_(True)
            th = m(z, t_n, Xte)
            psi = vg_psi_t(th, VG)
            dpsi = torch.autograd.grad(psi.sum(), z)[0] / Z_MAX
            K = vg_K_t(th, VG)
            q = K * (1.0 - dpsi)                                      # downward flux, z positive downward
            f = lambda x: x.detach().cpu().numpy().ravel()
            out[f"theta_{name}"], out[f"psi_{name}"], out[f"dpsi_{name}"] = f(th), f(psi), f(dpsi)
            out[f"q_{name}_Ks"], out[f"qrel_{name}"] = f(q) / VG["Ks_ms"], f(q) / f(K)
        m.train()
        return out

    def _bc_met(lbc, d):
        if lbc == "free_drainage": return bool(abs(np.median(d["dpsi_base"])) < TOL_DPSI)
        if lbc == "no_flux":       return bool(abs(np.median(d["qrel_base"])) < TOL_QREL)
        if lbc == "water_table":   return bool(abs(np.median(d["psi_base"]) - PSI_WT) < TOL_PSI)

    n_total = len(BC_SCENARIOS) * len(BC_SEEDS); t_start = time.time()
    try:
        globals()["RichardsPINN"] = _recording_pinn
        for lbc, ic, lam in BC_SCENARIOS:
            for s in BC_SEEDS:
                key = (lbc, ic, lam, s)
                if key in bc_store: continue
                LBC_MODE, IC_MODE = lbc, ic                         # read by the boundary-mode loss functions
                r = train_piml_sens({**SENS_DEFAULT, "LAM_LBC": lam}, s, SENS_ADAM_EPOCHS, SENS_LBFGS_EPOCHS)
                d = _profile_diag(_last["m"])
                bc_store[key] = dict(metrics=r, prof=d, lbc_version=LBC_VERSION)
                pickle.dump(bc_store, open(BC_PKL, "wb"))            # resumable
                el = time.time() - t_start
                print(f"  {lbc:<13} IC={ic:<7} λ_LBC={lam:<5g} seed {s}: val RMSE {r['val_rmse']:.5f} | "
                      f"θ(0, 0.3, 0.5 m) = {d['theta_surf'].mean():.3f}, {d['theta_sens'].mean():.3f}, {d['theta_base'].mean():.3f} | "
                      f"ψ_base {np.median(d['psi_base']):+.3f} m | dψ/dz_base {np.median(d['dpsi_base']):+.3f} | "
                      f"q_base/K {np.median(d['qrel_base']):+.3f} | BC met: {_bc_met(lbc, d)} "
                      f"[{len(bc_store)}/{n_total}, {el/60:.0f} min]", flush=True)
    finally:
        globals()["RichardsPINN"] = _ORIG_PINN
        LBC_MODE, IC_MODE = "free_drainage", "sensor"               # restore main-run setting

    # ── summary table (mean ± SD over seeds) ─────────────────────────────────────
    rows = []
    for (lbc, ic, lam, s), v in bc_store.items():
        d, r = v["prof"], v["metrics"]
        rows.append(dict(lbc=lbc, ic=ic, lam_lbc=lam, seed=s, val_rmse=r["val_rmse"], test_rmse=r["test_rmse"],
                         theta_surf=d["theta_surf"].mean(), theta_sens=d["theta_sens"].mean(), theta_base=d["theta_base"].mean(),
                         psi_base=np.median(d["psi_base"]), dpsi_base=np.median(d["dpsi_base"]),
                         q_base_over_Ks=np.median(d["q_base_Ks"]), q_base_over_K=np.median(d["qrel_base"]),
                         bc_met=_bc_met(lbc, d),
                         fos_min_sensor=_fos(d["theta_sens"]).min(), fos_mean_sensor=_fos(d["theta_sens"]).mean(),
                         fos_min_0p5m=_fos(d["theta_base"]).min(), fos_mean_0p5m=_fos(d["theta_base"]).mean()))
    bc_df = pd.DataFrame(rows)
    bc_df.to_csv(os.path.join(OUT, "sensitivity_lower_bc_ic_per_seed.csv"), index=False)
    num = [c for c in bc_df.columns if c not in ("lbc", "ic", "lam_lbc", "seed", "bc_met")]
    summ = bc_df.groupby(["lbc", "ic", "lam_lbc"])[num].agg(["mean", "std"])
    summ["bc_met_all_seeds"] = bc_df.groupby(["lbc", "ic", "lam_lbc"])["bc_met"].all()
    summ.to_csv(os.path.join(OUT, "sensitivity_lower_bc_ic_summary.csv"))
    show = ["val_rmse", "theta_surf", "theta_sens", "theta_base", "psi_base", "dpsi_base", "q_base_over_K", "fos_min_sensor", "fos_min_0p5m"]
    print("\nMean over seeds:")
    print(bc_df.groupby(["lbc", "ic", "lam_lbc"])[show + ["bc_met"]].mean().round(4).to_string())

    # ── identifiability: how far each scenario moves the profile, vs the test error ──
    test_rmse_ref = bc_df[(bc_df.lbc == REF[0]) & (bc_df.ic == REF[1]) & (bc_df.lam_lbc == REF[2])].test_rmse.mean()
    print(f"\nMean |Δθ| vs {REF[0]} (IC={REF[1]}, λ_LBC={REF[2]}), same seed, test inputs "
          f"[reference test RMSE at the sensor = {test_rmse_ref:.4f}]:")
    id_rows = []
    for lbc, ic, lam in BC_SCENARIOS:
        if (lbc, ic, lam) == REF: continue
        dz = {z: np.mean([np.abs(bc_store[(lbc, ic, lam, s)]["prof"][f"theta_{z}"] - bc_store[REF + (s,)]["prof"][f"theta_{z}"]).mean()
                          for s in BC_SEEDS]) for z in ("surf", "sens", "base")}
        dval = np.array([bc_store[(lbc, ic, lam, s)]["metrics"]["val_rmse"] - bc_store[REF + (s,)]["metrics"]["val_rmse"] for s in BC_SEEDS])
        met = all(_bc_met(lbc, bc_store[(lbc, ic, lam, s)]["prof"]) for s in BC_SEEDS)
        id_rows.append(dict(lbc=lbc, ic=ic, lam_lbc=lam, bc_met=met, d_theta_sensor=dz["sens"], d_theta_0p5m=dz["base"],
                            d_theta_surface=dz["surf"], d_val_rmse=dval.mean(), ratio_0p5m_to_test_rmse=dz["base"] / test_rmse_ref))
        print(f"  {lbc:<13} IC={ic:<7} λ_LBC={lam:<5g} BC met={str(met):<5} | sensor {dz['sens']:.4f} | 0.5 m {dz['base']:.4f} "
              f"| surface {dz['surf']:.4f} m³/m³ | Δ val RMSE {dval.mean():+.5f}")
    id_df = pd.DataFrame(id_rows); id_df.to_csv(os.path.join(OUT, "sensitivity_lower_bc_identifiability.csv"), index=False)

    # ── plain-language verdict for the response letter ───────────────────────────
    met = id_df[id_df.bc_met & (id_df.ic == "sensor")]
    ic_row = id_df[(id_df.lbc == "free_drainage") & (id_df.ic == "uniform")]
    print("\nVERDICT")
    if len(ic_row):
        print(f"  Initial condition: sensor-only vs uniform profile changes θ by ≤ "
              f"{ic_row[['d_theta_sensor','d_theta_0p5m','d_theta_surface']].values.max():.4f} m³/m³ at any depth.")
    if len(met):
        print(f"  Lower boundary (scenarios where the condition was met): θ at the sensor changes by ≤ {met.d_theta_sensor.max():.4f}, "
              f"but θ at 0.5 m changes by up to {met.d_theta_0p5m.max():.4f} m³/m³ "
              f"({met.ratio_0p5m_to_test_rmse.max():.1f} × the sensor test RMSE).")
        print("  -> The 0.30 m observations do not constrain the profile below the sensor; the deep state depends on the "
              "assumed lower boundary (free drainage adopted; a deeper probe is needed to verify it).")
    else:
        print("  No alternative boundary condition was met even at λ_LBC = 10 — increase λ_LBC (e.g. 100) or impose the "
              "condition as a hard constraint before drawing conclusions.")
    not_met = id_df[~id_df.bc_met]
    if len(not_met):
        print("  Not met (do NOT use for conclusions): " +
              ", ".join(f"{r.lbc} λ={r.lam_lbc:g}" for _, r in not_met.iterrows()))

    # ── figure: depth profiles and stability under each scenario ─────────────────
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    zs = [0.0, Z_SENSOR, float(Z_COLLOC.max())]
    for lbc, ic, lam in BC_SCENARIOS:
        g_ = bc_df[(bc_df.lbc == lbc) & (bc_df.ic == ic) & (bc_df.lam_lbc == lam)]
        prof = [g_.theta_surf.mean(), g_.theta_sens.mean(), g_.theta_base.mean()]
        ok = g_.bc_met.all() or lbc == "free_drainage"
        ax[0].plot(prof, zs, marker="o" if ok else "x", ls="none", ms=8, label=f"{lbc}, IC={ic}, λ={lam:g}" + ("" if ok else " (BC not met)"))
    ax[0].axhline(Z_SENSOR, color="gray", lw=0.6, ls="--"); ax[0].text(ax[0].get_xlim()[0], Z_SENSOR - 0.01, " sensor", fontsize=8, color="gray")
    ax[0].invert_yaxis(); ax[0].set_xlabel(r"Mean predicted $\theta$ on test inputs (m$^3$ m$^{-3}$)"); ax[0].set_ylabel("Depth z (m)")
    ax[0].set_title("(a) Predicted moisture profile"); ax[0].legend(fontsize=7)
    lab = [f"{l}\n{i}, λ={x:g}" for l, i, x in BC_SCENARIOS]
    xs = np.arange(len(BC_SCENARIOS))
    for off, col, nm in [(-0.18, "fos_min_sensor", "θ at sensor (paper)"), (0.18, "fos_min_0p5m", "θ at 0.5 m")]:
        vals = [bc_df[(bc_df.lbc == l) & (bc_df.ic == i) & (bc_df.lam_lbc == x)][col] for l, i, x in BC_SCENARIOS]
        ax[1].errorbar(xs + off, [v.mean() for v in vals], yerr=[v.std() for v in vals], fmt="o", capsize=3, label=nm)
    ax[1].set_xticks(xs, lab, fontsize=7); ax[1].set_ylabel("Minimum FoS (test window)")
    lo = bc_df[["fos_min_sensor", "fos_min_0p5m"]].values.min(); ax[1].set_ylim(lo - 0.02, None)
    ax[1].set_title("(b) Minimum FoS (mean ± SD, 3 seeds); warning level 1.3 is far below"); ax[1].legend(fontsize=8)
    fig.tight_layout(); savefig(fig, "fig_sensitivity_lower_bc_ic.png")


## 6. Multi-seed result (full budget) — main result of the paper
PIML and data-only network, 3 seeds each, 400 Adam + 200 L-BFGS. Output: `multiseed_summary.csv` (per-seed test R² and RMSE).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 6. Multi-seed result — PIML vs data-only vs persistence (FULL budget)
#    Main result of the paper. Each seed trains both networks with the main-run budget.
#    Finished seeds are saved and skipped after a disconnect.
# ═══════════════════════════════════════════════════════════════════════════
RUN_MULTISEED = True
MS_SEEDS      = BUDGET["seeds"]

def train_ml_seed(seed, adam_ep=ADAM_EPOCHS, lbfgs_ep=LBFGS_EPOCHS):
    """Data-only baseline, same architecture / schedule as the main-run ML baseline."""
    torch.manual_seed(seed); np.random.seed(seed)
    loader.generator.manual_seed(seed)
    m   = RichardsPINN(feat_dim=X_tr.shape[1]).to(DEVICE)
    opt = optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    sch = optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    best_val, best_state = np.inf, None

    def _val():
        m.eval()
        with torch.no_grad():
            v = mse_loss((m(z_v_rep, t_v_norm, Xv) - y_min_t) * y_scl_t, yv).item()
        m.train()
        return v

    for _ in range(adam_ep):
        m.train()
        for (Xb, yb, rb, tb_norm, dtb, tb_sec) in loader:
            opt.zero_grad()
            z_d  = z_sens_norm.expand(Xb.shape[0], -1)
            loss = mse_loss((m(z_d, tb_norm, Xb) - y_min_t) * y_scl_t, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
        sch.step()
        v = _val()
        if v < best_val:
            best_val, best_state = v, {k: x.clone() for k, x in m.state_dict().items()}

    m.load_state_dict(best_state)
    opt_lb = optim.LBFGS(m.parameters(), lr=1.0, max_iter=LBFGS_MAX_ITER,
                         history_size=LBFGS_HISTORY, line_search_fn="strong_wolfe")
    def closure():
        opt_lb.zero_grad()
        z_d  = z_sens_norm.expand(Xt.shape[0], -1)
        loss = mse_loss((m(z_d, t_tr_norm_t, Xt) - y_min_t) * y_scl_t, yt)
        loss.backward()
        return loss
    for _ in range(lbfgs_ep):
        opt_lb.step(closure)
        v = _val()
        if v < best_val:
            best_val, best_state = v, {k: x.clone() for k, x in m.state_dict().items()}
    m.load_state_dict(best_state)

    p_te = _sens_predict(m, X_te, t_sec_te)
    return float(r2_score(theta_te, p_te)), float(np.sqrt(mean_squared_error(theta_te, p_te)))

if RUN_MULTISEED:
    import time
    pers_r2   = float(r2_score(theta_te, X_te_raw[:, 1]))
    pers_rmse = float(np.sqrt(mean_squared_error(theta_te, X_te_raw[:, 1])))
    _ms_csv = os.path.join(OUT, f"multiseed_partial_{RUN_TAG}.csv")       # resumable: finished seeds are reused
    rows = pd.read_csv(_ms_csv).to_dict("records") if os.path.exists(_ms_csv) else []
    for s in MS_SEEDS:
        if s in {int(r["seed"]) for r in rows}:
            print(f"  seed {s}: already saved, skipped"); continue
        t0 = time.time()
        pr = train_piml_sens(SENS_DEFAULT, s, ADAM_EPOCHS, LBFGS_EPOCHS)
        ml_r2, ml_rmse = train_ml_seed(s)
        rows.append(dict(seed=s, piml_r2=pr["test_r2"], piml_rmse=pr["test_rmse"],
                         ml_r2=ml_r2, ml_rmse=ml_rmse))
        pd.DataFrame(rows).to_csv(_ms_csv, index=False)
        print(f"  seed {s}: PIML R²={pr['test_r2']:.4f} RMSE={pr['test_rmse']:.5f} | "
              f"ML R²={ml_r2:.4f} RMSE={ml_rmse:.5f} | {time.time()-t0:.0f}s")
    ms = pd.DataFrame(rows)
    ms["rmse_gain_ml_minus_piml"] = ms["ml_rmse"] - ms["piml_rmse"]
    ddof = 1 if len(ms) > 1 else 0
    print("\n  ── Test-set summary (mean ± SD over seeds) ──")
    for name, col in [("PIML", "piml"), ("ML  ", "ml")]:
        print(f"  {name}: R² = {ms[col+'_r2'].mean():.4f} ± {ms[col+'_r2'].std(ddof=ddof):.4f} | "
              f"RMSE = {ms[col+'_rmse'].mean():.5f} ± {ms[col+'_rmse'].std(ddof=ddof):.5f}")
    print(f"  Persistence: R² = {pers_r2:.4f} | RMSE = {pers_rmse:.5f}")
    print(f"  Paired RMSE gain (ML − PIML): {ms['rmse_gain_ml_minus_piml'].mean():.5f} "
          f"± {ms['rmse_gain_ml_minus_piml'].std(ddof=ddof):.5f}  (>0 favours PIML)")
    ms.to_csv(os.path.join(OUT, "multiseed_summary.csv"), index=False)


## 7. Skill per seed, paired test and raw-reading check
Skill = 1 − MSE/MSE_persistence per seed, paired t-test PIML vs data-only, and scores of the main-run models against the **unsmoothed** sensor readings. Output: `leakage_check_raw_target.csv`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 7. Skill per seed, paired test, and raw-reading check
#    Run after the multi-seed cell. The models are trained on a smoothed target (rolling median);
#    here they are also scored against the UNSMOOTHED sensor reading, so smoothing cannot
#    flatter the result.
# ═══════════════════════════════════════════════════════════════════════════
ms = pd.read_csv(os.path.join(OUT, "multiseed_summary.csv"))
p_mse = mean_squared_error(theta_te, X_te_raw[:, 1])
for c in ("piml", "ml"):
    ms[f"{c}_skill"] = 1 - ms[f"{c}_rmse"] ** 2 / p_mse
print(ms.round(5).to_string(index=False))
dd = ms.ml_rmse - ms.piml_rmse
tt = stats.ttest_rel(ms.piml_rmse, ms.ml_rmse) if len(ms) > 1 else None
print(f"\nPIML: R² {ms.piml_r2.mean():.4f} ± {ms.piml_r2.std(ddof=1):.4f} | skill {ms.piml_skill.mean():.2f}")
print(f"ML  : R² {ms.ml_r2.mean():.4f} ± {ms.ml_r2.std(ddof=1):.4f} | skill {ms.ml_skill.mean():.2f}")
if tt is not None:
    print(f"Paired RMSE (ML − PIML) = {dd.mean():+.5f}; paired t = {tt.statistic:.2f}, p = {tt.pvalue:.3f} (n = {len(ms)}, low power)")

theta_te_raw = THETA_RAW_BY_TS.reindex(pd.to_datetime(ts_te)).values
ok = ~np.isnan(theta_te_raw)
rows_raw = []
for name, p in [("Persistence (lag-1 input)", X_te_raw[:, 1]), ("PIML (main run)", theta_pred_te), ("ML (main run)", theta_ml_te)]:
    rows_raw.append(dict(model=name,
                         r2_smoothed=r2_score(theta_te, p), rmse_smoothed=np.sqrt(mean_squared_error(theta_te, p)),
                         r2_raw=r2_score(theta_te_raw[ok], p[ok]), rmse_raw=np.sqrt(mean_squared_error(theta_te_raw[ok], p[ok]))))
print("\nTarget = smoothed (as trained) vs raw sensor reading:")
print(pd.DataFrame(rows_raw).round(5).to_string(index=False))
pd.DataFrame(rows_raw).to_csv(os.path.join(OUT, "leakage_check_raw_target.csv"), index=False)

## 9. LSTM baseline
Two-layer LSTM (64 hidden units, ≈57k parameters), same split, target, loss and schedule; rain as the current value or as a 30-min history. Output: `lstm_results.csv`, `lstm_final.pt`.


In [ ]:
if not RUN["lstm"]:
    print("LSTM baseline skipped (RUN['lstm'] = False).")
else:
    # ═══════════════════════════════════════════════════════════════════════════
    # 9. LSTM baseline (same split, target, loss and optimiser schedule as PIML / data-only)
    #    current: rain R(t) repeated over the window (same information as the PIML);
    #    history: R(t-14..t), one value per step (stronger baseline).
    #    Saves one checkpoint, OUT/lstm_final.pt (best validation RMSE over all runs).
    # ═══════════════════════════════════════════════════════════════════════════
    import time

    LSTM_SEEDS    = BUDGET["seeds"]
    LSTM_VARIANTS = ["current", "history"]   # current: R(t) repeated (same info as PIML/ML)
                                             # history: R(t-14..t) per step (stronger baseline)
    LSTM_HIDDEN, LSTM_LAYERS = 64, 2         # ≈57k parameters (PINN 76,801)


    def _lstm_inputs(d, lag, rain_mode):
        """(N, lag, 2), oldest → newest, channels [rain/RAIN_MAX, scaled θ]. Rows align with build_features()."""
        seg = d["seg"].values
        th  = d["theta"].values.astype(np.float32)
        rn  = d["rain"].values.astype(np.float32)
        idx = np.arange(lag, len(d))
        idx = idx[seg[idx] == seg[idx - lag]]                                   # never cross a data gap
        if "held" in d: idx = idx[~d["held"].values[idx].astype(bool)]        # same targets as build_features()
        th_win = np.stack([th[idx - lag + j] for j in range(lag)], axis=1)      # θ(t-lag)…θ(t-1)
        if rain_mode == "current":
            r_win = np.repeat(rn[idx][:, None], lag, axis=1)
        else:                                                                   # "history"
            r_win = np.stack([rn[idx - lag + 1 + j] for j in range(lag)], axis=1)  # R(t-lag+1)…R(t)
        th_n = (th_win - VG["theta_r"]) / (VG["theta_s"] - VG["theta_r"])
        return np.stack([r_win / RAIN_MAX, th_n], axis=2).astype(np.float32)


    _SEQ = {}
    for _mode in LSTM_VARIANTS:
        _S = {k: _lstm_inputs(d, LAG, _mode) for k, d in [("tr", df_tr), ("val", df_val), ("te", df_te)]}
        for k, Xraw in [("tr", X_tr_raw), ("val", X_val_raw), ("te", X_te_raw)]:      # alignment checks
            assert _S[k].shape == (len(Xraw), LAG, 2)
            assert np.allclose(_S[k][:, ::-1, 1] * (VG["theta_s"] - VG["theta_r"]) + VG["theta_r"],
                               Xraw[:, 1:], atol=1e-4)
            assert np.allclose(_S[k][:, -1, 0] * RAIN_MAX, Xraw[:, 0], atol=1e-4)
        _SEQ[_mode] = {k: to_t(v) for k, v in _S.items()}


    class SoilLSTM(nn.Module):
        def __init__(self, in_ch=2, h=LSTM_HIDDEN, layers=LSTM_LAYERS):
            super().__init__()
            self.lstm = nn.LSTM(in_ch, h, num_layers=layers, batch_first=True)
            self.head = nn.Sequential(nn.Linear(h, h), nn.SiLU(),
                                      nn.Linear(h, h // 2), nn.SiLU(),
                                      nn.Linear(h // 2, 1), nn.Sigmoid())
            self.theta_lo, self.theta_hi = VG["theta_r"] + 0.01, VG["theta_s"] - 0.01

        def forward(self, x):
            out, _ = self.lstm(x)
            return self.theta_lo + (self.theta_hi - self.theta_lo) * self.head(out[:, -1, :])


    print(f"Parameters → PINN {sum(p.numel() for p in model.parameters()):,} | "
          f"LSTM {sum(p.numel() for p in SoilLSTM().parameters()):,}")


    def train_lstm_seed(seed, rain_mode, adam_ep=ADAM_EPOCHS, lbfgs_ep=LBFGS_EPOCHS):
        torch.manual_seed(seed); np.random.seed(seed)
        S   = _SEQ[rain_mode]
        m   = SoilLSTM().to(DEVICE)
        opt = optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
        sch = optim.lr_scheduler.LambdaLR(opt, lr_lambda)
        ld  = DataLoader(TensorDataset(S["tr"], yt), batch_size=BATCH, shuffle=True,
                         generator=torch.Generator().manual_seed(seed))
        best_val, best_state = np.inf, None

        def _val():
            m.eval()
            with torch.no_grad():
                v = mse_loss((m(S["val"]) - y_min_t) * y_scl_t, yv).item()
            m.train()                                    # cuDNN RNN backward needs train mode
            return v

        for _ in range(adam_ep):                         # Adam (mini-batch)
            m.train()
            for Xb, yb in ld:
                opt.zero_grad()
                loss = mse_loss((m(Xb) - y_min_t) * y_scl_t, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
            sch.step()
            v = _val()
            if v < best_val:
                best_val, best_state = v, {k: x.clone() for k, x in m.state_dict().items()}

        m.load_state_dict(best_state); m.train()         # L-BFGS (full-batch) from Adam best
        opt_lb = optim.LBFGS(m.parameters(), lr=1.0, max_iter=LBFGS_MAX_ITER,
                             history_size=LBFGS_HISTORY, line_search_fn="strong_wolfe")
        def closure():
            opt_lb.zero_grad()
            loss = mse_loss((m(S["tr"]) - y_min_t) * y_scl_t, yt)
            loss.backward()
            return loss
        for _ in range(lbfgs_ep):
            opt_lb.step(closure)
            v = _val()
            if v < best_val:
                best_val, best_state = v, {k: x.clone() for k, x in m.state_dict().items()}
        m.load_state_dict(best_state); m.eval()

        with torch.no_grad():
            p_val = m(S["val"]).cpu().numpy().flatten()
            p_te  = m(S["te"]).cpu().numpy().flatten()
        out = dict(seed=seed, variant=rain_mode,
                   val_rmse=float(np.sqrt(mean_squared_error(theta_val, p_val))),
                   test_r2=float(r2_score(theta_te, p_te)),
                   test_rmse=float(np.sqrt(mean_squared_error(theta_te, p_te))),
                   test_mae=float(np.mean(np.abs(theta_te - p_te))))
        out["_state"] = {k: v.detach().cpu() for k, v in m.state_dict().items()}   # not written to CSV
        return out


    rows, best = [], None
    for mode in LSTM_VARIANTS:
        for s in LSTM_SEEDS:
            t0 = time.time()
            _p = os.path.join(OUT, "lstm_runs", f"lstm_{mode}_seed{s}.pt")      # resumable
            os.makedirs(os.path.dirname(_p), exist_ok=True)
            if os.path.exists(_p):
                r = torch.load(_p, weights_only=False, map_location="cpu"); print("  (resumed)", end="")
            else:
                r = train_lstm_seed(s, mode); torch.save(r, _p)
            if best is None or r["val_rmse"] < best["val_rmse"]:
                best = r                                  # keep only the best-validation run in memory
            rows.append({k: v for k, v in r.items() if k != "_state"})
            print(f"  LSTM-{mode:<8} seed {s}: val RMSE={r['val_rmse']:.5f} | test R²={r['test_r2']:.4f}  "
                  f"RMSE={r['test_rmse']:.5f}  ({time.time()-t0:.0f}s)")

    # ── single final checkpoint ──
    torch.save({
        "model_state_dict": best["_state"],
        "model_architecture": {"class": "SoilLSTM", "in_ch": 2, "hidden": LSTM_HIDDEN,
                               "num_layers": LSTM_LAYERS, "rain_mode": best["variant"], "lag": LAG},
        "selected_by": "lowest validation RMSE over all (variant, seed) runs",
        "seed": best["seed"],
        "hyperparameters": {"lr": LR, "batch": BATCH, "adam_epochs": ADAM_EPOCHS, "lbfgs_epochs": LBFGS_EPOCHS},
        "scalers": {"rain_max": RAIN_MAX, "vg_theta_r": VG["theta_r"], "vg_theta_s": VG["theta_s"],
                    "y_min": y_min_t.detach().cpu().numpy().tolist(),
                    "y_scl": y_scl_t.detach().cpu().numpy().tolist()},
        "test_metrics": {k: best[k] for k in ("test_r2", "test_rmse", "test_mae")},
        "device_id": 108,
    }, os.path.join(OUT, "lstm_final.pt"))
    print(f"\n  Saved lstm_final.pt → LSTM-{best['variant']}, seed {best['seed']}, "
          f"val RMSE={best['val_rmse']:.5f}, test R²={best['test_r2']:.4f}")

    lstm_df = pd.DataFrame(rows)
    pers_r2   = float(r2_score(theta_te, X_te_raw[:, 1]))
    pers_rmse = float(np.sqrt(mean_squared_error(theta_te, X_te_raw[:, 1])))

    print("\n  ── LSTM test summary (mean ± SD over seeds) ──")
    for mode in LSTM_VARIANTS:
        g = lstm_df[lstm_df["variant"] == mode]; dd = 1 if len(g) > 1 else 0
        print(f"  LSTM-{mode:<8}: R² = {g['test_r2'].mean():.4f} ± {g['test_r2'].std(ddof=dd):.4f} | "
              f"RMSE = {g['test_rmse'].mean():.5f} ± {g['test_rmse'].std(ddof=dd):.5f}")
    print(f"  Persistence    : R² = {pers_r2:.4f} | RMSE = {pers_rmse:.5f}")
    print(f"  Main run (seed {SEED}) PIML: R² = {r2_te:.4f} | RMSE = {rmse_te:.5f}")
    print(f"  Main run (seed {SEED}) ML  : R² = {r2_ml_te:.4f} | RMSE = {rmse_ml_te:.5f}")

    lstm_df.to_csv(os.path.join(OUT, "lstm_results.csv"), index=False)

## 10. Transfer Dev108 → Dev107
Zero-shot, 2-parameter recalibration, last-layer and full fine-tuning (with and without physics) and training from scratch, with the last 3, 7, 14 and 30 days of Dev107 data; scored on the same test window as Dev108. Output: `transfer_107_<tag>.csv` (per seed), `transfer_107_summary.csv`, `fig_transfer_108_to_107.png`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 10. Transfer to a second node: Dev108 -> Dev107
#     Both nodes are sandy clay loam on the same hillslope (Table 1), so they share θr, α, n.
#     They differ in installation: θs is anchored to each probe's own record (0.55 vs 0.463)
#     and Ks comes from each node's texture (0.43 vs 0.41 cm/h). Transfer is therefore done
#     in EFFECTIVE-SATURATION space: inputs, output bounds and the physics loss all use the
#     TARGET node's θr, θs, Ks; the network weights carry over.
#
#     A Dev107 model is set up with only the last N days of Dev107 data (N = 3, 7, 14, 30 days
#     with data) before the test window, and is scored on the SAME window as Dev108's test set.
#       zero_shot          Dev108 PINN, no Dev107 data, no weights changed
#       recal              zero-shot output + linear map θ = a·θ̂ + b fitted on the N days (2 numbers)
#       head_pinn          last layer only retrained (65 of 76,801 weights), with physics
#       full_pinn          all layers retrained from the Dev108 PINN, with physics
#       full_data_only     all layers retrained from the Dev108 data-only model, no physics
#       scratch_pinn       same network from random weights on the N days, with physics
#       scratch_data_only  same, no physics (shows whether transfer helps at all)
#       persistence        θ(t) = θ(t − 2 min)
#     Physics weights = the final ones (λ_phys, λ_BC, λ_LBC, λ_zvar); no IC term (the model does
#     not start at t0). Same step budget and early stopping for every trained strategy.
#     Source models = the main-run Dev108 PINN and data-only network (seed 42).
#     Results saved after every run (resumable).
# ═══════════════════════════════════════════════════════════════════════════
import copy, time
if not RUN["transfer"]:
    print("Transfer cell skipped (RUN['transfer'] = False).")
else:
    VG107 = dict(theta_r=0.05, theta_s=0.463, alpha=5.9, n=1.48, m=1 - 1 / 1.48, Ks=0.0041)   # Table 1 / §3.4
    VG107["Ks_ms"] = VG107["Ks"] / 3600.0
    TL_DAYS   = [3, 7, 14, 30]
    TL_SEEDS  = BUDGET["seeds"]
    TL_STEPS, TL_BATCH, TL_EVAL, TL_PATIENCE = BUDGET["tl_steps"], 256, BUDGET["tl_eval"], 10   # 2000, 256, 50, 10
    TL_LR = {"head": 1e-3, "all": 5e-4}
    TL_CSV = os.path.join(OUT, f"transfer_107_{RUN_TAG}.csv")

    class _TargetSite:
        """Temporarily switch the soil constants used everywhere (VG is shared by reference)."""
        def __enter__(self):
            self._saved = dict(VG); VG.update(VG107); return self
        def __exit__(self, *a):
            VG.clear(); VG.update(self._saved)

    # ── Dev107 data: same pipeline as Dev108 (2-min grid, gaps, fill dropped, smoothed target) ──
    d7 = prepare_site(DATA_CSV_107, 107, warmup_theta=None, label="Dev107")
    d7["theta_raw"] = d7["theta"].values
    d7["theta"] = d7.groupby("seg")["theta"].transform(lambda s: s.rolling(ROLL_W, min_periods=1).median())
    # Dev107 times on Dev108's model time axis (same calendar time -> same t)
    _ts8 = df["timestamp"].values.astype("datetime64[ns]").astype(np.int64)
    d7["t_sec"] = np.interp(d7["timestamp"].values.astype("datetime64[ns]").astype(np.int64), _ts8, df["t_sec"].values)

    TEST_START_TL = pd.Timestamp(ts_te[0]); TEST_END_TL = pd.Timestamp(ts_te[-1])
    d7_test = d7[(d7.timestamp >= TEST_START_TL) & (d7.timestamp <= TEST_END_TL)].reset_index(drop=True)
    d7_pre  = d7[d7.timestamp < TEST_START_TL].reset_index(drop=True)
    print(f"  test window {TEST_START_TL:%Y-%m-%d %H:%M} -> {TEST_END_TL:%Y-%m-%d %H:%M} | Dev107 test points {len(d7_test):,}")

    def _feats(d):
        Xr, y, rain_, th_, ts_sec, ts_ = build_features(d, LAG)
        return dict(Xr=Xr, y=y.ravel(), rain=rain_, t=ts_sec, ts=ts_)
    TE = _feats(d7_test)
    TE["raw"] = d7_test.set_index("timestamp")["theta_raw"].reindex(pd.to_datetime(TE["ts"])).values
    pers = TE["Xr"][:, 1]
    mse_pers = float(np.mean((TE["y"] - pers) ** 2))

    def _calib(n_days):
        days = pd.Series(d7_pre.timestamp.dt.normalize().unique()).sort_values()
        keep = set(days.iloc[-n_days:])
        d = d7_pre[d7_pre.timestamp.dt.normalize().isin(keep)].reset_index(drop=True)
        F = _feats(d); n = len(F["y"]); k = max(int(0.8 * n), n - 2000)
        tr = {kk: v[:k] for kk, v in F.items()}; va = {kk: v[k:] for kk, v in F.items()}
        return F, tr, va, d.timestamp.min(), d.timestamp.max()

    def _tn(t_sec):  return to_t((((t_sec - T_MIN) / (T_MAX - T_MIN)).astype(np.float32))[:, None])
    def _predict(m, F):
        m.eval(); out = []
        with torch.no_grad():
            Xn = physics_normalize_X(F["Xr"])
            for i in range(0, len(Xn), 8192):
                xb = to_t(Xn[i:i + 8192]); tb = _tn(F["t"][i:i + 8192])
                out.append(m(z_sens_norm.expand(len(xb), -1), tb, xb).cpu().numpy().ravel())
        m.train(); return np.concatenate(out)
    def _test_phys(m, n_max=4000):
        idx = np.linspace(0, len(TE["y"]) - 1, min(n_max, len(TE["y"]))).astype(int)
        zc, tc, fc = make_colloc_batch(TE["t"][idx], None, physics_normalize_X(TE["Xr"][idx]))
        r, _, _ = richards_residual(m, zc, tc, fc)
        return float(((r / (DTHDT_SCALE + 1e-12)) ** 2).mean().item())

    def _loss(m, Xb, yb, rb, tsec_b, lam):
        tb = _tn(tsec_b)
        loss = (((m(z_sens_norm.expand(len(Xb), -1), tb, Xb).ravel() - yb) * y_scl_t) ** 2).mean()
        if lam["LAM"] > 0:
            zc, tc, fc = make_colloc_batch(tsec_b, None, Xb.detach().cpu().numpy())
            r, _, dthdz = richards_residual(m, zc, tc, fc)
            loss = loss + lam["LAM"] * ((r / (DTHDT_SCALE + 1e-12)) ** 2).mean()
            loss = loss + lam["LAM_ZVAR"] * (-dthdz.view(len(Xb), len(Z_COLLOC)).var(dim=1).mean())
        if lam["LAM_BC"] > 0:
            z0 = torch.zeros(len(Xb), 1, device=DEVICE).requires_grad_(True)
            q = surface_bc_flux(m, z0, tb.clone().requires_grad_(True), Xb)
            q_t = torch.clamp(rb[:, None] * RAIN_MM_TO_M / RAIN_WINDOW_S, max=VG["Ks_ms"])
            loss = loss + lam["LAM_BC"] * (((q - q_t) / VG["Ks_ms"]) ** 2).mean()
        if lam["LAM_LBC"] > 0:
            loss = loss + lam["LAM_LBC"] * lower_bc_free_drainage_loss(m, tb, Xb)
        return loss

    def _train(m, tr, va, lam, which, seed):
        torch.manual_seed(seed); rng = np.random.default_rng(seed)
        lin = [l for l in m.net if isinstance(l, nn.Linear)]
        for p in m.parameters(): p.requires_grad_(which == "all")
        if which == "head":
            for p in lin[-1].parameters(): p.requires_grad_(True)
        params = [p for p in m.parameters() if p.requires_grad]
        opt = optim.Adam(params, lr=TL_LR[which])
        Xn = to_t(physics_normalize_X(tr["Xr"])); y = to_t(tr["y"]); rain = to_t(tr["rain"])
        best, best_state, bad = np.inf, copy.deepcopy(m.state_dict()), 0
        for step in range(1, TL_STEPS + 1):
            ib = rng.choice(len(y), size=min(TL_BATCH, len(y)), replace=False)
            opt.zero_grad()
            _loss(m, Xn[ib], y[ib], rain[ib], tr["t"][ib], lam).backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step()
            if step % TL_EVAL == 0:
                v = float(np.mean((_predict(m, va) - va["y"]) ** 2))
                if v < best: best, best_state, bad = v, copy.deepcopy(m.state_dict()), 0
                else:
                    bad += 1
                    if bad >= TL_PATIENCE: break
        m.load_state_dict(best_state)
        for p in m.parameters(): p.requires_grad_(True)
        return m, step

    def _score(pred, m=None):
        mse = float(np.mean((TE["y"] - pred) ** 2)); ok = ~np.isnan(TE["raw"])
        return dict(test_rmse=np.sqrt(mse), test_r2=float(r2_score(TE["y"], pred)),
                    skill_vs_persistence=1 - mse / mse_pers,
                    test_rmse_raw=float(np.sqrt(np.mean((TE["raw"][ok] - pred[ok]) ** 2))),
                    test_pde_residual=_test_phys(m) if m is not None else np.nan)

    SRC = {"pinn": copy.deepcopy(best_state), "data_only": copy.deepcopy(best_state_ml)}   # Dev108 main-run models
    LAM_TL  = dict(LAM=LAM, LAM_BC=LAM_BC, LAM_LBC=LAM_LBC, LAM_ZVAR=LAM_ZVAR)
    LAM_OFF = {k: 0.0 for k in LAM_TL}
    PLAN = [("head_pinn", "pinn", "head", LAM_TL), ("full_pinn", "pinn", "all", LAM_TL),
            ("full_data_only", "data_only", "all", LAM_OFF),
            ("scratch_pinn", None, "all", LAM_TL), ("scratch_data_only", None, "all", LAM_OFF)]

    rows = pd.read_csv(TL_CSV).to_dict("records") if os.path.exists(TL_CSV) else []
    done = {(r["strategy"], int(r["n_days"]), int(r["seed"])) for r in rows}
    if done: print(f"  resuming: {len(done)} transfer runs already saved")
    def _save(r):
        rows.append(r); pd.DataFrame(rows).to_csv(TL_CSV, index=False)

    t0 = time.time()
    with _TargetSite():
        def _fresh(state=None):
            m = RichardsPINN(feat_dim=X_tr.shape[1]).to(DEVICE)
            if state is not None: m.load_state_dict(state)
            return m
        n_par = sum(p.numel() for p in _fresh().parameters())
        n_head = sum(p.numel() for p in [l for l in _fresh().net if isinstance(l, nn.Linear)][-1].parameters())
        # no-data references (independent of N)
        for name, st in [("zero_shot", "pinn"), ("zero_shot_data_only", "data_only")]:
            if (name, 0, 0) not in done:
                m = _fresh(SRC[st]); _save(dict(strategy=name, n_days=0, seed=0, trainable=0, steps=0, **_score(_predict(m, TE), m)))
        if ("persistence", 0, 0) not in done:
            _save(dict(strategy="persistence", n_days=0, seed=0, trainable=0, steps=0, test_rmse=np.sqrt(mse_pers),
                       test_r2=float(r2_score(TE["y"], pers)), skill_vs_persistence=0.0,
                       test_rmse_raw=float(np.sqrt(np.nanmean((TE["raw"] - pers) ** 2))), test_pde_residual=np.nan))
        for N in TL_DAYS:
            F, tr, va, c0, c1 = _calib(N)
            print(f"\n  N = {N:>2} days of Dev107 data ({c0:%Y-%m-%d} -> {c1:%Y-%m-%d}, {len(F['y']):,} samples)")
            if ("recal", N, 0) not in done:
                m = _fresh(SRC["pinn"]); a, b = np.polyfit(_predict(m, F), F["y"], 1)
                r = _score(a * _predict(m, TE) + b); r["test_pde_residual"] = np.nan
                _save(dict(strategy="recal", n_days=N, seed=0, trainable=2, steps=0, **r))
                print(f"    recal: θ = {a:.3f}·θ̂ + {b:+.4f} | test RMSE {r['test_rmse']:.5f}")
            for name, st, which, lam in PLAN:
                for s in TL_SEEDS:
                    if (name, N, s) in done: continue
                    torch.manual_seed(s)
                    m = _fresh(SRC[st] if st else None)
                    m, steps = _train(m, tr, va, lam, which, s)
                    r = _score(_predict(m, TE), m)
                    _save(dict(strategy=name, n_days=N, seed=s, trainable=n_head if which == "head" else n_par, steps=steps, **r))
                    print(f"    {name:<18s} seed {s}: test RMSE {r['test_rmse']:.5f} | skill {r['skill_vs_persistence']:+.2f} | "
                          f"PDE res {r['test_pde_residual']:.2e} | {steps} steps [{(time.time() - t0) / 60:.0f} min]", flush=True)

    tl = pd.DataFrame(rows)
    summ = (tl.groupby(["strategy", "n_days"])
              .agg(test_rmse=("test_rmse", "mean"), test_rmse_sd=("test_rmse", "std"), skill=("skill_vs_persistence", "mean"),
                   test_rmse_raw=("test_rmse_raw", "mean"), pde_residual=("test_pde_residual", "mean"),
                   trainable=("trainable", "first"), seeds=("seed", "nunique")).reset_index())
    summ.to_csv(os.path.join(OUT, "transfer_107_summary.csv"), index=False)
    print("\nTRANSFER Dev108 -> Dev107 (mean over seeds; test = Dev108 test window)")
    print(summ.round(5).to_string(index=False))

    # paired: does starting from Dev108 beat starting from scratch, and does physics help?
    print("\nPaired differences in test RMSE (same N and seed; < 0 favours the first):")
    piv = tl.pivot_table(index=["n_days", "seed"], columns="strategy", values="test_rmse")
    for a_, b_ in [("full_pinn", "scratch_pinn"), ("full_data_only", "scratch_data_only"),
                   ("full_pinn", "full_data_only"), ("scratch_pinn", "scratch_data_only")]:
        if a_ in piv and b_ in piv:
            for N in TL_DAYS:
                d = (piv[a_] - piv[b_]).xs(N, level="n_days").dropna()
                if len(d) > 1:
                    h = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))
                    print(f"  {a_} − {b_}, N={N:>2}: {d.mean():+.5f} ± {h:.5f} {'(differs)' if abs(d.mean()) > h else '(n.s.)'}")

    # figure: test RMSE vs days of Dev107 data
    fig, ax = plt.subplots(figsize=(8, 4.6))
    sty = {"full_pinn": ("#0b6e4f", "o-"), "full_data_only": ("#c0392b", "s--"), "head_pinn": ("#7d3c98", "^-"),
           "scratch_pinn": ("#0b6e4f", "o:"), "scratch_data_only": ("#c0392b", "s:"), "recal": ("#555555", "d-")}
    for k, (c, ls) in sty.items():
        g_ = summ[summ.strategy == k].sort_values("n_days")
        if len(g_): ax.errorbar(g_.n_days, g_.test_rmse, yerr=g_.test_rmse_sd.fillna(0), fmt=ls, color=c, capsize=3, label=k)
    for k, c in [("zero_shot", "#0b6e4f"), ("persistence", "black")]:
        v = summ[summ.strategy == k].test_rmse
        if len(v): ax.axhline(float(v.iloc[0]), color=c, ls="-." if k == "zero_shot" else ":", lw=1, label=k)
    ax.set_xscale("log"); ax.set_xticks(TL_DAYS); ax.set_xticklabels(TL_DAYS)
    ax.set_xlabel("Days of Dev107 data used"); ax.set_ylabel("Test RMSE at Dev107 (m³/m³)")
    ax.set_title("Setting up a model for Dev107 from Dev108", loc="left"); ax.legend(fontsize=7, ncol=2)
    fig.tight_layout(); savefig(fig, "fig_transfer_108_to_107.png")

## 11. Alert levels and lead time
Recomputes the alert layer of the manuscript (lead time T_L to FoS = 1.3, RED ≤ 2 h, YELLOW ≤ 6 h, debounced escalation) on the final main-run PIML. Output: `alert_stream_summary.txt`, `alert_episodes.csv`, `alert_debounce_policies.csv`, `fig_alert_stream.png`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 11. ALERT LEVELS AND LEAD TIME — final main-run PIML, Device 108
#     Same definitions as the alert layer of the submitted manuscript (Eq. 17, Section 3.8):
#       lead time T_L = (FoS − FoS_crit) / |dFoS/dt|, falling FoS only (causal backward
#       difference), capped at 48 h; RED if T_L ≤ 2 h, YELLOW if 2 h < T_L ≤ 6 h.
#       Analysed record: first 24 h excluded, samples right after an outage > 1 h excluded,
#       operational moisture range Se ≥ 0.5.
#       Debounced policy: escalation when 2 of the last 3 samples are non-GREEN, released after
#       30 min of sustained GREEN; while escalated a single RED sample issues RED.
#     Input: fos_timeseries_main.csv written by the main cell.
#     Output: alert_stream_summary.txt, alert_episodes.csv, alert_debounce_policies.csv,
#             fig_alert_stream.png
# ═══════════════════════════════════════════════════════════════════════════
if not RUN["alerts"]:
    print("Alert cell skipped (RUN['alerts'] = False).")
else:
    import matplotlib.dates as mdates

    FOSCRIT     = 1.30     # critical factor of safety (Eq. 17)
    TL_CAP_H    = 48.0     # lead-time cap (h)
    YELLOW_H    = 6.0      # T_L ≤ 6 h -> at least YELLOW
    RED_H       = 2.0      # T_L ≤ 2 h -> RED
    SPINUP_H    = 24.0     # first 24 h excluded
    OUTAGE_S    = 3600.0   # a gap > 1 h marks a post-outage sample (excluded)
    CADENCE_MIN = 2.0      # grid step (min), used for durations
    ESC_WIN, ESC_K = 3, 2  # escalation: 2 of the last 3 samples non-GREEN
    RELEASE_MIN = 30.0     # release after 30 min of sustained GREEN
    SE_MIN      = 0.50     # operational moisture range
    EVENTS = {             # showcase storms of the manuscript (Section 5.3), [start, end)
        "Ev1 (10 Nov)":    ("2025-11-10", "2025-11-11"),
        "Ev2 (18-19 Nov)": ("2025-11-18", "2025-11-20"),
        "Ev3 (24-25 Jan)": ("2026-01-24", "2026-01-26"),
    }

    def load_stream(csv_path):
        """FoS series -> lead time T_L and the analysed-record mask."""
        f = pd.read_csv(csv_path, parse_dates=["timestamp"]).sort_values("timestamp")
        f = f.drop_duplicates(subset="timestamp", keep="first").reset_index(drop=True)
        gap_s = f["timestamp"].diff().dt.total_seconds()
        rate  = f["FoS_piml"].diff() / (gap_s / 3600.0)            # dFoS/dt, per hour
        decay = rate.clip(upper=0.0).abs()                         # falling FoS only
        with np.errstate(divide="ignore", invalid="ignore"):
            TL = (f["FoS_piml"] - FOSCRIT) / decay
        TL = TL.clip(upper=TL_CAP_H)
        TL[(decay < 1e-12) | decay.isna()] = TL_CAP_H               # flat / rising / first sample
        f["TL"] = TL
        Se = (f["theta_piml"] - VG["theta_r"]) / (VG["theta_s"] - VG["theta_r"])
        t0 = f["timestamp"].iloc[0]
        f["analysed"] = ((f["timestamp"] >= t0 + pd.Timedelta(hours=SPINUP_H))
                         & ~(gap_s > OUTAGE_S) & (Se >= SE_MIN)).values
        return f

    def classify(tl):
        red, nong = (tl <= RED_H).values, (tl <= YELLOW_H).values
        return red, nong, nong & ~red                              # RED, non-GREEN, YELLOW only

    def roll_ge(flags, w, k):
        """True where at least k of the trailing w samples are True (causal)."""
        c = np.convolve(flags.astype(int), np.ones(w, dtype=int), mode="full")[:len(flags)]
        return c >= k

    def debounced_red(red, nong):
        trig = roll_ge(nong, ESC_WIN, ESC_K)
        release_k = int(round(RELEASE_MIN / CADENCE_MIN))
        esc = np.zeros(len(red), dtype=bool); on, green_run = False, 0
        for i in range(len(red)):
            if not on and trig[i]:
                on, green_run = True, 0
            if on:
                green_run = green_run + 1 if not nong[i] else 0
                if green_run >= release_k:
                    on = False
            esc[i] = on
        return esc, red & esc

    def episodes(flags):
        out, start = [], None
        for i, v in enumerate(flags):
            if v and start is None: start = i
            elif not v and start is not None: out.append((start, i - 1)); start = None
        if start is not None: out.append((start, len(flags) - 1))
        return out

    in_window = lambda ts_, win: pd.Timestamp(win[0]) <= ts_ < pd.Timestamp(win[1])

    f_al = load_stream(os.path.join(OUT, "fos_timeseries_main.csv"))
    fa = f_al[f_al["analysed"]].reset_index(drop=True)
    red, nong, yell = classify(fa["TL"])
    esc, dred = debounced_red(red, nong)
    e_red, e_nong, e_esc, e_dred = episodes(red), episodes(nong), episodes(esc), episodes(dred)
    nc = fa["TL"][fa["TL"] < TL_CAP_H]
    days = fa["timestamp"].diff().dt.total_seconds().clip(upper=OUTAGE_S).sum() / 86400.0

    L = ["ALERT-STREAM SUMMARY  (final main-run PIML; T_L per Eq. 17, causal)", "=" * 68,
         f"analysed steps n              : {len(fa):,} of {len(f_al):,}",
         f"monitored coverage           : {days:.1f} days",
         f"FoS in analysed record       : min {fa['FoS_piml'].min():.3f} | mean {fa['FoS_piml'].mean():.3f}",
         f"GREEN / YELLOW / RED (%)     : {(fa['TL'] > YELLOW_H).mean() * 100:.2f} / "
         f"{((fa['TL'] > RED_H) & (fa['TL'] <= YELLOW_H)).mean() * 100:.2f} / {(fa['TL'] <= RED_H).mean() * 100:.2f}",
         f"RED samples / YELLOW samples : {int(red.sum())} / {int(yell.sum())}",
         f"non-capped T_L steps         : {len(nc):,} ({len(nc) / max(len(fa), 1) * 100:.1f}% of record)",
         f"median non-capped T_L        : {nc.median():.2f} h" if len(nc) else "median non-capped T_L        : —",
         f"minimum T_L                  : {fa['TL'].min():.2f} h at {fa['timestamp'].iloc[int(np.argmin(fa['TL'].values))]}"
         if len(fa) else "minimum T_L                  : —",
         "", "EPISODE COUNTS (analysed record)",
         f"raw RED episodes             : {len(e_red)}",
         f"raw non-GREEN episodes       : {len(e_nong)}",
         f"confirmed escalations        : {len(e_esc)}",
         f"debounced RED episodes       : {len(e_dred)}",
         "", "SHOWCASE EVENTS (debounced policy)"]
    for name, win in EVENTS.items():
        kept = any(in_window(fa["timestamp"].iloc[a], win) for a, _ in e_dred)
        runs = [b - a + 1 for a, b in e_red if in_window(fa["timestamp"].iloc[a], win)]
        L.append(f"  {name:16s}: {'RED issued' if kept else 'no RED':10s} raw RED runs (samples) = {runs}")
    summary = "\n".join(L)
    print(summary)
    open(os.path.join(OUT, "alert_stream_summary.txt"), "w", encoding="utf-8").write(summary + "\n")

    ep_rows = []
    for kind, eps in [("raw_red", e_red), ("escalation", e_esc), ("debounced_red", e_dred)]:
        for a, b in eps:
            ep_rows.append(dict(kind=kind, start=fa["timestamp"].iloc[a], end=fa["timestamp"].iloc[b],
                                n_samples=b - a + 1, min_TL_h=float(fa["TL"].iloc[a:b + 1].min()),
                                min_FoS=float(fa["FoS_piml"].iloc[a:b + 1].min())))
    pd.DataFrame(ep_rows, columns=["kind", "start", "end", "n_samples", "min_TL_h", "min_FoS"]).to_csv(
        os.path.join(OUT, "alert_episodes.csv"), index=False)

    # debounce policies (Table 11 of the submission)
    POLICIES = {"Raw single-sample RED": red,
                "RED persistence, 2-of-3": roll_ge(red, 3, 2),
                "RED persistence, 3-of-5": roll_ge(red, 5, 3),
                "Adopted: 2-of-3 escalation + single-sample RED (30-min GREEN release)": dred}
    pol_rows = []
    for name, flags in POLICIES.items():
        eps = episodes(flags)
        row = {"policy": name, "red_episodes": len(eps),
               "reduction_vs_raw_pct": (1 - len(eps) / len(e_red)) * 100 if e_red else np.nan}
        delays = []
        for ename, win in EVENTS.items():
            raw_t = [fa["timestamp"].iloc[a] for a, _ in e_red if in_window(fa["timestamp"].iloc[a], win)]
            pol_t = [fa["timestamp"].iloc[a] for a, _ in eps if in_window(fa["timestamp"].iloc[a], win)]
            row[ename] = "preserved" if pol_t else ("suppressed" if raw_t else "no raw RED")
            if pol_t and raw_t:
                delays.append((min(pol_t) - min(raw_t)).total_seconds() / 60.0)
        row["max_confirmation_delay_min"] = max(delays) if delays else np.nan
        pol_rows.append(row)
    pol_df = pd.DataFrame(pol_rows)
    print("\nDebounce policies:\n" + pol_df.to_string(index=False))
    pol_df.to_csv(os.path.join(OUT, "alert_debounce_policies.csv"), index=False)

    fig, ax = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    ax[0].plot(*_gb(fa["timestamp"], fa["FoS_piml"]), color="#555555", lw=0.7, label="FoS (PIML)")
    ax[0].scatter(fa["timestamp"][yell], fa["FoS_piml"][yell], s=12, color="#F9A825", label="YELLOW (2 h < T_L ≤ 6 h)", zorder=4)
    ax[0].scatter(fa["timestamp"][red], fa["FoS_piml"][red], s=16, color="#C62828", label="RED (T_L ≤ 2 h)", zorder=5)
    for a, b in e_esc:
        ax[0].axvspan(fa["timestamp"][a], fa["timestamp"][b], color="#F9A825", alpha=0.15, lw=0)
    ax[0].axhline(FOSCRIT, color="#C62828", ls="--", lw=1.0, label="FoS = 1.3")
    ax[0].set_ylabel("FoS (−)"); ax[0].legend(fontsize=8, ncol=4, loc="upper right"); light_grid(ax[0])
    ax[0].set_title(f"Alert stream, analysed record (n = {len(fa):,}); shaded = confirmed escalations", loc="left")
    ax[1].plot(*_gb(fa["timestamp"], fa["TL"]), color="#14407D", lw=0.7)
    ax[1].axhspan(0, RED_H, color="#C62828", alpha=0.12); ax[1].axhspan(RED_H, YELLOW_H, color="#F9A825", alpha=0.12)
    ax[1].set_ylim(0, TL_CAP_H); ax[1].set_ylabel("T_L (h)"); light_grid(ax[1])
    ax[1].xaxis.set_major_locator(mdates.MonthLocator())
    ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    fig.tight_layout(); savefig(fig, "fig_alert_stream.png")


## 12. Architecture search (optional, off by default)
Repeats the two-stage search that selected the 5×128 + 64 tanh network (`RUN["arch_search"] = True`, ≈ 3 h GPU). The tables of the original search are in `results/architecture_search/`.


In [ ]:
# 12. Architecture search (optional; RUN['arch_search'] in the setup cell, ≈ 3 h GPU).
#     The selected network (ARCH in the setup cell) came from this search; its result tables
#     are in results/architecture_search/.
if not RUN["arch_search"]:
    print(f"Architecture search skipped — using ARCH = {ARCH}.")
else:
    # ═══════════════════════════════════════════════════════════════════════════
    # Two-stage random search, ranked on VALIDATION only.
    #    Stage 1: random screen, 1 seed, λ-check budget.
    #    Stage 2: shortlist x 3 seeds at the same budget; choose the SMALLEST network whose
    #             mean validation RMSE is within one SD of the best.
    # ═══════════════════════════════════════════════════════════════════════════
    import itertools, random, functools

    class ArchPINN(nn.Module):
        """Same inputs/outputs as RichardsPINN; depth = hidden layers of `width`,
        taper adds one more hidden layer of width//2 (so 3+taper = the original 4 layers)."""
        def __init__(self, feat_dim, width=128, depth=4, act="silu", taper=True):
            super().__init__()
            A = {"tanh": nn.Tanh, "silu": nn.SiLU}[act]; d, L = feat_dim + 2, []
            for _ in range(depth): L += [nn.Linear(d, width), A()]; d = width
            if taper: L += [nn.Linear(d, width // 2), A()]; d = width // 2
            L += [nn.Linear(d, 1), nn.Sigmoid()]
            self.net = nn.Sequential(*L)
            self.theta_lo, self.theta_hi = VG["theta_r"] + 0.01, VG["theta_s"] - 0.01
        def forward(self, z_norm, t_norm, feat):
            return self.theta_lo + (self.theta_hi - self.theta_lo) * self.net(torch.cat([z_norm, t_norm, feat], 1))

    def run_arch(arch, seed):
        globals()["RichardsPINN"] = functools.partial(ArchPINN, **arch)   # train_piml_sens builds RichardsPINN(...)
        try:
            return train_piml_sens(SENS_DEFAULT, seed, SENS_ADAM_EPOCHS, SENS_LBFGS_EPOCHS)
        finally:
            globals()["RichardsPINN"] = _ARCH_ORIG

    _ARCH_ORIG = RichardsPINN
    n_params = lambda a: sum(p.numel() for p in ArchPINN(X_tr.shape[1], **a).parameters())
    key = lambda a: (int(a["depth"]), int(a["width"]), str(a["act"]), bool(a["taper"]))

    SPACE = dict(depth=[2, 3, 4, 5], width=[32, 64, 128, 256], act=["tanh", "silu"], taper=[False, True])
    ALWAYS = [dict(depth=3, width=128, act="tanh", taper=True),    # submitted network (4 hidden layers)
              dict(depth=4, width=128, act="silu", taper=True),    # earlier candidate (SiLU)
              dict(depth=5, width=32,  act="silu", taper=True)]    # small network that screened well
    N_SCREEN, RS_SEED = 16, 0
    combos = [dict(zip(SPACE, v)) for v in itertools.product(*SPACE.values())]
    random.seed(RS_SEED)
    pool = [a for a in combos if key(a) not in {key(x) for x in ALWAYS}]
    screen = ALWAYS + random.sample(pool, N_SCREEN - len(ALWAYS))
    print(f"{len(combos)} configurations in the space; screening {len(screen)}")

    csv1 = os.path.join(OUT, "arch_stage1.csv")
    s1 = pd.read_csv(csv1).to_dict("records") if os.path.exists(csv1) else []
    done = {key(r) for r in s1}
    for a in screen:
        if key(a) in done: continue
        t0 = time.time(); r = run_arch(a, 42)
        s1.append(dict(**a, n_params=n_params(a), val_rmse=r["val_rmse"], val_phys=r["val_phys"]))
        pd.DataFrame(s1).to_csv(csv1, index=False)                               # resumable
        print(f"  {a} | {n_params(a):>7,} params | val RMSE {r['val_rmse']:.5f} | {(time.time()-t0)/60:.1f} min")
    s1 = pd.DataFrame(s1).sort_values("val_rmse").reset_index(drop=True)

    # Stage 2: top 4 + the always-included networks, 3 seeds each
    short = {key(r): dict(depth=int(r.depth), width=int(r.width), act=r.act, taper=bool(r.taper)) for _, r in s1.head(4).iterrows()}
    short.update({key(a): a for a in ALWAYS})
    csv2 = os.path.join(OUT, "arch_stage2.csv")
    s2 = pd.read_csv(csv2).to_dict("records") if os.path.exists(csv2) else []
    done2 = {(key(r), int(r["seed"])) for r in s2}
    for a in short.values():
        for s in [42, 43, 44]:
            if (key(a), s) in done2: continue
            r = run_arch(a, s)
            s2.append(dict(**a, seed=s, n_params=n_params(a), val_rmse=r["val_rmse"], val_phys=r["val_phys"],
                           test_r2=r["test_r2"], test_rmse=r["test_rmse"]))
            pd.DataFrame(s2).to_csv(csv2, index=False)
    s2 = pd.DataFrame(s2)
    pers_val = float(np.sqrt(mean_squared_error(theta_val, X_val_raw[:, 1])))
    g = (s2.groupby(["depth", "width", "act", "taper", "n_params"])
           .agg(val_rmse=("val_rmse", "mean"), val_sd=("val_rmse", "std"),
                val_phys=("val_phys", "mean"), test_r2=("test_r2", "mean")).reset_index())
    g["skill_vs_persistence"] = 1 - g.val_rmse ** 2 / pers_val ** 2
    g["hidden_layers"] = g.depth + g.taper.astype(int)
    best = g.loc[g.val_rmse.idxmin()]
    g["within_1sd_of_best"] = g.val_rmse <= best.val_rmse + best.val_sd
    chosen = g[g.within_1sd_of_best].sort_values("n_params").iloc[0]
    print("\n", g.sort_values("val_rmse").round(5).to_string(index=False))
    print(f"\nPersistence val RMSE = {pers_val:.5f}")
    print(f"Selected (smallest within 1 SD of best): {int(chosen.hidden_layers)} hidden layers, width {int(chosen.width)}, "
          f"{chosen.act}, taper={bool(chosen.taper)}, {int(chosen.n_params):,} params")
    g.to_csv(os.path.join(OUT, "arch_selection_table.csv"), index=False)

## 13. λ_phys extension (optional, off by default)
λ_phys = 2, 3, 5, 10 at the λ-check budget (`RUN["lam_ext"] = True`, ≈ 4 h GPU).


In [ ]:
# 13. λ_phys extension, λ_phys = 2–10 (optional; RUN['lam_ext'] in the setup cell, ~4 h GPU).
#     Run after the λ-check cell in the same session.
if not RUN["lam_ext"]:
    print("λ_phys extension skipped (RUN['lam_ext'] = False).")
else:
    # ═══════════════════════════════════════════════════════════════════════════
    # λ_phys EXTENSION — upper range only (λ_phys = 2, 3, 5, 10)
    #    The λ check does not reach λ·Phys/Data ≥ 1, so the regime where the physics term
    #    dominates the data loss is tested here. Same loss, budget, seeds and metrics as the
    #    λ check; the default and data-only runs are reused from it when present.
    # ═══════════════════════════════════════════════════════════════════════════
    import time
    from scipy import stats

    LAM_EXT = [2.0, 3.0, 5.0, 10.0]          # fixed in advance; report all of them

    import pickle
    SENS_PKL = os.path.join(OUT, f"sens_cache_{RUN_TAG}.pkl")
    if "_cache" not in globals():
        _cache = pickle.load(open(SENS_PKL, "rb")) if os.path.exists(SENS_PKL) else {}

    def _run_ext(cfg):
        key = tuple(sorted(cfg.items()))
        if key not in _cache:
            t0 = time.time()
            _cache[key] = [train_piml_sens(cfg, s, SENS_ADAM_EPOCHS, SENS_LBFGS_EPOCHS)
                           for s in SENS_SEEDS]
            pickle.dump(_cache, open(SENS_PKL, "wb"))
            print(f"    trained {dict(cfg)['LAM']:g} x {len(SENS_SEEDS)} seeds "
                  f"in {(time.time()-t0)/60:.1f} min")
        return _cache[key]

    print("═" * 70)
    print(f"[8b] λ_phys extension {LAM_EXT} | seeds={SENS_SEEDS} | "
          f"budget {SENS_ADAM_EPOCHS}+{SENS_LBFGS_EPOCHS}")
    print("═" * 70)

    ref_runs = _run_ext(SENS_ZERO);    ref_agg = _agg(ref_runs)       # reused if cached
    def_runs = _run_ext(SENS_DEFAULT); def_val = np.array([r["val_rmse"] for r in def_runs])

    rows_ext = []
    for v in LAM_EXT:
        cfg = dict(SENS_DEFAULT); cfg["LAM"] = v
        runs = _run_ext(cfg)
        row = {"param": "LAM", "value": v, "is_default": False, "n_seeds": len(SENS_SEEDS)}
        row.update(_agg(runs))
        d = np.array([r["val_rmse"] for r in runs]) - def_val          # >0 = worse than default
        half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else np.nan
        row["d_val_rmse_mean"], row["d_val_rmse_ci95"] = float(d.mean()), float(half)
        row["differs_from_default"] = bool(abs(d.mean()) > half) if len(d) > 1 else False
        rows_ext.append(row)
        print(f"  LAM={v:<6g} | val RMSE {row['val_rmse']:.5f}±{row['val_rmse_std']:.5f}"
              f" | val phys {row['val_phys']:.2e} | test R² {row['test_r2']:.4f}"
              f" | λP/D {row['phys_over_data']:.3f} | LBC rms {row['lbc_rms']:.4f}"
              f" | Δ(val RMSE) {row['d_val_rmse_mean']:+.5f} ± {row['d_val_rmse_ci95']:.5f}"
              f" {'DIFFERS' if row['differs_from_default'] else 'n.s.'}")

    ext_df = pd.DataFrame(rows_ext)
    ext_df.to_csv(os.path.join(OUT, "sensitivity_lambda_phys_extension.csv"), index=False)
    print(f"  data-only reference: val RMSE {ref_agg['val_rmse']:.5f}±{ref_agg['val_rmse_std']:.5f}"
          f" | val phys {ref_agg['val_phys']:.2e} | test R² {ref_agg['test_r2']:.4f}")
    print("  NOTE: n.s. with few seeds is low power; it does not prove equivalence.")

    # ── figure: full λ_phys range (main sweep if available + extension) ──
    parts = [ext_df]
    if "sens_df" in globals():
        parts.insert(0, sens_df[sens_df.param == "LAM"])
    full = pd.concat(parts, ignore_index=True).sort_values("value").reset_index(drop=True)

    cols = [("val_rmse", "Val RMSE (m³/m³)"), ("val_phys", "Val Richards residual (norm.)"),
            ("test_r2", "Test R²"), ("phys_over_data", "λ·Phys / Data (val)")]
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
    for ax, (col, lab) in zip(axes, cols):
        ax.errorbar(full.value, full[col], yerr=full[col + "_std"].fillna(0),
                    marker="o", lw=1.2, capsize=3, color=C["pred"])
        dd = full[full.is_default.fillna(False).astype(bool)]
        ax.plot(dd.value, dd[col], "o", ms=11, mfc="none", mec=C["warn"], label="default")
        if col in ref_agg and col != "phys_over_data":
            ax.axhline(ref_agg[col], color=C["obs"], ls="--", lw=1.0, label="data-only (λ=0)")
        ax.set_xscale("log"); ax.set_xlabel("λ_phys"); ax.set_ylabel(lab)
        if col in ("val_phys", "phys_over_data"): ax.set_yscale("log")
        light_grid(ax)
    axes[0].legend(fontsize=7)
    fig.suptitle("λ_phys sensitivity over the full range (mean ± SD over seeds, reduced budget)",
                 fontweight="bold")
    fig.tight_layout()
    savefig(fig, "fig_sensitivity_lambda_phys_extended.png")

## 14. Package the results
Zips the output folder. Copy the CSV tables, figures, `run_log.txt` and `environment.txt` into `results/<split>/` of the repository.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 14. PACKAGE THE RESULTS — one zip of RUN_DIR (models, figures, CSV tables, logs)
# ═══════════════════════════════════════════════════════════════════════════
import shutil
archive = shutil.make_archive(os.path.join(os.path.dirname(RUN_DIR), os.path.basename(RUN_DIR)), "zip", RUN_DIR)
print(f"Results folder : {RUN_DIR}")
print(f"Zip archive    : {archive}  ({os.path.getsize(archive) / 1e6:.1f} MB)")
print("Copy the CSV tables, figures, run_log.txt and environment.txt into results/<split>/ of the "
      "repository (see README). Checkpoints (.pt) and resume caches (.pkl) are not needed there.")
